# PyTorch Broadcasting — 110 Progressive Exercises


> **Working copy:** This file may contain learner answers and scratch work. Use the paired `_virgin.ipynb` notebook whenever you want a clean retry.

This workbook builds broadcasting intuition from scalars and vectors to higher-rank machine-learning patterns, invalid-shape diagnosis, explicit axis insertion, `expand`, masks, backward unbroadcasting, and integrated capstones. The scaffold contains no official answer key. This working copy may contain learner solutions; type new predictions and tensor expressions in the answer cells.

## How to use the workbook

1. Run the supplied setup cell once after starting or restarting the kernel.
2. Work from top to bottom; later exercises rely on the same shape vocabulary.
3. Before touching PyTorch, write every required shape as a literal Python tuple.
4. For aligned-shape exercises, prepend conceptual leading `1`s on paper; do not reshape the tensors merely to fill the audit.
5. Predict expansion axes as a tuple of zero-based axes in the aligned result, such as `(0,)`, `(1,)`, or `()`.
6. Only then write the requested tensor operation and run the supplied test.
7. Move on only after every required variable prints `PASS`.

Tests obtain expected shapes and values from private runtime references. Visible test cells contain no expected tuples or tensors. Fixtures expose input construction but deliberately do not print shape answers.


## Before Exercise 001: what broadcasting means

### A tensor's shape describes its axes

A tensor's **shape** tells you how many positions it has along each axis.

For a matrix with shape `(2, 3)`:

- axis `0` has size `2` and usually represents the rows;
- axis `1` has size `3` and usually represents the columns.

You can picture it as:

```text
shape: (2, 3)
        │  └─ 3 columns
        └──── 2 rows
```

A rank-3 tensor might instead use axes such as `(batch, time, features)`. PyTorch stores only the sizes—it does not store labels like “rows” or “features.” The surrounding code gives each axis its meaning.

Some important shape syntax:

- `()` is a scalar tensor with no axes;
- `(3,)` is a rank-1 vector with one axis of size `3`;
- `(2, 3)` is a rank-2 matrix;
- `(2, 3, 4)` is a rank-3 tensor.

The comma in `(3,)` matters: it makes this a one-item Python tuple rather than the number `3` inside parentheses.

### Broadcasting is automatic reuse

Elementwise operations pair up corresponding positions:

```python
left + right
left * right
```

When both tensors have the same shape, each position has an obvious partner. Broadcasting is PyTorch's rule for some cases where the shapes differ.

A smaller tensor may act **as if** its values were reused across a larger tensor. PyTorch usually does not make all those physical copies first; this is a conceptual way to understand the operation.

For example, adding one scalar to a matrix reuses that scalar at every matrix position:

```text
matrix shape: (2, 3)
scalar shape: ()
result shape: (2, 3)
```

A dimension whose size is `1` is called a **singleton dimension**. It says, “there is one value along this axis.” Broadcasting may reuse that one value when the other tensor needs more positions along the same axis.

Broadcasting is not matrix multiplication and it is not a reduction:

- broadcasting reuses values so an elementwise operation can happen;
- matrix multiplication combines a contracted dimension;
- a reduction such as `sum` combines several values into fewer values.


## How PyTorch decides whether shapes can broadcast

PyTorch uses a mechanical rule. It does not guess what your axes mean.

### Step 1: line shapes up on the right

Suppose one tensor has shape `(2, 3)` and another has shape `(3,)`. PyTorch places their final dimensions under each other:

```text
matrix: (2, 3)
vector: (   3)
```

For shape reasoning, the missing position on the **left** can be imagined as a `1`:

```text
matrix: (2, 3)
vector: (1, 3)   # conceptual aligned shape
```

This does not physically reshape the vector. It is just the aligned shape PyTorch uses for the comparison.

PyTorch aligns on the right because broadcasting is defined that way—not because it understands rows or columns.

### Step 2: compare one axis at a time

After right-aligning the shapes, each pair of axis sizes has only three possibilities:

1. **The sizes are equal.** Keep that size.
2. **One size is `1`.** Reuse that operand's one value along this axis.
3. **The sizes differ and neither is `1`.** Broadcasting cannot happen; PyTorch raises an error.

A useful mnemonic is:

> **Same stays, one stretches, otherwise stop.**

### Example: a length-3 vector works with `(2, 3)`

```text
matrix: (2, 3)
vector: (1, 3)
```

Compare from the right:

- `3` and `3` are equal;
- `2` and conceptual `1` are compatible, so the vector is reused over two rows.

Conceptually:

```text
[10, 20, 30]
```

acts like:

```text
[[10, 20, 30],
 [10, 20, 30]]
```

### Example: why `(2,) * (2, 3)` fails

A length-2 vector is still aligned under the **last** matrix axis:

```text
matrix: (2, 3)
vector: (1, 2)
             ↑
```

The rightmost comparison is `3` versus `2`. They differ, and neither is `1`, so PyTorch stops with a broadcasting error.

PyTorch does not know that you intended the two values to belong to the two rows.

To express one value per row, the compact tensor needs shape `(2, 1)`:

```text
matrix:     (2, 3)
row values: (2, 1)
```

Now:

- the row sizes `2` and `2` match;
- the singleton column size `1` can be reused across `3` columns.

For row values `[10, 20]`, the prepared tensor looks like:

```text
[[10],
 [20]]
```

and broadcasting lets it act like:

```text
[[10, 10, 10],
 [20, 20, 20]]
```

This is why inserting a singleton axis with `[:, None]` can change an invalid operation into the intended row-wise operation.


## Vocabulary used in the exercise answers

The notebook asks you to predict several related facts. They answer different questions.

### Original shape

The original shape is the tensor's actual shape before broadcasting. You can derive it from the visible fixture construction.

### Aligned shape

The aligned shape is the shape after **conceptually** adding leading `1`s until all participating shapes have the same number of axes.

Example:

```text
original shape: (5,)
aligned against rank 3: (1, 1, 5)
```

Writing an aligned-shape prediction does not mean you should reshape the tensor in code.

### Compatibility

`compatible` answers one yes/no question: can PyTorch perform this elementwise operation under the broadcasting rules?

Write it as a Python Boolean:

```python
True
False
```

### Output shape

For a compatible axis:

- matching sizes remain unchanged;
- if one operand has size `1`, the result uses the other operand's size.

Apply that rule to every aligned axis to obtain the output shape.

### Expanded axes

An operand expands on an axis when its aligned size is `1` but the result needs a different size there.

Axes are numbered from left to right starting at zero.

Consider these shapes:

```text
left:   (3, 1)
right:  (1, 5)
result: (3, 5)
axes:     0  1
```

- `left` expands on axis `1`, so its expanded-axes tuple is `(1,)`;
- `right` expands on axis `0`, so its expanded-axes tuple is `(0,)`.

Use `()` when an operand does not expand on any axis. Use `(0, 1)` when it expands on both axes.

### Prepared views are real shape changes

An expression such as:

```python
values[:, None]
```

really inserts a singleton axis and creates a view with a new shape. That preparation is different from the later broadcasting:

1. `[:, None]` creates a broadcast-compatible shape;
2. the following elementwise operation performs broadcasting.


## The beginner broadcasting checklist

Use this short process for every exercise:

1. **Write the actual shapes.** Include `()` for a scalar and the comma in `(3,)` for a vector.
2. **Say what every axis means.** For example: rows and columns, or batch, time, and features.
3. **Right-align the shapes.** Add conceptual `1`s only on the left of shorter shapes.
4. **Compare from right to left.** For each axis, remember: **same stays, one stretches, otherwise stop**.
5. **Write the result shape.** Use the matching size or the non-`1` size at each compatible axis.
6. **Mark expanded axes.** Record where an aligned `1` had to act like a different result size.
7. **Check your intended meaning.** If two values are supposed to belong to two rows, is their shape `(2, 1)` rather than `(2,)`?
8. **Only then run the operation.** Use the supplied test to check your reasoning without revealing the expected answer.

When an operation fails, do not immediately try random `reshape` or `unsqueeze` calls. First draw the right-aligned shapes and find the exact incompatible axis.


## Notebook map

1. Foundations: scalars, vectors, rows, and columns — Exercises 001–010
2. Right alignment across ranks — Exercises 011–020
3. Inserting singleton axes deliberately — Exercises 021–030
4. Higher-rank and semantic-axis patterns — Exercises 031–045
5. Diagnosing incompatible shapes — Exercises 046–055
6. `expand`, `broadcast_to`, `repeat`, and storage — Exercises 056–065
7. Multiple operands, masks, and elementwise functions — Exercises 066–075
8. Machine-learning broadcasting patterns — Exercises 076–090
9. Backward must unbroadcast — Exercises 091–102
10. Broadcasting mastery capstones — Exercises 103–110


In [1]:
# Supplied test infrastructure: run once after starting or restarting the kernel.
import importlib.util as _fixture_importlib
from pathlib import Path as _FixturePath

import torch

DTYPE = torch.float64
_REFS = {}
_MISSING = object()

_fixture_filename = "_torch_broadcasting_fixtures.py"
_fixture_relatives = (
    _FixturePath(_fixture_filename),
    _FixturePath("notebooks") / _fixture_filename,
    _FixturePath("coursework/05-backprop-ninja/notebooks") / _fixture_filename,
    _FixturePath("karpathy_ml_course/coursework/05-backprop-ninja/notebooks") / _fixture_filename,
)
_fixture_path = next(
    (
        base / relative
        for base in (_FixturePath.cwd(), *_FixturePath.cwd().parents)
        for relative in _fixture_relatives
        if (base / relative).is_file()
    ),
    None,
)
if _fixture_path is None:
    raise FileNotFoundError(
        f"Could not find {_fixture_filename}. Keep it beside the exercise notebook."
    )
_fixture_spec = _fixture_importlib.spec_from_file_location(
    "_torch_broadcasting_fixtures", _fixture_path
)
assert _fixture_spec is not None and _fixture_spec.loader is not None
_fixture_module = _fixture_importlib.module_from_spec(_fixture_spec)
_fixture_spec.loader.exec_module(_fixture_module)


def _register_case(key, tensors):
    """Build private references from the exercise's visible tensors."""
    _REFS[key] = _fixture_module.build_reference(key, tensors)


def _check_private_value(variable_name, key, field):
    """Check a tuple/bool value without printing the private expected answer."""
    actual = globals().get(variable_name, _MISSING)
    assert actual is not _MISSING, f"Define `{variable_name}` in the answer cell first."
    expected = _REFS[key][field]
    assert type(actual) is type(expected), (
        f"`{variable_name}` must have type {type(expected).__name__}."
    )
    assert actual == expected, f"`{variable_name}` is not correct; redo the alignment checklist."
    print(f"PASS: {variable_name}")


def _check_private_tensor(variable_name, key, field):
    """Check tensor type, shape, dtype, and values without revealing references."""
    actual = globals().get(variable_name, _MISSING)
    assert actual is not _MISSING, f"Define `{variable_name}` in the answer cell first."
    assert isinstance(actual, torch.Tensor), f"`{variable_name}` must be a torch.Tensor."
    expected = _REFS[key][field]
    assert actual.shape == expected.shape, (
        f"`{variable_name}` has the wrong shape; revisit right alignment before retrying."
    )
    assert actual.dtype == expected.dtype, f"`{variable_name}` has the wrong dtype."
    try:
        torch.testing.assert_close(actual, expected, rtol=1e-7, atol=1e-9)
    except AssertionError:
        raise AssertionError(
            f"`{variable_name}` has the right shape but incorrect values."
        ) from None
    print(f"PASS: {variable_name}")


print("Broadcasting helpers ready. Start at Exercise 001.")


C:\Users\giloz\dev\pytorch_master_through_exercises\.venv\Lib\site-packages\torch\_subclasses\functional_tensor.py:368: UserWarning: Failed to initialize NumPy: No module named 'numpy' (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\torch\csrc\utils\tensor_numpy.cpp:84.)
  cpu = _conversion_method_template(device=torch.device("cpu"))


Broadcasting helpers ready. Start at Exercise 001.


## 1. Foundations: scalars, vectors, rows, and columns

Start by separating equal-shape elementwise operations from genuine expansion. A scalar can be reused everywhere; a short vector aligns with trailing axes.


### Exercise 001 — Scalar plus scalar

**Purpose:** Start with rank-zero tensors, where no axis expansion is needed.

**Visible inputs:** `a`, `b`. Run the fixture below and read its construction code; it deliberately prints no shape answers.

**Operation to reason about:** `a + b`

**Task:** Predict both input shapes and the output shape, then add the tensors.

**Ingredients:** A scalar tensor has rank zero and shape `()`.

**Required predictions and outputs:**

- `ex001_a_shape`: a literal Python shape tuple
- `ex001_b_shape`: a literal Python shape tuple
- `ex001_out_shape`: a literal Python shape tuple
- `ex001_out`: the requested PyTorch tensor

**Order:** Fill every shape/alignment prediction before writing tensor operations.

**Next concept:** Scalar plus vector


In [2]:
# Supplied visible fixture: inspect these values, but predict shapes without printing them.
a = torch.tensor(2.0, dtype=DTYPE)
b = torch.tensor(-0.5, dtype=DTYPE)
_register_case("ex001", {"a": a, "b": b})


In [3]:
# Exercise 001: predict shapes first; do not use autograd.
# Define `ex001_a_shape`.
# Define `ex001_b_shape`.
# Define `ex001_out_shape`.
# Define `ex001_out` with PyTorch tensor operations.
# Write your work below, then run the supplied test cell.
ex001_a_shape = ()
ex001_b_shape = ()
ex001_out_shape = ()
ex001_out = torch.tensor(1.5, dtype=DTYPE)

In [4]:
# Supplied test: expected shapes and values remain private.
_check_private_value("ex001_a_shape", "ex001", "a_shape")
_check_private_value("ex001_b_shape", "ex001", "b_shape")
_check_private_value("ex001_out_shape", "ex001", "out_shape")
_check_private_tensor("ex001_out", "ex001", "out")


PASS: ex001_a_shape
PASS: ex001_b_shape
PASS: ex001_out_shape
PASS: ex001_out


### Exercise 002 — Scalar plus vector

**Purpose:** See a scalar reused at every vector position.

**Visible inputs:** `a`, `b`. Run the fixture below and read its construction code; it deliberately prints no shape answers.

**Operation to reason about:** `a + b`

**Task:** Predict the shapes, then add the scalar to the vector.

**Ingredients:** A scalar can expand across every axis of the other operand.

**Required predictions and outputs:**

- `ex002_a_shape`: a literal Python shape tuple
- `ex002_b_shape`: a literal Python shape tuple
- `ex002_out_shape`: a literal Python shape tuple
- `ex002_out`: the requested PyTorch tensor

**Order:** Fill every shape/alignment prediction before writing tensor operations.

**Next concept:** Vector times scalar


In [5]:
# Supplied visible fixture: inspect these values, but predict shapes without printing them.
a = torch.tensor(10.0, dtype=DTYPE)
b = torch.tensor([1.0, 2.0, 3.0], dtype=DTYPE)
_register_case("ex002", {"a": a, "b": b})


In [6]:
# Exercise 002: predict shapes first; do not use autograd.
# Define `ex002_a_shape`.
# Define `ex002_b_shape`.
# Define `ex002_out_shape`.
# Define `ex002_out` with PyTorch tensor operations.
# Write your work below, then run the supplied test cell.
ex002_a_shape = ()
ex002_b_shape = (3,)
ex002_out_shape = (3,)
ex002_out = torch.tensor([11.0, 12.0, 13.0], dtype=DTYPE)

In [7]:
# Supplied test: expected shapes and values remain private.
_check_private_value("ex002_a_shape", "ex002", "a_shape")
_check_private_value("ex002_b_shape", "ex002", "b_shape")
_check_private_value("ex002_out_shape", "ex002", "out_shape")
_check_private_tensor("ex002_out", "ex002", "out")


PASS: ex002_a_shape
PASS: ex002_b_shape
PASS: ex002_out_shape
PASS: ex002_out


### Exercise 003 — Vector times scalar

**Purpose:** See that a scalar is reused for every vector element even when the vector is written first and the scalar second.

**Visible inputs:** `a`, `b`. Run the fixture below and read its construction code; it deliberately prints no shape answers.

**Operation to reason about:** `a * b`

**Task:** Predict the shapes, then multiply elementwise.

**Ingredients:** In Exercise 002, `a` was the scalar and `b` was the vector. Here, `a` is the vector and `b` is the scalar. Operand order means only which tensor appears before or after the operator; it does not reverse the vector's elements.

**Required predictions and outputs:**

- `ex003_a_shape`: a literal Python shape tuple
- `ex003_b_shape`: a literal Python shape tuple
- `ex003_out_shape`: a literal Python shape tuple
- `ex003_out`: the requested PyTorch tensor

**Order:** Fill every shape/alignment prediction before writing tensor operations.

**Next concept:** Equal vectors


In [8]:
# Supplied visible fixture: inspect these values, but predict shapes without printing them.
a = torch.tensor([1.0, -2.0, 3.0], dtype=DTYPE)
b = torch.tensor(4.0, dtype=DTYPE)
_register_case("ex003", {"a": a, "b": b})


In [9]:
# Exercise 003: predict shapes first; do not use autograd.
# Define `ex003_a_shape`.
# Define `ex003_b_shape`.
# Define `ex003_out_shape`.
# Define `ex003_out` with PyTorch tensor operations.
# Write your work below, then run the supplied test cell.
ex003_a_shape = (3,)
ex003_b_shape = ()
ex003_out_shape = (3,)
ex003_out = torch.tensor([4.0, -8.0, 12.0], dtype=DTYPE)

In [10]:
# Supplied test: expected shapes and values remain private.
_check_private_value("ex003_a_shape", "ex003", "a_shape")
_check_private_value("ex003_b_shape", "ex003", "b_shape")
_check_private_value("ex003_out_shape", "ex003", "out_shape")
_check_private_tensor("ex003_out", "ex003", "out")


PASS: ex003_a_shape
PASS: ex003_b_shape
PASS: ex003_out_shape
PASS: ex003_out


### Exercise 004 — Equal vectors

**Purpose:** Distinguish ordinary elementwise work from actual expansion.

**Visible inputs:** `a`, `b`. Run the fixture below and read its construction code; it deliberately prints no shape answers.

**Operation to reason about:** `a - b`

**Task:** Subtract equal-shaped vectors and decide whether any axis expands.

**Ingredients:** Equal dimensions match directly; no repetition is needed.

**Required predictions and outputs:**

- `ex004_a_shape`: a literal Python shape tuple
- `ex004_b_shape`: a literal Python shape tuple
- `ex004_out_shape`: a literal Python shape tuple
- `ex004_out`: the requested PyTorch tensor

**Order:** Fill every shape/alignment prediction before writing tensor operations.

**Next concept:** Scalar plus matrix


In [11]:
# Supplied visible fixture: inspect these values, but predict shapes without printing them.
a = torch.tensor([2.0, 4.0, 6.0], dtype=DTYPE)
b = torch.tensor([1.0, 3.0, 5.0], dtype=DTYPE)
_register_case("ex004", {"a": a, "b": b})


In [12]:
# Exercise 004: predict shapes first; do not use autograd.
# Define `ex004_a_shape`.
# Define `ex004_b_shape`.
# Define `ex004_out_shape`.
# Define `ex004_out` with PyTorch tensor operations.
# Write your work below, then run the supplied test cell.
ex004_a_shape = (3,)
ex004_b_shape = (3,)
ex004_out_shape = (3,)
ex004_out = torch.tensor([1.0, 1.0, 1.0], dtype=DTYPE)

In [13]:
# Supplied test: expected shapes and values remain private.
_check_private_value("ex004_a_shape", "ex004", "a_shape")
_check_private_value("ex004_b_shape", "ex004", "b_shape")
_check_private_value("ex004_out_shape", "ex004", "out_shape")
_check_private_tensor("ex004_out", "ex004", "out")


PASS: ex004_a_shape
PASS: ex004_b_shape
PASS: ex004_out_shape
PASS: ex004_out


### Exercise 005 — Scalar plus matrix

**Purpose:** Broadcast one rank-zero value over rows and columns.

**Visible inputs:** `a`, `b`. Run the fixture below and read its construction code; it deliberately prints no shape answers.

**Operation to reason about:** `a + b`

**Task:** Predict the matrix result shape, then perform the addition.

**Ingredients:** Imagine the scalar available at every matrix position.

**Required predictions and outputs:**

- `ex005_a_shape`: a literal Python shape tuple
- `ex005_b_shape`: a literal Python shape tuple
- `ex005_out_shape`: a literal Python shape tuple
- `ex005_out`: the requested PyTorch tensor

**Order:** Fill every shape/alignment prediction before writing tensor operations.

**Next concept:** Equal matrices


In [14]:
# Supplied visible fixture: inspect these values, but predict shapes without printing them.
a = torch.tensor([[0.0, 1.0, 2.0], [3.0, 4.0, 5.0]], dtype=DTYPE)
b = torch.tensor(0.5, dtype=DTYPE)
_register_case("ex005", {"a": a, "b": b})


In [24]:
# Exercise 005: predict shapes first; do not use autograd.
# Define `ex005_a_shape`.
# Define `ex005_b_shape`.
# Define `ex005_out_shape`.
# Define `ex005_out` with PyTorch tensor operations.
# Write your work below, then run the supplied test cell.
ex005_a_shape = (2, 3)
ex005_b_shape = ()
ex005_out_shape = (2, 3)
ex005_out = torch.tensor(
    [
        [0.5, 1.5, 2.5],
        [3.5, 4.5, 5.5],
    ],
    dtype=DTYPE
)

In [25]:
# Supplied test: expected shapes and values remain private.
_check_private_value("ex005_a_shape", "ex005", "a_shape")
_check_private_value("ex005_b_shape", "ex005", "b_shape")
_check_private_value("ex005_out_shape", "ex005", "out_shape")
_check_private_tensor("ex005_out", "ex005", "out")


PASS: ex005_a_shape
PASS: ex005_b_shape
PASS: ex005_out_shape
PASS: ex005_out


### Exercise 006 — Equal matrices

**Purpose:** Confirm that matching matrix shapes use elementwise multiplication without expansion.

**Visible inputs:** `a`, `b`. Run the fixture below and read its construction code; it deliberately prints no shape answers.

**Operation to reason about:** `a * b`

**Task:** Predict the shapes, then multiply matching entries.

**Ingredients:** Matching ranks and sizes need no broadcasting.

**Required predictions and outputs:**

- `ex006_a_shape`: a literal Python shape tuple
- `ex006_b_shape`: a literal Python shape tuple
- `ex006_out_shape`: a literal Python shape tuple
- `ex006_out`: the requested PyTorch tensor

**Order:** Fill every shape/alignment prediction before writing tensor operations.

**Next concept:** Implicit row vector


In [ ]:
# Supplied visible fixture: inspect these values, but predict shapes without printing them.
a = torch.arange(6, dtype=DTYPE).reshape(2, 3) + 1
b = torch.linspace(0.5, 1.0, steps=6, dtype=DTYPE).reshape(2, 3)
_register_case("ex006", {"a": a, "b": b})


In [ ]:
# Exercise 006: predict shapes first; do not use autograd.
# Define `ex006_a_shape`.
# Define `ex006_b_shape`.
# Define `ex006_out_shape`.
# Define `ex006_out` with PyTorch tensor operations.
# Write your work below, then run the supplied test cell.


In [ ]:
# Supplied test: expected shapes and values remain private.
_check_private_value("ex006_a_shape", "ex006", "a_shape")
_check_private_value("ex006_b_shape", "ex006", "b_shape")
_check_private_value("ex006_out_shape", "ex006", "out_shape")
_check_private_tensor("ex006_out", "ex006", "out")


### Exercise 007 — Implicit row vector

**Purpose:** Learn why a length-three vector naturally lines up with three matrix columns.

**Visible inputs:** `a`, `b`. Run the fixture below and read its construction code; it deliberately prints no shape answers.

**Operation to reason about:** `a + b`

**Task:** Add one feature value to each matrix column.

**Ingredients:** Right-align `(3,)` beneath the matrix's final axis.

**Required predictions and outputs:**

- `ex007_a_shape`: a literal Python shape tuple
- `ex007_b_shape`: a literal Python shape tuple
- `ex007_a_aligned_shape`: a literal Python shape tuple
- `ex007_a_expanded_axes`: a tuple of zero-based aligned axes
- `ex007_b_aligned_shape`: a literal Python shape tuple
- `ex007_b_expanded_axes`: a tuple of zero-based aligned axes
- `ex007_compatible`: a Python `bool`
- `ex007_out_shape`: a literal Python shape tuple
- `ex007_out`: the requested PyTorch tensor

**Order:** Fill every shape/alignment prediction before writing tensor operations.

**Next concept:** Explicit row vector


In [ ]:
# Supplied visible fixture: inspect these values, but predict shapes without printing them.
a = torch.arange(6, dtype=DTYPE).reshape(2, 3)
b = torch.tensor([10.0, 20.0, 30.0], dtype=DTYPE)
_register_case("ex007", {"a": a, "b": b})


In [ ]:
# Exercise 007: predict shapes first; do not use autograd.
# Define `ex007_a_shape`.
# Define `ex007_b_shape`.
# Define `ex007_a_aligned_shape`.
# Define `ex007_a_expanded_axes`.
# Define `ex007_b_aligned_shape`.
# Define `ex007_b_expanded_axes`.
# Define `ex007_compatible`.
# Define `ex007_out_shape`.
# Define `ex007_out` with PyTorch tensor operations.
# Write your work below, then run the supplied test cell.


In [ ]:
# Supplied test: expected shapes and values remain private.
_check_private_value("ex007_a_shape", "ex007", "a_shape")
_check_private_value("ex007_b_shape", "ex007", "b_shape")
_check_private_value("ex007_a_aligned_shape", "ex007", "a_aligned_shape")
_check_private_value("ex007_a_expanded_axes", "ex007", "a_expanded_axes")
_check_private_value("ex007_b_aligned_shape", "ex007", "b_aligned_shape")
_check_private_value("ex007_b_expanded_axes", "ex007", "b_expanded_axes")
_check_private_value("ex007_compatible", "ex007", "compatible")
_check_private_value("ex007_out_shape", "ex007", "out_shape")
_check_private_tensor("ex007_out", "ex007", "out")


### Exercise 008 — Explicit row vector

**Purpose:** Compare `(1, 3)` with the implicit `(3,)` row case.

**Visible inputs:** `a`, `b`. Run the fixture below and read its construction code; it deliberately prints no shape answers.

**Operation to reason about:** `a + b`

**Task:** Add the explicit one-row tensor to both rows.

**Ingredients:** The singleton row axis may expand.

**Required predictions and outputs:**

- `ex008_a_shape`: a literal Python shape tuple
- `ex008_b_shape`: a literal Python shape tuple
- `ex008_a_aligned_shape`: a literal Python shape tuple
- `ex008_a_expanded_axes`: a tuple of zero-based aligned axes
- `ex008_b_aligned_shape`: a literal Python shape tuple
- `ex008_b_expanded_axes`: a tuple of zero-based aligned axes
- `ex008_compatible`: a Python `bool`
- `ex008_out_shape`: a literal Python shape tuple
- `ex008_out`: the requested PyTorch tensor

**Order:** Fill every shape/alignment prediction before writing tensor operations.

**Next concept:** Explicit column vector


In [ ]:
# Supplied visible fixture: inspect these values, but predict shapes without printing them.
a = torch.arange(6, dtype=DTYPE).reshape(2, 3)
b = torch.tensor([[10.0, 20.0, 30.0]], dtype=DTYPE)
_register_case("ex008", {"a": a, "b": b})


In [ ]:
# Exercise 008: predict shapes first; do not use autograd.
# Define `ex008_a_shape`.
# Define `ex008_b_shape`.
# Define `ex008_a_aligned_shape`.
# Define `ex008_a_expanded_axes`.
# Define `ex008_b_aligned_shape`.
# Define `ex008_b_expanded_axes`.
# Define `ex008_compatible`.
# Define `ex008_out_shape`.
# Define `ex008_out` with PyTorch tensor operations.
# Write your work below, then run the supplied test cell.


In [ ]:
# Supplied test: expected shapes and values remain private.
_check_private_value("ex008_a_shape", "ex008", "a_shape")
_check_private_value("ex008_b_shape", "ex008", "b_shape")
_check_private_value("ex008_a_aligned_shape", "ex008", "a_aligned_shape")
_check_private_value("ex008_a_expanded_axes", "ex008", "a_expanded_axes")
_check_private_value("ex008_b_aligned_shape", "ex008", "b_aligned_shape")
_check_private_value("ex008_b_expanded_axes", "ex008", "b_expanded_axes")
_check_private_value("ex008_compatible", "ex008", "compatible")
_check_private_value("ex008_out_shape", "ex008", "out_shape")
_check_private_tensor("ex008_out", "ex008", "out")


### Exercise 009 — Explicit column vector

**Purpose:** Use `(2, 1)` to attach one value to each row.

**Visible inputs:** `a`, `b`. Run the fixture below and read its construction code; it deliberately prints no shape answers.

**Operation to reason about:** `a + b`

**Task:** Add one value across every column of each row.

**Ingredients:** The singleton column axis may expand.

**Required predictions and outputs:**

- `ex009_a_shape`: a literal Python shape tuple
- `ex009_b_shape`: a literal Python shape tuple
- `ex009_a_aligned_shape`: a literal Python shape tuple
- `ex009_a_expanded_axes`: a tuple of zero-based aligned axes
- `ex009_b_aligned_shape`: a literal Python shape tuple
- `ex009_b_expanded_axes`: a tuple of zero-based aligned axes
- `ex009_compatible`: a Python `bool`
- `ex009_out_shape`: a literal Python shape tuple
- `ex009_out`: the requested PyTorch tensor

**Order:** Fill every shape/alignment prediction before writing tensor operations.

**Next concept:** Outer addition grid


In [ ]:
# Supplied visible fixture: inspect these values, but predict shapes without printing them.
a = torch.arange(6, dtype=DTYPE).reshape(2, 3)
b = torch.tensor([[10.0], [20.0]], dtype=DTYPE)
_register_case("ex009", {"a": a, "b": b})


In [ ]:
# Exercise 009: predict shapes first; do not use autograd.
# Define `ex009_a_shape`.
# Define `ex009_b_shape`.
# Define `ex009_a_aligned_shape`.
# Define `ex009_a_expanded_axes`.
# Define `ex009_b_aligned_shape`.
# Define `ex009_b_expanded_axes`.
# Define `ex009_compatible`.
# Define `ex009_out_shape`.
# Define `ex009_out` with PyTorch tensor operations.
# Write your work below, then run the supplied test cell.


In [ ]:
# Supplied test: expected shapes and values remain private.
_check_private_value("ex009_a_shape", "ex009", "a_shape")
_check_private_value("ex009_b_shape", "ex009", "b_shape")
_check_private_value("ex009_a_aligned_shape", "ex009", "a_aligned_shape")
_check_private_value("ex009_a_expanded_axes", "ex009", "a_expanded_axes")
_check_private_value("ex009_b_aligned_shape", "ex009", "b_aligned_shape")
_check_private_value("ex009_b_expanded_axes", "ex009", "b_expanded_axes")
_check_private_value("ex009_compatible", "ex009", "compatible")
_check_private_value("ex009_out_shape", "ex009", "out_shape")
_check_private_tensor("ex009_out", "ex009", "out")


### Exercise 010 — Outer addition grid

**Purpose:** Combine a column and a row so both operands expand on different axes.

**Visible inputs:** `a`, `b`. Run the fixture below and read its construction code; it deliberately prints no shape answers.

**Operation to reason about:** `a + b`

**Task:** Create every row-value and column-value sum.

**Ingredients:** Audit which axis expands for each operand.

**Required predictions and outputs:**

- `ex010_a_shape`: a literal Python shape tuple
- `ex010_b_shape`: a literal Python shape tuple
- `ex010_a_aligned_shape`: a literal Python shape tuple
- `ex010_a_expanded_axes`: a tuple of zero-based aligned axes
- `ex010_b_aligned_shape`: a literal Python shape tuple
- `ex010_b_expanded_axes`: a tuple of zero-based aligned axes
- `ex010_compatible`: a Python `bool`
- `ex010_out_shape`: a literal Python shape tuple
- `ex010_out`: the requested PyTorch tensor

**Order:** Fill every shape/alignment prediction before writing tensor operations.

**Next concept:** Vector over the last rank-3 axis


In [ ]:
# Supplied visible fixture: inspect these values, but predict shapes without printing them.
a = torch.tensor([[1.0], [2.0]], dtype=DTYPE)
b = torch.tensor([[10.0, 20.0, 30.0]], dtype=DTYPE)
_register_case("ex010", {"a": a, "b": b})


In [ ]:
# Exercise 010: predict shapes first; do not use autograd.
# Define `ex010_a_shape`.
# Define `ex010_b_shape`.
# Define `ex010_a_aligned_shape`.
# Define `ex010_a_expanded_axes`.
# Define `ex010_b_aligned_shape`.
# Define `ex010_b_expanded_axes`.
# Define `ex010_compatible`.
# Define `ex010_out_shape`.
# Define `ex010_out` with PyTorch tensor operations.
# Write your work below, then run the supplied test cell.


In [ ]:
# Supplied test: expected shapes and values remain private.
_check_private_value("ex010_a_shape", "ex010", "a_shape")
_check_private_value("ex010_b_shape", "ex010", "b_shape")
_check_private_value("ex010_a_aligned_shape", "ex010", "a_aligned_shape")
_check_private_value("ex010_a_expanded_axes", "ex010", "a_expanded_axes")
_check_private_value("ex010_b_aligned_shape", "ex010", "b_aligned_shape")
_check_private_value("ex010_b_expanded_axes", "ex010", "b_expanded_axes")
_check_private_value("ex010_compatible", "ex010", "compatible")
_check_private_value("ex010_out_shape", "ex010", "out_shape")
_check_private_tensor("ex010_out", "ex010", "out")


## 2. Right alignment across ranks

PyTorch compares dimensions from right to left. Missing dimensions are conceptual leading ones, never trailing ones.


### Exercise 011 — Vector over the last rank-3 axis

**Purpose:** Right-align a feature vector beneath a batch tensor.

**Visible inputs:** `a`, `b`. Run the fixture below and read its construction code; it deliberately prints no shape answers.

**Operation to reason about:** `a + b`

**Task:** Add one value per final-axis feature.

**Ingredients:** Pad the vector with leading singleton dimensions mentally.

**Required predictions and outputs:**

- `ex011_a_shape`: a literal Python shape tuple
- `ex011_b_shape`: a literal Python shape tuple
- `ex011_a_aligned_shape`: a literal Python shape tuple
- `ex011_a_expanded_axes`: a tuple of zero-based aligned axes
- `ex011_b_aligned_shape`: a literal Python shape tuple
- `ex011_b_expanded_axes`: a tuple of zero-based aligned axes
- `ex011_compatible`: a Python `bool`
- `ex011_out_shape`: a literal Python shape tuple
- `ex011_out`: the requested PyTorch tensor

**Order:** Fill every shape/alignment prediction before writing tensor operations.

**Next concept:** Matrix over rank 3


In [ ]:
# Supplied visible fixture: inspect these values, but predict shapes without printing them.
a = torch.arange(24, dtype=DTYPE).reshape(2, 3, 4)
b = torch.tensor([1.0, 2.0, 3.0, 4.0], dtype=DTYPE)
_register_case("ex011", {"a": a, "b": b})


In [ ]:
# Exercise 011: predict shapes first; do not use autograd.
# Define `ex011_a_shape`.
# Define `ex011_b_shape`.
# Define `ex011_a_aligned_shape`.
# Define `ex011_a_expanded_axes`.
# Define `ex011_b_aligned_shape`.
# Define `ex011_b_expanded_axes`.
# Define `ex011_compatible`.
# Define `ex011_out_shape`.
# Define `ex011_out` with PyTorch tensor operations.
# Write your work below, then run the supplied test cell.


In [ ]:
# Supplied test: expected shapes and values remain private.
_check_private_value("ex011_a_shape", "ex011", "a_shape")
_check_private_value("ex011_b_shape", "ex011", "b_shape")
_check_private_value("ex011_a_aligned_shape", "ex011", "a_aligned_shape")
_check_private_value("ex011_a_expanded_axes", "ex011", "a_expanded_axes")
_check_private_value("ex011_b_aligned_shape", "ex011", "b_aligned_shape")
_check_private_value("ex011_b_expanded_axes", "ex011", "b_expanded_axes")
_check_private_value("ex011_compatible", "ex011", "compatible")
_check_private_value("ex011_out_shape", "ex011", "out_shape")
_check_private_tensor("ex011_out", "ex011", "out")


### Exercise 012 — Matrix over rank 3

**Purpose:** Reuse one `(3, 4)` plane across a leading batch axis.

**Visible inputs:** `a`, `b`. Run the fixture below and read its construction code; it deliberately prints no shape answers.

**Operation to reason about:** `a - b`

**Task:** Subtract the same matrix from each batch plane.

**Ingredients:** Right alignment treats the missing leading axis as size one.

**Required predictions and outputs:**

- `ex012_a_shape`: a literal Python shape tuple
- `ex012_b_shape`: a literal Python shape tuple
- `ex012_a_aligned_shape`: a literal Python shape tuple
- `ex012_a_expanded_axes`: a tuple of zero-based aligned axes
- `ex012_b_aligned_shape`: a literal Python shape tuple
- `ex012_b_expanded_axes`: a tuple of zero-based aligned axes
- `ex012_compatible`: a Python `bool`
- `ex012_out_shape`: a literal Python shape tuple
- `ex012_out`: the requested PyTorch tensor

**Order:** Fill every shape/alignment prediction before writing tensor operations.

**Next concept:** One row over rank 3


In [ ]:
# Supplied visible fixture: inspect these values, but predict shapes without printing them.
a = torch.arange(24, dtype=DTYPE).reshape(2, 3, 4)
b = torch.arange(12, dtype=DTYPE).reshape(3, 4)
_register_case("ex012", {"a": a, "b": b})


In [ ]:
# Exercise 012: predict shapes first; do not use autograd.
# Define `ex012_a_shape`.
# Define `ex012_b_shape`.
# Define `ex012_a_aligned_shape`.
# Define `ex012_a_expanded_axes`.
# Define `ex012_b_aligned_shape`.
# Define `ex012_b_expanded_axes`.
# Define `ex012_compatible`.
# Define `ex012_out_shape`.
# Define `ex012_out` with PyTorch tensor operations.
# Write your work below, then run the supplied test cell.


In [ ]:
# Supplied test: expected shapes and values remain private.
_check_private_value("ex012_a_shape", "ex012", "a_shape")
_check_private_value("ex012_b_shape", "ex012", "b_shape")
_check_private_value("ex012_a_aligned_shape", "ex012", "a_aligned_shape")
_check_private_value("ex012_a_expanded_axes", "ex012", "a_expanded_axes")
_check_private_value("ex012_b_aligned_shape", "ex012", "b_aligned_shape")
_check_private_value("ex012_b_expanded_axes", "ex012", "b_expanded_axes")
_check_private_value("ex012_compatible", "ex012", "compatible")
_check_private_value("ex012_out_shape", "ex012", "out_shape")
_check_private_tensor("ex012_out", "ex012", "out")


### Exercise 013 — One row over rank 3

**Purpose:** Expand one explicit row across batch and middle axes.

**Visible inputs:** `a`, `b`. Run the fixture below and read its construction code; it deliberately prints no shape answers.

**Operation to reason about:** `a * b`

**Task:** Scale every last-axis feature.

**Ingredients:** Left-pad `(1, 4)` before comparing all three axes.

**Required predictions and outputs:**

- `ex013_a_shape`: a literal Python shape tuple
- `ex013_b_shape`: a literal Python shape tuple
- `ex013_a_aligned_shape`: a literal Python shape tuple
- `ex013_a_expanded_axes`: a tuple of zero-based aligned axes
- `ex013_b_aligned_shape`: a literal Python shape tuple
- `ex013_b_expanded_axes`: a tuple of zero-based aligned axes
- `ex013_compatible`: a Python `bool`
- `ex013_out_shape`: a literal Python shape tuple
- `ex013_out`: the requested PyTorch tensor

**Order:** Fill every shape/alignment prediction before writing tensor operations.

**Next concept:** One column over rank 3


In [ ]:
# Supplied visible fixture: inspect these values, but predict shapes without printing them.
a = torch.arange(24, dtype=DTYPE).reshape(2, 3, 4)
b = torch.tensor([[0.5, 1.0, 1.5, 2.0]], dtype=DTYPE)
_register_case("ex013", {"a": a, "b": b})


In [ ]:
# Exercise 013: predict shapes first; do not use autograd.
# Define `ex013_a_shape`.
# Define `ex013_b_shape`.
# Define `ex013_a_aligned_shape`.
# Define `ex013_a_expanded_axes`.
# Define `ex013_b_aligned_shape`.
# Define `ex013_b_expanded_axes`.
# Define `ex013_compatible`.
# Define `ex013_out_shape`.
# Define `ex013_out` with PyTorch tensor operations.
# Write your work below, then run the supplied test cell.


In [ ]:
# Supplied test: expected shapes and values remain private.
_check_private_value("ex013_a_shape", "ex013", "a_shape")
_check_private_value("ex013_b_shape", "ex013", "b_shape")
_check_private_value("ex013_a_aligned_shape", "ex013", "a_aligned_shape")
_check_private_value("ex013_a_expanded_axes", "ex013", "a_expanded_axes")
_check_private_value("ex013_b_aligned_shape", "ex013", "b_aligned_shape")
_check_private_value("ex013_b_expanded_axes", "ex013", "b_expanded_axes")
_check_private_value("ex013_compatible", "ex013", "compatible")
_check_private_value("ex013_out_shape", "ex013", "out_shape")
_check_private_tensor("ex013_out", "ex013", "out")


### Exercise 014 — One column over rank 3

**Purpose:** Use `(3, 1)` to preserve the middle axis and expand the final axis.

**Visible inputs:** `a`, `b`. Run the fixture below and read its construction code; it deliberately prints no shape answers.

**Operation to reason about:** `a + b`

**Task:** Add one value per middle-axis position.

**Ingredients:** After left-padding, identify both expanded axes.

**Required predictions and outputs:**

- `ex014_a_shape`: a literal Python shape tuple
- `ex014_b_shape`: a literal Python shape tuple
- `ex014_a_aligned_shape`: a literal Python shape tuple
- `ex014_a_expanded_axes`: a tuple of zero-based aligned axes
- `ex014_b_aligned_shape`: a literal Python shape tuple
- `ex014_b_expanded_axes`: a tuple of zero-based aligned axes
- `ex014_compatible`: a Python `bool`
- `ex014_out_shape`: a literal Python shape tuple
- `ex014_out`: the requested PyTorch tensor

**Order:** Fill every shape/alignment prediction before writing tensor operations.

**Next concept:** Middle-axis parameter


In [ ]:
# Supplied visible fixture: inspect these values, but predict shapes without printing them.
a = torch.arange(24, dtype=DTYPE).reshape(2, 3, 4)
b = torch.tensor([[1.0], [2.0], [3.0]], dtype=DTYPE)
_register_case("ex014", {"a": a, "b": b})


In [ ]:
# Exercise 014: predict shapes first; do not use autograd.
# Define `ex014_a_shape`.
# Define `ex014_b_shape`.
# Define `ex014_a_aligned_shape`.
# Define `ex014_a_expanded_axes`.
# Define `ex014_b_aligned_shape`.
# Define `ex014_b_expanded_axes`.
# Define `ex014_compatible`.
# Define `ex014_out_shape`.
# Define `ex014_out` with PyTorch tensor operations.
# Write your work below, then run the supplied test cell.


In [ ]:
# Supplied test: expected shapes and values remain private.
_check_private_value("ex014_a_shape", "ex014", "a_shape")
_check_private_value("ex014_b_shape", "ex014", "b_shape")
_check_private_value("ex014_a_aligned_shape", "ex014", "a_aligned_shape")
_check_private_value("ex014_a_expanded_axes", "ex014", "a_expanded_axes")
_check_private_value("ex014_b_aligned_shape", "ex014", "b_aligned_shape")
_check_private_value("ex014_b_expanded_axes", "ex014", "b_expanded_axes")
_check_private_value("ex014_compatible", "ex014", "compatible")
_check_private_value("ex014_out_shape", "ex014", "out_shape")
_check_private_tensor("ex014_out", "ex014", "out")


### Exercise 015 — Middle-axis parameter

**Purpose:** Read `(1, 3, 1)` as one parameter per middle-axis entry.

**Visible inputs:** `a`, `b`. Run the fixture below and read its construction code; it deliberately prints no shape answers.

**Operation to reason about:** `a + b`

**Task:** Add the middle-axis parameters throughout the tensor.

**Ingredients:** Name each axis before applying the size-one rule.

**Required predictions and outputs:**

- `ex015_a_shape`: a literal Python shape tuple
- `ex015_b_shape`: a literal Python shape tuple
- `ex015_a_aligned_shape`: a literal Python shape tuple
- `ex015_a_expanded_axes`: a tuple of zero-based aligned axes
- `ex015_b_aligned_shape`: a literal Python shape tuple
- `ex015_b_expanded_axes`: a tuple of zero-based aligned axes
- `ex015_compatible`: a Python `bool`
- `ex015_out_shape`: a literal Python shape tuple
- `ex015_out`: the requested PyTorch tensor

**Order:** Fill every shape/alignment prediction before writing tensor operations.

**Next concept:** Leading-axis parameter


In [ ]:
# Supplied visible fixture: inspect these values, but predict shapes without printing them.
a = torch.arange(24, dtype=DTYPE).reshape(2, 3, 4)
b = torch.tensor([[[1.0], [10.0], [100.0]]], dtype=DTYPE)
_register_case("ex015", {"a": a, "b": b})


In [ ]:
# Exercise 015: predict shapes first; do not use autograd.
# Define `ex015_a_shape`.
# Define `ex015_b_shape`.
# Define `ex015_a_aligned_shape`.
# Define `ex015_a_expanded_axes`.
# Define `ex015_b_aligned_shape`.
# Define `ex015_b_expanded_axes`.
# Define `ex015_compatible`.
# Define `ex015_out_shape`.
# Define `ex015_out` with PyTorch tensor operations.
# Write your work below, then run the supplied test cell.


In [ ]:
# Supplied test: expected shapes and values remain private.
_check_private_value("ex015_a_shape", "ex015", "a_shape")
_check_private_value("ex015_b_shape", "ex015", "b_shape")
_check_private_value("ex015_a_aligned_shape", "ex015", "a_aligned_shape")
_check_private_value("ex015_a_expanded_axes", "ex015", "a_expanded_axes")
_check_private_value("ex015_b_aligned_shape", "ex015", "b_aligned_shape")
_check_private_value("ex015_b_expanded_axes", "ex015", "b_expanded_axes")
_check_private_value("ex015_compatible", "ex015", "compatible")
_check_private_value("ex015_out_shape", "ex015", "out_shape")
_check_private_tensor("ex015_out", "ex015", "out")


### Exercise 016 — Leading-axis parameter

**Purpose:** Read `(2, 1, 1)` as one scalar per batch item.

**Visible inputs:** `a`, `b`. Run the fixture below and read its construction code; it deliberately prints no shape answers.

**Operation to reason about:** `a * b`

**Task:** Scale each batch item by its own scalar.

**Ingredients:** The final two singleton axes expand.

**Required predictions and outputs:**

- `ex016_a_shape`: a literal Python shape tuple
- `ex016_b_shape`: a literal Python shape tuple
- `ex016_a_aligned_shape`: a literal Python shape tuple
- `ex016_a_expanded_axes`: a tuple of zero-based aligned axes
- `ex016_b_aligned_shape`: a literal Python shape tuple
- `ex016_b_expanded_axes`: a tuple of zero-based aligned axes
- `ex016_compatible`: a Python `bool`
- `ex016_out_shape`: a literal Python shape tuple
- `ex016_out`: the requested PyTorch tensor

**Order:** Fill every shape/alignment prediction before writing tensor operations.

**Next concept:** Scalar over rank 4


In [ ]:
# Supplied visible fixture: inspect these values, but predict shapes without printing them.
a = torch.arange(24, dtype=DTYPE).reshape(2, 3, 4)
b = torch.tensor([[[2.0]], [[-1.0]]], dtype=DTYPE)
_register_case("ex016", {"a": a, "b": b})


In [ ]:
# Exercise 016: predict shapes first; do not use autograd.
# Define `ex016_a_shape`.
# Define `ex016_b_shape`.
# Define `ex016_a_aligned_shape`.
# Define `ex016_a_expanded_axes`.
# Define `ex016_b_aligned_shape`.
# Define `ex016_b_expanded_axes`.
# Define `ex016_compatible`.
# Define `ex016_out_shape`.
# Define `ex016_out` with PyTorch tensor operations.
# Write your work below, then run the supplied test cell.


In [ ]:
# Supplied test: expected shapes and values remain private.
_check_private_value("ex016_a_shape", "ex016", "a_shape")
_check_private_value("ex016_b_shape", "ex016", "b_shape")
_check_private_value("ex016_a_aligned_shape", "ex016", "a_aligned_shape")
_check_private_value("ex016_a_expanded_axes", "ex016", "a_expanded_axes")
_check_private_value("ex016_b_aligned_shape", "ex016", "b_aligned_shape")
_check_private_value("ex016_b_expanded_axes", "ex016", "b_expanded_axes")
_check_private_value("ex016_compatible", "ex016", "compatible")
_check_private_value("ex016_out_shape", "ex016", "out_shape")
_check_private_tensor("ex016_out", "ex016", "out")


### Exercise 017 — Scalar over rank 4

**Purpose:** Mentally align a rank-zero tensor against four axes.

**Visible inputs:** `a`, `b`. Run the fixture below and read its construction code; it deliberately prints no shape answers.

**Operation to reason about:** `a / b`

**Task:** Divide every entry by one scalar.

**Ingredients:** A scalar's conceptual aligned shape is all ones.

**Required predictions and outputs:**

- `ex017_a_shape`: a literal Python shape tuple
- `ex017_b_shape`: a literal Python shape tuple
- `ex017_a_aligned_shape`: a literal Python shape tuple
- `ex017_a_expanded_axes`: a tuple of zero-based aligned axes
- `ex017_b_aligned_shape`: a literal Python shape tuple
- `ex017_b_expanded_axes`: a tuple of zero-based aligned axes
- `ex017_compatible`: a Python `bool`
- `ex017_out_shape`: a literal Python shape tuple
- `ex017_out`: the requested PyTorch tensor

**Order:** Fill every shape/alignment prediction before writing tensor operations.

**Next concept:** Feature vector over rank 4


In [ ]:
# Supplied visible fixture: inspect these values, but predict shapes without printing them.
a = torch.arange(48, dtype=DTYPE).reshape(2, 3, 4, 2)
b = torch.tensor(2.0, dtype=DTYPE)
_register_case("ex017", {"a": a, "b": b})


In [ ]:
# Exercise 017: predict shapes first; do not use autograd.
# Define `ex017_a_shape`.
# Define `ex017_b_shape`.
# Define `ex017_a_aligned_shape`.
# Define `ex017_a_expanded_axes`.
# Define `ex017_b_aligned_shape`.
# Define `ex017_b_expanded_axes`.
# Define `ex017_compatible`.
# Define `ex017_out_shape`.
# Define `ex017_out` with PyTorch tensor operations.
# Write your work below, then run the supplied test cell.


In [ ]:
# Supplied test: expected shapes and values remain private.
_check_private_value("ex017_a_shape", "ex017", "a_shape")
_check_private_value("ex017_b_shape", "ex017", "b_shape")
_check_private_value("ex017_a_aligned_shape", "ex017", "a_aligned_shape")
_check_private_value("ex017_a_expanded_axes", "ex017", "a_expanded_axes")
_check_private_value("ex017_b_aligned_shape", "ex017", "b_aligned_shape")
_check_private_value("ex017_b_expanded_axes", "ex017", "b_expanded_axes")
_check_private_value("ex017_compatible", "ex017", "compatible")
_check_private_value("ex017_out_shape", "ex017", "out_shape")
_check_private_tensor("ex017_out", "ex017", "out")


### Exercise 018 — Feature vector over rank 4

**Purpose:** Reuse a length-five feature vector over three leading axes.

**Visible inputs:** `a`, `b`. Run the fixture below and read its construction code; it deliberately prints no shape answers.

**Operation to reason about:** `a * b`

**Task:** Scale the final feature axis.

**Ingredients:** Right alignment always starts at the final axis.

**Required predictions and outputs:**

- `ex018_a_shape`: a literal Python shape tuple
- `ex018_b_shape`: a literal Python shape tuple
- `ex018_a_aligned_shape`: a literal Python shape tuple
- `ex018_a_expanded_axes`: a tuple of zero-based aligned axes
- `ex018_b_aligned_shape`: a literal Python shape tuple
- `ex018_b_expanded_axes`: a tuple of zero-based aligned axes
- `ex018_compatible`: a Python `bool`
- `ex018_out_shape`: a literal Python shape tuple
- `ex018_out`: the requested PyTorch tensor

**Order:** Fill every shape/alignment prediction before writing tensor operations.

**Next concept:** Channel-like tensor over rank 4


In [ ]:
# Supplied visible fixture: inspect these values, but predict shapes without printing them.
a = torch.arange(120, dtype=DTYPE).reshape(2, 3, 4, 5)
b = torch.linspace(1.0, 2.0, steps=5, dtype=DTYPE)
_register_case("ex018", {"a": a, "b": b})


In [ ]:
# Exercise 018: predict shapes first; do not use autograd.
# Define `ex018_a_shape`.
# Define `ex018_b_shape`.
# Define `ex018_a_aligned_shape`.
# Define `ex018_a_expanded_axes`.
# Define `ex018_b_aligned_shape`.
# Define `ex018_b_expanded_axes`.
# Define `ex018_compatible`.
# Define `ex018_out_shape`.
# Define `ex018_out` with PyTorch tensor operations.
# Write your work below, then run the supplied test cell.


In [ ]:
# Supplied test: expected shapes and values remain private.
_check_private_value("ex018_a_shape", "ex018", "a_shape")
_check_private_value("ex018_b_shape", "ex018", "b_shape")
_check_private_value("ex018_a_aligned_shape", "ex018", "a_aligned_shape")
_check_private_value("ex018_a_expanded_axes", "ex018", "a_expanded_axes")
_check_private_value("ex018_b_aligned_shape", "ex018", "b_aligned_shape")
_check_private_value("ex018_b_expanded_axes", "ex018", "b_expanded_axes")
_check_private_value("ex018_compatible", "ex018", "compatible")
_check_private_value("ex018_out_shape", "ex018", "out_shape")
_check_private_tensor("ex018_out", "ex018", "out")


### Exercise 019 — Channel-like tensor over rank 4

**Purpose:** Align `(3, 1, 1)` with the final three axes of rank 4.

**Visible inputs:** `a`, `b`. Run the fixture below and read its construction code; it deliberately prints no shape answers.

**Operation to reason about:** `a + b`

**Task:** Add one value per channel-like axis.

**Ingredients:** A missing leading axis is treated as one.

**Required predictions and outputs:**

- `ex019_a_shape`: a literal Python shape tuple
- `ex019_b_shape`: a literal Python shape tuple
- `ex019_a_aligned_shape`: a literal Python shape tuple
- `ex019_a_expanded_axes`: a tuple of zero-based aligned axes
- `ex019_b_aligned_shape`: a literal Python shape tuple
- `ex019_b_expanded_axes`: a tuple of zero-based aligned axes
- `ex019_compatible`: a Python `bool`
- `ex019_out_shape`: a literal Python shape tuple
- `ex019_out`: the requested PyTorch tensor

**Order:** Fill every shape/alignment prediction before writing tensor operations.

**Next concept:** Two-way rank-4 expansion


In [ ]:
# Supplied visible fixture: inspect these values, but predict shapes without printing them.
a = torch.arange(120, dtype=DTYPE).reshape(2, 3, 4, 5)
b = torch.tensor([[[1.0]], [[2.0]], [[3.0]]], dtype=DTYPE)
_register_case("ex019", {"a": a, "b": b})


In [ ]:
# Exercise 019: predict shapes first; do not use autograd.
# Define `ex019_a_shape`.
# Define `ex019_b_shape`.
# Define `ex019_a_aligned_shape`.
# Define `ex019_a_expanded_axes`.
# Define `ex019_b_aligned_shape`.
# Define `ex019_b_expanded_axes`.
# Define `ex019_compatible`.
# Define `ex019_out_shape`.
# Define `ex019_out` with PyTorch tensor operations.
# Write your work below, then run the supplied test cell.


In [ ]:
# Supplied test: expected shapes and values remain private.
_check_private_value("ex019_a_shape", "ex019", "a_shape")
_check_private_value("ex019_b_shape", "ex019", "b_shape")
_check_private_value("ex019_a_aligned_shape", "ex019", "a_aligned_shape")
_check_private_value("ex019_a_expanded_axes", "ex019", "a_expanded_axes")
_check_private_value("ex019_b_aligned_shape", "ex019", "b_aligned_shape")
_check_private_value("ex019_b_expanded_axes", "ex019", "b_expanded_axes")
_check_private_value("ex019_compatible", "ex019", "compatible")
_check_private_value("ex019_out_shape", "ex019", "out_shape")
_check_private_tensor("ex019_out", "ex019", "out")


### Exercise 020 — Two-way rank-4 expansion

**Purpose:** Track two operands that each contribute different non-singleton axes.

**Visible inputs:** `a`, `b`. Run the fixture below and read its construction code; it deliberately prints no shape answers.

**Operation to reason about:** `a + b`

**Task:** Create the combined rank-4 grid.

**Ingredients:** Compare every right-aligned axis and record both expansion sets.

**Required predictions and outputs:**

- `ex020_a_shape`: a literal Python shape tuple
- `ex020_b_shape`: a literal Python shape tuple
- `ex020_a_aligned_shape`: a literal Python shape tuple
- `ex020_a_expanded_axes`: a tuple of zero-based aligned axes
- `ex020_b_aligned_shape`: a literal Python shape tuple
- `ex020_b_expanded_axes`: a tuple of zero-based aligned axes
- `ex020_compatible`: a Python `bool`
- `ex020_out_shape`: a literal Python shape tuple
- `ex020_out`: the requested PyTorch tensor

**Order:** Fill every shape/alignment prediction before writing tensor operations.

**Next concept:** Turn row weights into a column


In [ ]:
# Supplied visible fixture: inspect these values, but predict shapes without printing them.
a = torch.arange(8, dtype=DTYPE).reshape(2, 1, 4, 1)
b = torch.arange(15, dtype=DTYPE).reshape(1, 3, 1, 5)
_register_case("ex020", {"a": a, "b": b})


In [ ]:
# Exercise 020: predict shapes first; do not use autograd.
# Define `ex020_a_shape`.
# Define `ex020_b_shape`.
# Define `ex020_a_aligned_shape`.
# Define `ex020_a_expanded_axes`.
# Define `ex020_b_aligned_shape`.
# Define `ex020_b_expanded_axes`.
# Define `ex020_compatible`.
# Define `ex020_out_shape`.
# Define `ex020_out` with PyTorch tensor operations.
# Write your work below, then run the supplied test cell.


In [ ]:
# Supplied test: expected shapes and values remain private.
_check_private_value("ex020_a_shape", "ex020", "a_shape")
_check_private_value("ex020_b_shape", "ex020", "b_shape")
_check_private_value("ex020_a_aligned_shape", "ex020", "a_aligned_shape")
_check_private_value("ex020_a_expanded_axes", "ex020", "a_expanded_axes")
_check_private_value("ex020_b_aligned_shape", "ex020", "b_aligned_shape")
_check_private_value("ex020_b_expanded_axes", "ex020", "b_expanded_axes")
_check_private_value("ex020_compatible", "ex020", "compatible")
_check_private_value("ex020_out_shape", "ex020", "out_shape")
_check_private_tensor("ex020_out", "ex020", "out")


## 3. Inserting singleton axes deliberately

Use `None` or `unsqueeze` to place a compact tensor's semantic axis where you intend. Inserting an axis changes only shape metadata; the following elementwise operation performs broadcasting.


### Exercise 021 — Turn row weights into a column

**Purpose:** Use indexing syntax to attach each length-two value to one matrix row.

**Visible inputs:** `rows`, `grid`. Run the fixture below and read its construction code; it deliberately prints no shape answers.

**Operation to reason about:** Create `rows_col` from `rows` with one inserted axis, then multiply it across `grid`.

**Task:** Create `rows_col` from `rows` with one inserted axis, then multiply it across `grid`.

**Ingredients:** Use `:` to keep the existing axis and `None` to insert a trailing singleton axis.

**Required predictions and outputs:**

- `ex021_rows_shape`: a literal Python shape tuple
- `ex021_grid_shape`: a literal Python shape tuple
- `ex021_rows_col_shape`: a literal Python shape tuple
- `ex021_rows_col_aligned_shape`: a literal Python shape tuple
- `ex021_rows_col_expanded_axes`: a tuple of zero-based aligned axes
- `ex021_grid_aligned_shape`: a literal Python shape tuple
- `ex021_grid_expanded_axes`: a tuple of zero-based aligned axes
- `ex021_compatible`: a Python `bool`
- `ex021_out_shape`: a literal Python shape tuple
- `ex021_rows_col`: the requested PyTorch tensor
- `ex021_out`: the requested PyTorch tensor

**Order:** Fill every shape/alignment prediction before writing tensor operations.

**Next concept:** Turn column weights into a row


In [ ]:
# Supplied visible fixture: inspect these values, but predict shapes without printing them.
rows = torch.tensor([2.0, -1.0], dtype=DTYPE)
grid = torch.ones((2, 3), dtype=DTYPE)
_register_case("ex021", {"rows": rows, "grid": grid})


In [ ]:
# Exercise 021: predict shapes first; do not use autograd.
# Define `ex021_rows_shape`.
# Define `ex021_grid_shape`.
# Define `ex021_rows_col_shape`.
# Define `ex021_rows_col_aligned_shape`.
# Define `ex021_rows_col_expanded_axes`.
# Define `ex021_grid_aligned_shape`.
# Define `ex021_grid_expanded_axes`.
# Define `ex021_compatible`.
# Define `ex021_out_shape`.
# Define `ex021_rows_col` with PyTorch tensor operations.
# Define `ex021_out` with PyTorch tensor operations.
# Write your work below, then run the supplied test cell.


In [ ]:
# Supplied test: expected shapes and values remain private.
_check_private_value("ex021_rows_shape", "ex021", "rows_shape")
_check_private_value("ex021_grid_shape", "ex021", "grid_shape")
_check_private_value("ex021_rows_col_shape", "ex021", "rows_col_shape")
_check_private_value("ex021_rows_col_aligned_shape", "ex021", "rows_col_aligned_shape")
_check_private_value("ex021_rows_col_expanded_axes", "ex021", "rows_col_expanded_axes")
_check_private_value("ex021_grid_aligned_shape", "ex021", "grid_aligned_shape")
_check_private_value("ex021_grid_expanded_axes", "ex021", "grid_expanded_axes")
_check_private_value("ex021_compatible", "ex021", "compatible")
_check_private_value("ex021_out_shape", "ex021", "out_shape")
_check_private_tensor("ex021_rows_col", "ex021", "rows_col")
_check_private_tensor("ex021_out", "ex021", "out")


### Exercise 022 — Turn column weights into a row

**Purpose:** Insert a leading singleton axis before combining three column values with a matrix.

**Visible inputs:** `cols`, `grid`. Run the fixture below and read its construction code; it deliberately prints no shape answers.

**Operation to reason about:** Create `cols_row` and add it across both rows.

**Task:** Create `cols_row` and add it across both rows.

**Ingredients:** Use `None` before `:` to insert a leading axis.

**Required predictions and outputs:**

- `ex022_cols_shape`: a literal Python shape tuple
- `ex022_grid_shape`: a literal Python shape tuple
- `ex022_cols_row_shape`: a literal Python shape tuple
- `ex022_cols_row_aligned_shape`: a literal Python shape tuple
- `ex022_cols_row_expanded_axes`: a tuple of zero-based aligned axes
- `ex022_grid_aligned_shape`: a literal Python shape tuple
- `ex022_grid_expanded_axes`: a tuple of zero-based aligned axes
- `ex022_compatible`: a Python `bool`
- `ex022_out_shape`: a literal Python shape tuple
- `ex022_cols_row`: the requested PyTorch tensor
- `ex022_out`: the requested PyTorch tensor

**Order:** Fill every shape/alignment prediction before writing tensor operations.

**Next concept:** Feature vector into rank 3


In [ ]:
# Supplied visible fixture: inspect these values, but predict shapes without printing them.
cols = torch.tensor([1.0, 10.0, 100.0], dtype=DTYPE)
grid = torch.zeros((2, 3), dtype=DTYPE)
_register_case("ex022", {"cols": cols, "grid": grid})


In [ ]:
# Exercise 022: predict shapes first; do not use autograd.
# Define `ex022_cols_shape`.
# Define `ex022_grid_shape`.
# Define `ex022_cols_row_shape`.
# Define `ex022_cols_row_aligned_shape`.
# Define `ex022_cols_row_expanded_axes`.
# Define `ex022_grid_aligned_shape`.
# Define `ex022_grid_expanded_axes`.
# Define `ex022_compatible`.
# Define `ex022_out_shape`.
# Define `ex022_cols_row` with PyTorch tensor operations.
# Define `ex022_out` with PyTorch tensor operations.
# Write your work below, then run the supplied test cell.


In [ ]:
# Supplied test: expected shapes and values remain private.
_check_private_value("ex022_cols_shape", "ex022", "cols_shape")
_check_private_value("ex022_grid_shape", "ex022", "grid_shape")
_check_private_value("ex022_cols_row_shape", "ex022", "cols_row_shape")
_check_private_value("ex022_cols_row_aligned_shape", "ex022", "cols_row_aligned_shape")
_check_private_value("ex022_cols_row_expanded_axes", "ex022", "cols_row_expanded_axes")
_check_private_value("ex022_grid_aligned_shape", "ex022", "grid_aligned_shape")
_check_private_value("ex022_grid_expanded_axes", "ex022", "grid_expanded_axes")
_check_private_value("ex022_compatible", "ex022", "compatible")
_check_private_value("ex022_out_shape", "ex022", "out_shape")
_check_private_tensor("ex022_cols_row", "ex022", "cols_row")
_check_private_tensor("ex022_out", "ex022", "out")


### Exercise 023 — Feature vector into rank 3

**Purpose:** Insert two leading axes so four features align with the final tensor axis.

**Visible inputs:** `features`, `cube`. Run the fixture below and read its construction code; it deliberately prints no shape answers.

**Operation to reason about:** Create `features_view`, then add it to `cube`.

**Task:** Create `features_view`, then add it to `cube`.

**Ingredients:** Preserve the feature axis and insert two axes before it.

**Required predictions and outputs:**

- `ex023_features_shape`: a literal Python shape tuple
- `ex023_cube_shape`: a literal Python shape tuple
- `ex023_features_view_shape`: a literal Python shape tuple
- `ex023_features_view_aligned_shape`: a literal Python shape tuple
- `ex023_features_view_expanded_axes`: a tuple of zero-based aligned axes
- `ex023_cube_aligned_shape`: a literal Python shape tuple
- `ex023_cube_expanded_axes`: a tuple of zero-based aligned axes
- `ex023_compatible`: a Python `bool`
- `ex023_out_shape`: a literal Python shape tuple
- `ex023_features_view`: the requested PyTorch tensor
- `ex023_out`: the requested PyTorch tensor

**Order:** Fill every shape/alignment prediction before writing tensor operations.

**Next concept:** Batch vector into rank 3


In [ ]:
# Supplied visible fixture: inspect these values, but predict shapes without printing them.
features = torch.tensor([1.0, 2.0, 3.0, 4.0], dtype=DTYPE)
cube = torch.zeros((2, 3, 4), dtype=DTYPE)
_register_case("ex023", {"features": features, "cube": cube})


In [ ]:
# Exercise 023: predict shapes first; do not use autograd.
# Define `ex023_features_shape`.
# Define `ex023_cube_shape`.
# Define `ex023_features_view_shape`.
# Define `ex023_features_view_aligned_shape`.
# Define `ex023_features_view_expanded_axes`.
# Define `ex023_cube_aligned_shape`.
# Define `ex023_cube_expanded_axes`.
# Define `ex023_compatible`.
# Define `ex023_out_shape`.
# Define `ex023_features_view` with PyTorch tensor operations.
# Define `ex023_out` with PyTorch tensor operations.
# Write your work below, then run the supplied test cell.


In [ ]:
# Supplied test: expected shapes and values remain private.
_check_private_value("ex023_features_shape", "ex023", "features_shape")
_check_private_value("ex023_cube_shape", "ex023", "cube_shape")
_check_private_value("ex023_features_view_shape", "ex023", "features_view_shape")
_check_private_value("ex023_features_view_aligned_shape", "ex023", "features_view_aligned_shape")
_check_private_value("ex023_features_view_expanded_axes", "ex023", "features_view_expanded_axes")
_check_private_value("ex023_cube_aligned_shape", "ex023", "cube_aligned_shape")
_check_private_value("ex023_cube_expanded_axes", "ex023", "cube_expanded_axes")
_check_private_value("ex023_compatible", "ex023", "compatible")
_check_private_value("ex023_out_shape", "ex023", "out_shape")
_check_private_tensor("ex023_features_view", "ex023", "features_view")
_check_private_tensor("ex023_out", "ex023", "out")


### Exercise 024 — Batch vector into rank 3

**Purpose:** Insert two trailing axes so one value belongs to each batch item.

**Visible inputs:** `batch_values`, `cube`. Run the fixture below and read its construction code; it deliberately prints no shape answers.

**Operation to reason about:** Create `batch_view`, then scale each batch item.

**Task:** Create `batch_view`, then scale each batch item.

**Ingredients:** Keep the batch axis first and add two singleton axes after it.

**Required predictions and outputs:**

- `ex024_batch_values_shape`: a literal Python shape tuple
- `ex024_cube_shape`: a literal Python shape tuple
- `ex024_batch_view_shape`: a literal Python shape tuple
- `ex024_batch_view_aligned_shape`: a literal Python shape tuple
- `ex024_batch_view_expanded_axes`: a tuple of zero-based aligned axes
- `ex024_cube_aligned_shape`: a literal Python shape tuple
- `ex024_cube_expanded_axes`: a tuple of zero-based aligned axes
- `ex024_compatible`: a Python `bool`
- `ex024_out_shape`: a literal Python shape tuple
- `ex024_batch_view`: the requested PyTorch tensor
- `ex024_out`: the requested PyTorch tensor

**Order:** Fill every shape/alignment prediction before writing tensor operations.

**Next concept:** Channel vector into images


In [ ]:
# Supplied visible fixture: inspect these values, but predict shapes without printing them.
batch_values = torch.tensor([2.0, -1.0], dtype=DTYPE)
cube = torch.ones((2, 3, 4), dtype=DTYPE)
_register_case("ex024", {"batch_values": batch_values, "cube": cube})


In [ ]:
# Exercise 024: predict shapes first; do not use autograd.
# Define `ex024_batch_values_shape`.
# Define `ex024_cube_shape`.
# Define `ex024_batch_view_shape`.
# Define `ex024_batch_view_aligned_shape`.
# Define `ex024_batch_view_expanded_axes`.
# Define `ex024_cube_aligned_shape`.
# Define `ex024_cube_expanded_axes`.
# Define `ex024_compatible`.
# Define `ex024_out_shape`.
# Define `ex024_batch_view` with PyTorch tensor operations.
# Define `ex024_out` with PyTorch tensor operations.
# Write your work below, then run the supplied test cell.


In [ ]:
# Supplied test: expected shapes and values remain private.
_check_private_value("ex024_batch_values_shape", "ex024", "batch_values_shape")
_check_private_value("ex024_cube_shape", "ex024", "cube_shape")
_check_private_value("ex024_batch_view_shape", "ex024", "batch_view_shape")
_check_private_value("ex024_batch_view_aligned_shape", "ex024", "batch_view_aligned_shape")
_check_private_value("ex024_batch_view_expanded_axes", "ex024", "batch_view_expanded_axes")
_check_private_value("ex024_cube_aligned_shape", "ex024", "cube_aligned_shape")
_check_private_value("ex024_cube_expanded_axes", "ex024", "cube_expanded_axes")
_check_private_value("ex024_compatible", "ex024", "compatible")
_check_private_value("ex024_out_shape", "ex024", "out_shape")
_check_private_tensor("ex024_batch_view", "ex024", "batch_view")
_check_private_tensor("ex024_out", "ex024", "out")


### Exercise 025 — Channel vector into images

**Purpose:** Prepare one channel value for a `(batch, channel, height, width)` tensor.

**Visible inputs:** `channels`, `images`. Run the fixture below and read its construction code; it deliberately prints no shape answers.

**Operation to reason about:** Create `channel_view`, then add the channel values to every pixel.

**Task:** Create `channel_view`, then add the channel values to every pixel.

**Ingredients:** The channel axis is second; surround it with the needed singleton axes.

**Required predictions and outputs:**

- `ex025_channels_shape`: a literal Python shape tuple
- `ex025_images_shape`: a literal Python shape tuple
- `ex025_channel_view_shape`: a literal Python shape tuple
- `ex025_channel_view_aligned_shape`: a literal Python shape tuple
- `ex025_channel_view_expanded_axes`: a tuple of zero-based aligned axes
- `ex025_images_aligned_shape`: a literal Python shape tuple
- `ex025_images_expanded_axes`: a tuple of zero-based aligned axes
- `ex025_compatible`: a Python `bool`
- `ex025_out_shape`: a literal Python shape tuple
- `ex025_channel_view`: the requested PyTorch tensor
- `ex025_out`: the requested PyTorch tensor

**Order:** Fill every shape/alignment prediction before writing tensor operations.

**Next concept:** Height vector into images


In [ ]:
# Supplied visible fixture: inspect these values, but predict shapes without printing them.
channels = torch.tensor([0.1, 0.2, 0.3], dtype=DTYPE)
images = torch.zeros((2, 3, 4, 5), dtype=DTYPE)
_register_case("ex025", {"channels": channels, "images": images})


In [ ]:
# Exercise 025: predict shapes first; do not use autograd.
# Define `ex025_channels_shape`.
# Define `ex025_images_shape`.
# Define `ex025_channel_view_shape`.
# Define `ex025_channel_view_aligned_shape`.
# Define `ex025_channel_view_expanded_axes`.
# Define `ex025_images_aligned_shape`.
# Define `ex025_images_expanded_axes`.
# Define `ex025_compatible`.
# Define `ex025_out_shape`.
# Define `ex025_channel_view` with PyTorch tensor operations.
# Define `ex025_out` with PyTorch tensor operations.
# Write your work below, then run the supplied test cell.


In [ ]:
# Supplied test: expected shapes and values remain private.
_check_private_value("ex025_channels_shape", "ex025", "channels_shape")
_check_private_value("ex025_images_shape", "ex025", "images_shape")
_check_private_value("ex025_channel_view_shape", "ex025", "channel_view_shape")
_check_private_value("ex025_channel_view_aligned_shape", "ex025", "channel_view_aligned_shape")
_check_private_value("ex025_channel_view_expanded_axes", "ex025", "channel_view_expanded_axes")
_check_private_value("ex025_images_aligned_shape", "ex025", "images_aligned_shape")
_check_private_value("ex025_images_expanded_axes", "ex025", "images_expanded_axes")
_check_private_value("ex025_compatible", "ex025", "compatible")
_check_private_value("ex025_out_shape", "ex025", "out_shape")
_check_private_tensor("ex025_channel_view", "ex025", "channel_view")
_check_private_tensor("ex025_out", "ex025", "out")


### Exercise 026 — Height vector into images

**Purpose:** Align four row values with the height axis of an image batch.

**Visible inputs:** `heights`, `images`. Run the fixture below and read its construction code; it deliberately prints no shape answers.

**Operation to reason about:** Create `height_view`, then multiply it into the images.

**Task:** Create `height_view`, then multiply it into the images.

**Ingredients:** Place the non-singleton axis at height, axis 2.

**Required predictions and outputs:**

- `ex026_heights_shape`: a literal Python shape tuple
- `ex026_images_shape`: a literal Python shape tuple
- `ex026_height_view_shape`: a literal Python shape tuple
- `ex026_height_view_aligned_shape`: a literal Python shape tuple
- `ex026_height_view_expanded_axes`: a tuple of zero-based aligned axes
- `ex026_images_aligned_shape`: a literal Python shape tuple
- `ex026_images_expanded_axes`: a tuple of zero-based aligned axes
- `ex026_compatible`: a Python `bool`
- `ex026_out_shape`: a literal Python shape tuple
- `ex026_height_view`: the requested PyTorch tensor
- `ex026_out`: the requested PyTorch tensor

**Order:** Fill every shape/alignment prediction before writing tensor operations.

**Next concept:** Width vector into images


In [ ]:
# Supplied visible fixture: inspect these values, but predict shapes without printing them.
heights = torch.tensor([1.0, 2.0, 3.0, 4.0], dtype=DTYPE)
images = torch.ones((2, 3, 4, 5), dtype=DTYPE)
_register_case("ex026", {"heights": heights, "images": images})


In [ ]:
# Exercise 026: predict shapes first; do not use autograd.
# Define `ex026_heights_shape`.
# Define `ex026_images_shape`.
# Define `ex026_height_view_shape`.
# Define `ex026_height_view_aligned_shape`.
# Define `ex026_height_view_expanded_axes`.
# Define `ex026_images_aligned_shape`.
# Define `ex026_images_expanded_axes`.
# Define `ex026_compatible`.
# Define `ex026_out_shape`.
# Define `ex026_height_view` with PyTorch tensor operations.
# Define `ex026_out` with PyTorch tensor operations.
# Write your work below, then run the supplied test cell.


In [ ]:
# Supplied test: expected shapes and values remain private.
_check_private_value("ex026_heights_shape", "ex026", "heights_shape")
_check_private_value("ex026_images_shape", "ex026", "images_shape")
_check_private_value("ex026_height_view_shape", "ex026", "height_view_shape")
_check_private_value("ex026_height_view_aligned_shape", "ex026", "height_view_aligned_shape")
_check_private_value("ex026_height_view_expanded_axes", "ex026", "height_view_expanded_axes")
_check_private_value("ex026_images_aligned_shape", "ex026", "images_aligned_shape")
_check_private_value("ex026_images_expanded_axes", "ex026", "images_expanded_axes")
_check_private_value("ex026_compatible", "ex026", "compatible")
_check_private_value("ex026_out_shape", "ex026", "out_shape")
_check_private_tensor("ex026_height_view", "ex026", "height_view")
_check_private_tensor("ex026_out", "ex026", "out")


### Exercise 027 — Width vector into images

**Purpose:** Align five values with the final width axis.

**Visible inputs:** `widths`, `images`. Run the fixture below and read its construction code; it deliberately prints no shape answers.

**Operation to reason about:** Create `width_view`, then multiply it into the images.

**Task:** Create `width_view`, then multiply it into the images.

**Ingredients:** The width axis is already last; insert three leading axes.

**Required predictions and outputs:**

- `ex027_widths_shape`: a literal Python shape tuple
- `ex027_images_shape`: a literal Python shape tuple
- `ex027_width_view_shape`: a literal Python shape tuple
- `ex027_width_view_aligned_shape`: a literal Python shape tuple
- `ex027_width_view_expanded_axes`: a tuple of zero-based aligned axes
- `ex027_images_aligned_shape`: a literal Python shape tuple
- `ex027_images_expanded_axes`: a tuple of zero-based aligned axes
- `ex027_compatible`: a Python `bool`
- `ex027_out_shape`: a literal Python shape tuple
- `ex027_width_view`: the requested PyTorch tensor
- `ex027_out`: the requested PyTorch tensor

**Order:** Fill every shape/alignment prediction before writing tensor operations.

**Next concept:** Insert a middle sequence axis


In [ ]:
# Supplied visible fixture: inspect these values, but predict shapes without printing them.
widths = torch.linspace(1.0, 2.0, steps=5, dtype=DTYPE)
images = torch.ones((2, 3, 4, 5), dtype=DTYPE)
_register_case("ex027", {"widths": widths, "images": images})


In [ ]:
# Exercise 027: predict shapes first; do not use autograd.
# Define `ex027_widths_shape`.
# Define `ex027_images_shape`.
# Define `ex027_width_view_shape`.
# Define `ex027_width_view_aligned_shape`.
# Define `ex027_width_view_expanded_axes`.
# Define `ex027_images_aligned_shape`.
# Define `ex027_images_expanded_axes`.
# Define `ex027_compatible`.
# Define `ex027_out_shape`.
# Define `ex027_width_view` with PyTorch tensor operations.
# Define `ex027_out` with PyTorch tensor operations.
# Write your work below, then run the supplied test cell.


In [ ]:
# Supplied test: expected shapes and values remain private.
_check_private_value("ex027_widths_shape", "ex027", "widths_shape")
_check_private_value("ex027_images_shape", "ex027", "images_shape")
_check_private_value("ex027_width_view_shape", "ex027", "width_view_shape")
_check_private_value("ex027_width_view_aligned_shape", "ex027", "width_view_aligned_shape")
_check_private_value("ex027_width_view_expanded_axes", "ex027", "width_view_expanded_axes")
_check_private_value("ex027_images_aligned_shape", "ex027", "images_aligned_shape")
_check_private_value("ex027_images_expanded_axes", "ex027", "images_expanded_axes")
_check_private_value("ex027_compatible", "ex027", "compatible")
_check_private_value("ex027_out_shape", "ex027", "out_shape")
_check_private_tensor("ex027_width_view", "ex027", "width_view")
_check_private_tensor("ex027_out", "ex027", "out")


### Exercise 028 — Insert a middle sequence axis

**Purpose:** Prepare a `(batch, feature)` matrix for a `(batch, time, feature)` tensor.

**Visible inputs:** `per_batch_feature`, `sequence`. Run the fixture below and read its construction code; it deliberately prints no shape answers.

**Operation to reason about:** Create `prepared` by inserting the time axis, then add it to `sequence`.

**Task:** Create `prepared` by inserting the time axis, then add it to `sequence`.

**Ingredients:** Keep batch and feature; insert a singleton between them.

**Required predictions and outputs:**

- `ex028_per_batch_feature_shape`: a literal Python shape tuple
- `ex028_sequence_shape`: a literal Python shape tuple
- `ex028_prepared_shape`: a literal Python shape tuple
- `ex028_prepared_aligned_shape`: a literal Python shape tuple
- `ex028_prepared_expanded_axes`: a tuple of zero-based aligned axes
- `ex028_sequence_aligned_shape`: a literal Python shape tuple
- `ex028_sequence_expanded_axes`: a tuple of zero-based aligned axes
- `ex028_compatible`: a Python `bool`
- `ex028_out_shape`: a literal Python shape tuple
- `ex028_prepared`: the requested PyTorch tensor
- `ex028_out`: the requested PyTorch tensor

**Order:** Fill every shape/alignment prediction before writing tensor operations.

**Next concept:** Insert a trailing feature axis


In [ ]:
# Supplied visible fixture: inspect these values, but predict shapes without printing them.
per_batch_feature = torch.arange(8, dtype=DTYPE).reshape(2, 4)
sequence = torch.zeros((2, 3, 4), dtype=DTYPE)
_register_case("ex028", {"per_batch_feature": per_batch_feature, "sequence": sequence})


In [ ]:
# Exercise 028: predict shapes first; do not use autograd.
# Define `ex028_per_batch_feature_shape`.
# Define `ex028_sequence_shape`.
# Define `ex028_prepared_shape`.
# Define `ex028_prepared_aligned_shape`.
# Define `ex028_prepared_expanded_axes`.
# Define `ex028_sequence_aligned_shape`.
# Define `ex028_sequence_expanded_axes`.
# Define `ex028_compatible`.
# Define `ex028_out_shape`.
# Define `ex028_prepared` with PyTorch tensor operations.
# Define `ex028_out` with PyTorch tensor operations.
# Write your work below, then run the supplied test cell.


In [ ]:
# Supplied test: expected shapes and values remain private.
_check_private_value("ex028_per_batch_feature_shape", "ex028", "per_batch_feature_shape")
_check_private_value("ex028_sequence_shape", "ex028", "sequence_shape")
_check_private_value("ex028_prepared_shape", "ex028", "prepared_shape")
_check_private_value("ex028_prepared_aligned_shape", "ex028", "prepared_aligned_shape")
_check_private_value("ex028_prepared_expanded_axes", "ex028", "prepared_expanded_axes")
_check_private_value("ex028_sequence_aligned_shape", "ex028", "sequence_aligned_shape")
_check_private_value("ex028_sequence_expanded_axes", "ex028", "sequence_expanded_axes")
_check_private_value("ex028_compatible", "ex028", "compatible")
_check_private_value("ex028_out_shape", "ex028", "out_shape")
_check_private_tensor("ex028_prepared", "ex028", "prepared")
_check_private_tensor("ex028_out", "ex028", "out")


### Exercise 029 — Insert a trailing feature axis

**Purpose:** Prepare a `(batch, time)` mask for `(batch, time, feature)` values.

**Visible inputs:** `mask`, `features`. Run the fixture below and read its construction code; it deliberately prints no shape answers.

**Operation to reason about:** Create `mask_view`, then mask every feature at each batch-time position.

**Task:** Create `mask_view`, then mask every feature at each batch-time position.

**Ingredients:** Insert one singleton axis at the end.

**Required predictions and outputs:**

- `ex029_mask_shape`: a literal Python shape tuple
- `ex029_features_shape`: a literal Python shape tuple
- `ex029_mask_view_shape`: a literal Python shape tuple
- `ex029_mask_view_aligned_shape`: a literal Python shape tuple
- `ex029_mask_view_expanded_axes`: a tuple of zero-based aligned axes
- `ex029_features_aligned_shape`: a literal Python shape tuple
- `ex029_features_expanded_axes`: a tuple of zero-based aligned axes
- `ex029_compatible`: a Python `bool`
- `ex029_out_shape`: a literal Python shape tuple
- `ex029_mask_view`: the requested PyTorch tensor
- `ex029_out`: the requested PyTorch tensor

**Order:** Fill every shape/alignment prediction before writing tensor operations.

**Next concept:** Insert two image axes


In [ ]:
# Supplied visible fixture: inspect these values, but predict shapes without printing them.
mask = torch.tensor([[1.0, 1.0, 0.0], [1.0, 0.0, 0.0]], dtype=DTYPE)
features = torch.arange(24, dtype=DTYPE).reshape(2, 3, 4)
_register_case("ex029", {"mask": mask, "features": features})


In [ ]:
# Exercise 029: predict shapes first; do not use autograd.
# Define `ex029_mask_shape`.
# Define `ex029_features_shape`.
# Define `ex029_mask_view_shape`.
# Define `ex029_mask_view_aligned_shape`.
# Define `ex029_mask_view_expanded_axes`.
# Define `ex029_features_aligned_shape`.
# Define `ex029_features_expanded_axes`.
# Define `ex029_compatible`.
# Define `ex029_out_shape`.
# Define `ex029_mask_view` with PyTorch tensor operations.
# Define `ex029_out` with PyTorch tensor operations.
# Write your work below, then run the supplied test cell.


In [ ]:
# Supplied test: expected shapes and values remain private.
_check_private_value("ex029_mask_shape", "ex029", "mask_shape")
_check_private_value("ex029_features_shape", "ex029", "features_shape")
_check_private_value("ex029_mask_view_shape", "ex029", "mask_view_shape")
_check_private_value("ex029_mask_view_aligned_shape", "ex029", "mask_view_aligned_shape")
_check_private_value("ex029_mask_view_expanded_axes", "ex029", "mask_view_expanded_axes")
_check_private_value("ex029_features_aligned_shape", "ex029", "features_aligned_shape")
_check_private_value("ex029_features_expanded_axes", "ex029", "features_expanded_axes")
_check_private_value("ex029_compatible", "ex029", "compatible")
_check_private_value("ex029_out_shape", "ex029", "out_shape")
_check_private_tensor("ex029_mask_view", "ex029", "mask_view")
_check_private_tensor("ex029_out", "ex029", "out")


### Exercise 030 — Insert two image axes

**Purpose:** Prepare a `(channel, width)` gain table for a rank-4 image tensor.

**Visible inputs:** `gain`, `images`. Run the fixture below and read its construction code; it deliberately prints no shape answers.

**Operation to reason about:** Create `gain_view` with batch and height singleton axes, then scale `images`.

**Task:** Create `gain_view` with batch and height singleton axes, then scale `images`.

**Ingredients:** Map the two existing axes to channel and width.

**Required predictions and outputs:**

- `ex030_gain_shape`: a literal Python shape tuple
- `ex030_images_shape`: a literal Python shape tuple
- `ex030_gain_view_shape`: a literal Python shape tuple
- `ex030_gain_view_aligned_shape`: a literal Python shape tuple
- `ex030_gain_view_expanded_axes`: a tuple of zero-based aligned axes
- `ex030_images_aligned_shape`: a literal Python shape tuple
- `ex030_images_expanded_axes`: a tuple of zero-based aligned axes
- `ex030_compatible`: a Python `bool`
- `ex030_out_shape`: a literal Python shape tuple
- `ex030_gain_view`: the requested PyTorch tensor
- `ex030_out`: the requested PyTorch tensor

**Order:** Fill every shape/alignment prediction before writing tensor operations.

**Next concept:** Rank-3 middle-axis offset


In [17]:
# Supplied visible fixture: inspect these values, but predict shapes without printing them.
gain = torch.linspace(0.5, 1.5, steps=15, dtype=DTYPE).reshape(3, 5)
images = torch.ones((2, 3, 4, 5), dtype=DTYPE)
_register_case("ex030", {"gain": gain, "images": images})


In [18]:
# Exercise 030: predict shapes first; do not use autograd.
# Define `ex030_gain_shape`.
# Define `ex030_images_shape`.
# Define `ex030_gain_view_shape`.
# Define `ex030_gain_view_aligned_shape`.
# Define `ex030_gain_view_expanded_axes`.
# Define `ex030_images_aligned_shape`.
# Define `ex030_images_expanded_axes`.
# Define `ex030_compatible`.
# Define `ex030_out_shape`.
# Define `ex030_gain_view` with PyTorch tensor operations.
# Define `ex030_out` with PyTorch tensor operations.
# Write your work below, then run the supplied test cell.


In [19]:
# Supplied test: expected shapes and values remain private.
_check_private_value("ex030_gain_shape", "ex030", "gain_shape")
_check_private_value("ex030_images_shape", "ex030", "images_shape")
_check_private_value("ex030_gain_view_shape", "ex030", "gain_view_shape")
_check_private_value("ex030_gain_view_aligned_shape", "ex030", "gain_view_aligned_shape")
_check_private_value("ex030_gain_view_expanded_axes", "ex030", "gain_view_expanded_axes")
_check_private_value("ex030_images_aligned_shape", "ex030", "images_aligned_shape")
_check_private_value("ex030_images_expanded_axes", "ex030", "images_expanded_axes")
_check_private_value("ex030_compatible", "ex030", "compatible")
_check_private_value("ex030_out_shape", "ex030", "out_shape")
_check_private_tensor("ex030_gain_view", "ex030", "gain_view")
_check_private_tensor("ex030_out", "ex030", "out")


AssertionError: Define `ex030_gain_shape` in the answer cell first.

## 4. Higher-rank and semantic-axis patterns

Name axes such as batch, channel, time, height, width, query, and key before comparing numerical sizes.


### Exercise 031 — Rank-3 middle-axis offset

**Purpose:** Audit a full three-axis broadcast with singleton ends.

**Visible inputs:** `a`, `b`. Run the fixture below and read its construction code; it deliberately prints no shape answers.

**Operation to reason about:** `a + b`

**Task:** Add one offset per middle-axis entry.

**Ingredients:** Write axis meanings before comparing sizes.

**Required predictions and outputs:**

- `ex031_a_shape`: a literal Python shape tuple
- `ex031_b_shape`: a literal Python shape tuple
- `ex031_a_aligned_shape`: a literal Python shape tuple
- `ex031_a_expanded_axes`: a tuple of zero-based aligned axes
- `ex031_b_aligned_shape`: a literal Python shape tuple
- `ex031_b_expanded_axes`: a tuple of zero-based aligned axes
- `ex031_compatible`: a Python `bool`
- `ex031_out_shape`: a literal Python shape tuple
- `ex031_out`: the requested PyTorch tensor

**Order:** Fill every shape/alignment prediction before writing tensor operations.

**Next concept:** Crossed rank-3 grid


In [ ]:
# Supplied visible fixture: inspect these values, but predict shapes without printing them.
a = torch.arange(24, dtype=DTYPE).reshape(2, 3, 4)
b = torch.tensor([[[1.0], [2.0], [3.0]]], dtype=DTYPE)
_register_case("ex031", {"a": a, "b": b})


In [ ]:
# Exercise 031: predict shapes first; do not use autograd.
# Define `ex031_a_shape`.
# Define `ex031_b_shape`.
# Define `ex031_a_aligned_shape`.
# Define `ex031_a_expanded_axes`.
# Define `ex031_b_aligned_shape`.
# Define `ex031_b_expanded_axes`.
# Define `ex031_compatible`.
# Define `ex031_out_shape`.
# Define `ex031_out` with PyTorch tensor operations.
# Write your work below, then run the supplied test cell.


In [ ]:
# Supplied test: expected shapes and values remain private.
_check_private_value("ex031_a_shape", "ex031", "a_shape")
_check_private_value("ex031_b_shape", "ex031", "b_shape")
_check_private_value("ex031_a_aligned_shape", "ex031", "a_aligned_shape")
_check_private_value("ex031_a_expanded_axes", "ex031", "a_expanded_axes")
_check_private_value("ex031_b_aligned_shape", "ex031", "b_aligned_shape")
_check_private_value("ex031_b_expanded_axes", "ex031", "b_expanded_axes")
_check_private_value("ex031_compatible", "ex031", "compatible")
_check_private_value("ex031_out_shape", "ex031", "out_shape")
_check_private_tensor("ex031_out", "ex031", "out")


### Exercise 032 — Crossed rank-3 grid

**Purpose:** Create a result where each operand supplies a different middle dimension.

**Visible inputs:** `a`, `b`. Run the fixture below and read its construction code; it deliberately prints no shape answers.

**Operation to reason about:** `a + b`

**Task:** Combine the tensors into one three-dimensional grid.

**Ingredients:** Both operands retain their non-singleton axes.

**Required predictions and outputs:**

- `ex032_a_shape`: a literal Python shape tuple
- `ex032_b_shape`: a literal Python shape tuple
- `ex032_a_aligned_shape`: a literal Python shape tuple
- `ex032_a_expanded_axes`: a tuple of zero-based aligned axes
- `ex032_b_aligned_shape`: a literal Python shape tuple
- `ex032_b_expanded_axes`: a tuple of zero-based aligned axes
- `ex032_compatible`: a Python `bool`
- `ex032_out_shape`: a literal Python shape tuple
- `ex032_out`: the requested PyTorch tensor

**Order:** Fill every shape/alignment prediction before writing tensor operations.

**Next concept:** Crossed rank-4 grid


In [ ]:
# Supplied visible fixture: inspect these values, but predict shapes without printing them.
a = torch.arange(35, dtype=DTYPE).reshape(5, 1, 7)
b = torch.arange(6, dtype=DTYPE).reshape(1, 6, 1)
_register_case("ex032", {"a": a, "b": b})


In [ ]:
# Exercise 032: predict shapes first; do not use autograd.
# Define `ex032_a_shape`.
# Define `ex032_b_shape`.
# Define `ex032_a_aligned_shape`.
# Define `ex032_a_expanded_axes`.
# Define `ex032_b_aligned_shape`.
# Define `ex032_b_expanded_axes`.
# Define `ex032_compatible`.
# Define `ex032_out_shape`.
# Define `ex032_out` with PyTorch tensor operations.
# Write your work below, then run the supplied test cell.


In [ ]:
# Supplied test: expected shapes and values remain private.
_check_private_value("ex032_a_shape", "ex032", "a_shape")
_check_private_value("ex032_b_shape", "ex032", "b_shape")
_check_private_value("ex032_a_aligned_shape", "ex032", "a_aligned_shape")
_check_private_value("ex032_a_expanded_axes", "ex032", "a_expanded_axes")
_check_private_value("ex032_b_aligned_shape", "ex032", "b_aligned_shape")
_check_private_value("ex032_b_expanded_axes", "ex032", "b_expanded_axes")
_check_private_value("ex032_compatible", "ex032", "compatible")
_check_private_value("ex032_out_shape", "ex032", "out_shape")
_check_private_tensor("ex032_out", "ex032", "out")


### Exercise 033 — Crossed rank-4 grid

**Purpose:** Track independent batch-like, channel-like, height-like, and width-like axes.

**Visible inputs:** `a`, `b`. Run the fixture below and read its construction code; it deliberately prints no shape answers.

**Operation to reason about:** `a * b`

**Task:** Multiply all compatible combinations.

**Ingredients:** Mark each `1` that must expand.

**Required predictions and outputs:**

- `ex033_a_shape`: a literal Python shape tuple
- `ex033_b_shape`: a literal Python shape tuple
- `ex033_a_aligned_shape`: a literal Python shape tuple
- `ex033_a_expanded_axes`: a tuple of zero-based aligned axes
- `ex033_b_aligned_shape`: a literal Python shape tuple
- `ex033_b_expanded_axes`: a tuple of zero-based aligned axes
- `ex033_compatible`: a Python `bool`
- `ex033_out_shape`: a literal Python shape tuple
- `ex033_out`: the requested PyTorch tensor

**Order:** Fill every shape/alignment prediction before writing tensor operations.

**Next concept:** Image channel bias


In [ ]:
# Supplied visible fixture: inspect these values, but predict shapes without printing them.
a = torch.arange(8, dtype=DTYPE).reshape(2, 1, 4, 1)
b = torch.arange(15, dtype=DTYPE).reshape(1, 3, 1, 5)
_register_case("ex033", {"a": a, "b": b})


In [ ]:
# Exercise 033: predict shapes first; do not use autograd.
# Define `ex033_a_shape`.
# Define `ex033_b_shape`.
# Define `ex033_a_aligned_shape`.
# Define `ex033_a_expanded_axes`.
# Define `ex033_b_aligned_shape`.
# Define `ex033_b_expanded_axes`.
# Define `ex033_compatible`.
# Define `ex033_out_shape`.
# Define `ex033_out` with PyTorch tensor operations.
# Write your work below, then run the supplied test cell.


In [ ]:
# Supplied test: expected shapes and values remain private.
_check_private_value("ex033_a_shape", "ex033", "a_shape")
_check_private_value("ex033_b_shape", "ex033", "b_shape")
_check_private_value("ex033_a_aligned_shape", "ex033", "a_aligned_shape")
_check_private_value("ex033_a_expanded_axes", "ex033", "a_expanded_axes")
_check_private_value("ex033_b_aligned_shape", "ex033", "b_aligned_shape")
_check_private_value("ex033_b_expanded_axes", "ex033", "b_expanded_axes")
_check_private_value("ex033_compatible", "ex033", "compatible")
_check_private_value("ex033_out_shape", "ex033", "out_shape")
_check_private_tensor("ex033_out", "ex033", "out")


### Exercise 034 — Image channel bias

**Purpose:** Interpret `(channel, 1, 1)` against `(batch, channel, height, width)`.

**Visible inputs:** `images`, `bias`. Run the fixture below and read its construction code; it deliberately prints no shape answers.

**Operation to reason about:** `images + bias`

**Task:** Add one bias per channel.

**Ingredients:** The missing batch axis is left-padded as one.

**Required predictions and outputs:**

- `ex034_images_shape`: a literal Python shape tuple
- `ex034_bias_shape`: a literal Python shape tuple
- `ex034_images_aligned_shape`: a literal Python shape tuple
- `ex034_images_expanded_axes`: a tuple of zero-based aligned axes
- `ex034_bias_aligned_shape`: a literal Python shape tuple
- `ex034_bias_expanded_axes`: a tuple of zero-based aligned axes
- `ex034_compatible`: a Python `bool`
- `ex034_out_shape`: a literal Python shape tuple
- `ex034_out`: the requested PyTorch tensor

**Order:** Fill every shape/alignment prediction before writing tensor operations.

**Next concept:** Image batch scale


In [ ]:
# Supplied visible fixture: inspect these values, but predict shapes without printing them.
images = torch.arange(120, dtype=DTYPE).reshape(2, 3, 4, 5)
bias = torch.tensor([[[0.1]], [[0.2]], [[0.3]]], dtype=DTYPE)
_register_case("ex034", {"images": images, "bias": bias})


In [ ]:
# Exercise 034: predict shapes first; do not use autograd.
# Define `ex034_images_shape`.
# Define `ex034_bias_shape`.
# Define `ex034_images_aligned_shape`.
# Define `ex034_images_expanded_axes`.
# Define `ex034_bias_aligned_shape`.
# Define `ex034_bias_expanded_axes`.
# Define `ex034_compatible`.
# Define `ex034_out_shape`.
# Define `ex034_out` with PyTorch tensor operations.
# Write your work below, then run the supplied test cell.


In [ ]:
# Supplied test: expected shapes and values remain private.
_check_private_value("ex034_images_shape", "ex034", "images_shape")
_check_private_value("ex034_bias_shape", "ex034", "bias_shape")
_check_private_value("ex034_images_aligned_shape", "ex034", "images_aligned_shape")
_check_private_value("ex034_images_expanded_axes", "ex034", "images_expanded_axes")
_check_private_value("ex034_bias_aligned_shape", "ex034", "bias_aligned_shape")
_check_private_value("ex034_bias_expanded_axes", "ex034", "bias_expanded_axes")
_check_private_value("ex034_compatible", "ex034", "compatible")
_check_private_value("ex034_out_shape", "ex034", "out_shape")
_check_private_tensor("ex034_out", "ex034", "out")


### Exercise 035 — Image batch scale

**Purpose:** Interpret `(batch, 1, 1, 1)` as one scale per image.

**Visible inputs:** `images`, `scale`. Run the fixture below and read its construction code; it deliberately prints no shape answers.

**Operation to reason about:** `images * scale`

**Task:** Scale each image independently.

**Ingredients:** Only the leading axis should remain non-singleton.

**Required predictions and outputs:**

- `ex035_images_shape`: a literal Python shape tuple
- `ex035_scale_shape`: a literal Python shape tuple
- `ex035_images_aligned_shape`: a literal Python shape tuple
- `ex035_images_expanded_axes`: a tuple of zero-based aligned axes
- `ex035_scale_aligned_shape`: a literal Python shape tuple
- `ex035_scale_expanded_axes`: a tuple of zero-based aligned axes
- `ex035_compatible`: a Python `bool`
- `ex035_out_shape`: a literal Python shape tuple
- `ex035_out`: the requested PyTorch tensor

**Order:** Fill every shape/alignment prediction before writing tensor operations.

**Next concept:** Spatial map across batch and channels


In [ ]:
# Supplied visible fixture: inspect these values, but predict shapes without printing them.
images = torch.ones((2, 3, 4, 5), dtype=DTYPE)
scale = torch.tensor([[[[2.0]]], [[[-1.0]]]], dtype=DTYPE)
_register_case("ex035", {"images": images, "scale": scale})


In [ ]:
# Exercise 035: predict shapes first; do not use autograd.
# Define `ex035_images_shape`.
# Define `ex035_scale_shape`.
# Define `ex035_images_aligned_shape`.
# Define `ex035_images_expanded_axes`.
# Define `ex035_scale_aligned_shape`.
# Define `ex035_scale_expanded_axes`.
# Define `ex035_compatible`.
# Define `ex035_out_shape`.
# Define `ex035_out` with PyTorch tensor operations.
# Write your work below, then run the supplied test cell.


In [ ]:
# Supplied test: expected shapes and values remain private.
_check_private_value("ex035_images_shape", "ex035", "images_shape")
_check_private_value("ex035_scale_shape", "ex035", "scale_shape")
_check_private_value("ex035_images_aligned_shape", "ex035", "images_aligned_shape")
_check_private_value("ex035_images_expanded_axes", "ex035", "images_expanded_axes")
_check_private_value("ex035_scale_aligned_shape", "ex035", "scale_aligned_shape")
_check_private_value("ex035_scale_expanded_axes", "ex035", "scale_expanded_axes")
_check_private_value("ex035_compatible", "ex035", "compatible")
_check_private_value("ex035_out_shape", "ex035", "out_shape")
_check_private_tensor("ex035_out", "ex035", "out")


### Exercise 036 — Spatial map across batch and channels

**Purpose:** Reuse one height-width map over batch and channel axes.

**Visible inputs:** `images`, `spatial`. Run the fixture below and read its construction code; it deliberately prints no shape answers.

**Operation to reason about:** `images + spatial`

**Task:** Add the same spatial map everywhere.

**Ingredients:** The two leading singleton axes expand.

**Required predictions and outputs:**

- `ex036_images_shape`: a literal Python shape tuple
- `ex036_spatial_shape`: a literal Python shape tuple
- `ex036_images_aligned_shape`: a literal Python shape tuple
- `ex036_images_expanded_axes`: a tuple of zero-based aligned axes
- `ex036_spatial_aligned_shape`: a literal Python shape tuple
- `ex036_spatial_expanded_axes`: a tuple of zero-based aligned axes
- `ex036_compatible`: a Python `bool`
- `ex036_out_shape`: a literal Python shape tuple
- `ex036_out`: the requested PyTorch tensor

**Order:** Fill every shape/alignment prediction before writing tensor operations.

**Next concept:** Per-image per-channel gain


In [ ]:
# Supplied visible fixture: inspect these values, but predict shapes without printing them.
images = torch.ones((2, 3, 4, 5), dtype=DTYPE)
spatial = torch.arange(20, dtype=DTYPE).reshape(1, 1, 4, 5)
_register_case("ex036", {"images": images, "spatial": spatial})


In [ ]:
# Exercise 036: predict shapes first; do not use autograd.
# Define `ex036_images_shape`.
# Define `ex036_spatial_shape`.
# Define `ex036_images_aligned_shape`.
# Define `ex036_images_expanded_axes`.
# Define `ex036_spatial_aligned_shape`.
# Define `ex036_spatial_expanded_axes`.
# Define `ex036_compatible`.
# Define `ex036_out_shape`.
# Define `ex036_out` with PyTorch tensor operations.
# Write your work below, then run the supplied test cell.


In [ ]:
# Supplied test: expected shapes and values remain private.
_check_private_value("ex036_images_shape", "ex036", "images_shape")
_check_private_value("ex036_spatial_shape", "ex036", "spatial_shape")
_check_private_value("ex036_images_aligned_shape", "ex036", "images_aligned_shape")
_check_private_value("ex036_images_expanded_axes", "ex036", "images_expanded_axes")
_check_private_value("ex036_spatial_aligned_shape", "ex036", "spatial_aligned_shape")
_check_private_value("ex036_spatial_expanded_axes", "ex036", "spatial_expanded_axes")
_check_private_value("ex036_compatible", "ex036", "compatible")
_check_private_value("ex036_out_shape", "ex036", "out_shape")
_check_private_tensor("ex036_out", "ex036", "out")


### Exercise 037 — Per-image per-channel gain

**Purpose:** Preserve batch and channel while expanding spatial axes.

**Visible inputs:** `images`, `gain`. Run the fixture below and read its construction code; it deliberately prints no shape answers.

**Operation to reason about:** `images * gain`

**Task:** Apply one gain to each image-channel pair.

**Ingredients:** The last two axes represent reusable spatial positions.

**Required predictions and outputs:**

- `ex037_images_shape`: a literal Python shape tuple
- `ex037_gain_shape`: a literal Python shape tuple
- `ex037_images_aligned_shape`: a literal Python shape tuple
- `ex037_images_expanded_axes`: a tuple of zero-based aligned axes
- `ex037_gain_aligned_shape`: a literal Python shape tuple
- `ex037_gain_expanded_axes`: a tuple of zero-based aligned axes
- `ex037_compatible`: a Python `bool`
- `ex037_out_shape`: a literal Python shape tuple
- `ex037_out`: the requested PyTorch tensor

**Order:** Fill every shape/alignment prediction before writing tensor operations.

**Next concept:** Positional embeddings


In [ ]:
# Supplied visible fixture: inspect these values, but predict shapes without printing them.
images = torch.ones((2, 3, 4, 5), dtype=DTYPE)
gain = torch.arange(6, dtype=DTYPE).reshape(2, 3, 1, 1) + 1
_register_case("ex037", {"images": images, "gain": gain})


In [ ]:
# Exercise 037: predict shapes first; do not use autograd.
# Define `ex037_images_shape`.
# Define `ex037_gain_shape`.
# Define `ex037_images_aligned_shape`.
# Define `ex037_images_expanded_axes`.
# Define `ex037_gain_aligned_shape`.
# Define `ex037_gain_expanded_axes`.
# Define `ex037_compatible`.
# Define `ex037_out_shape`.
# Define `ex037_out` with PyTorch tensor operations.
# Write your work below, then run the supplied test cell.


In [ ]:
# Supplied test: expected shapes and values remain private.
_check_private_value("ex037_images_shape", "ex037", "images_shape")
_check_private_value("ex037_gain_shape", "ex037", "gain_shape")
_check_private_value("ex037_images_aligned_shape", "ex037", "images_aligned_shape")
_check_private_value("ex037_images_expanded_axes", "ex037", "images_expanded_axes")
_check_private_value("ex037_gain_aligned_shape", "ex037", "gain_aligned_shape")
_check_private_value("ex037_gain_expanded_axes", "ex037", "gain_expanded_axes")
_check_private_value("ex037_compatible", "ex037", "compatible")
_check_private_value("ex037_out_shape", "ex037", "out_shape")
_check_private_tensor("ex037_out", "ex037", "out")


### Exercise 038 — Positional embeddings

**Purpose:** Add one `(time, feature)` table to every sequence in a batch.

**Visible inputs:** `tokens`, `positions`. Run the fixture below and read its construction code; it deliberately prints no shape answers.

**Operation to reason about:** `tokens + positions`

**Task:** Add positional values across the batch.

**Ingredients:** Right-align time and feature; the missing batch axis expands.

**Required predictions and outputs:**

- `ex038_tokens_shape`: a literal Python shape tuple
- `ex038_positions_shape`: a literal Python shape tuple
- `ex038_tokens_aligned_shape`: a literal Python shape tuple
- `ex038_tokens_expanded_axes`: a tuple of zero-based aligned axes
- `ex038_positions_aligned_shape`: a literal Python shape tuple
- `ex038_positions_expanded_axes`: a tuple of zero-based aligned axes
- `ex038_compatible`: a Python `bool`
- `ex038_out_shape`: a literal Python shape tuple
- `ex038_out`: the requested PyTorch tensor

**Order:** Fill every shape/alignment prediction before writing tensor operations.

**Next concept:** Attention head bias


In [ ]:
# Supplied visible fixture: inspect these values, but predict shapes without printing them.
tokens = torch.arange(24, dtype=DTYPE).reshape(2, 3, 4)
positions = torch.linspace(-1.0, 1.0, steps=12, dtype=DTYPE).reshape(3, 4)
_register_case("ex038", {"tokens": tokens, "positions": positions})


In [ ]:
# Exercise 038: predict shapes first; do not use autograd.
# Define `ex038_tokens_shape`.
# Define `ex038_positions_shape`.
# Define `ex038_tokens_aligned_shape`.
# Define `ex038_tokens_expanded_axes`.
# Define `ex038_positions_aligned_shape`.
# Define `ex038_positions_expanded_axes`.
# Define `ex038_compatible`.
# Define `ex038_out_shape`.
# Define `ex038_out` with PyTorch tensor operations.
# Write your work below, then run the supplied test cell.


In [ ]:
# Supplied test: expected shapes and values remain private.
_check_private_value("ex038_tokens_shape", "ex038", "tokens_shape")
_check_private_value("ex038_positions_shape", "ex038", "positions_shape")
_check_private_value("ex038_tokens_aligned_shape", "ex038", "tokens_aligned_shape")
_check_private_value("ex038_tokens_expanded_axes", "ex038", "tokens_expanded_axes")
_check_private_value("ex038_positions_aligned_shape", "ex038", "positions_aligned_shape")
_check_private_value("ex038_positions_expanded_axes", "ex038", "positions_expanded_axes")
_check_private_value("ex038_compatible", "ex038", "compatible")
_check_private_value("ex038_out_shape", "ex038", "out_shape")
_check_private_tensor("ex038_out", "ex038", "out")


### Exercise 039 — Attention head bias

**Purpose:** Align `(head, 1, 1)` beneath `(batch, head, query, key)`.

**Visible inputs:** `scores`, `head_bias`. Run the fixture below and read its construction code; it deliberately prints no shape answers.

**Operation to reason about:** `scores + head_bias`

**Task:** Add one scalar per attention head.

**Ingredients:** Mentally prepend the missing batch singleton.

**Required predictions and outputs:**

- `ex039_scores_shape`: a literal Python shape tuple
- `ex039_head_bias_shape`: a literal Python shape tuple
- `ex039_scores_aligned_shape`: a literal Python shape tuple
- `ex039_scores_expanded_axes`: a tuple of zero-based aligned axes
- `ex039_head_bias_aligned_shape`: a literal Python shape tuple
- `ex039_head_bias_expanded_axes`: a tuple of zero-based aligned axes
- `ex039_compatible`: a Python `bool`
- `ex039_out_shape`: a literal Python shape tuple
- `ex039_out`: the requested PyTorch tensor

**Order:** Fill every shape/alignment prediction before writing tensor operations.

**Next concept:** Pairwise query-key grid


In [ ]:
# Supplied visible fixture: inspect these values, but predict shapes without printing them.
scores = torch.zeros((2, 3, 4, 4), dtype=DTYPE)
head_bias = torch.tensor([[[0.1]], [[0.2]], [[0.3]]], dtype=DTYPE)
_register_case("ex039", {"scores": scores, "head_bias": head_bias})


In [ ]:
# Exercise 039: predict shapes first; do not use autograd.
# Define `ex039_scores_shape`.
# Define `ex039_head_bias_shape`.
# Define `ex039_scores_aligned_shape`.
# Define `ex039_scores_expanded_axes`.
# Define `ex039_head_bias_aligned_shape`.
# Define `ex039_head_bias_expanded_axes`.
# Define `ex039_compatible`.
# Define `ex039_out_shape`.
# Define `ex039_out` with PyTorch tensor operations.
# Write your work below, then run the supplied test cell.


In [ ]:
# Supplied test: expected shapes and values remain private.
_check_private_value("ex039_scores_shape", "ex039", "scores_shape")
_check_private_value("ex039_head_bias_shape", "ex039", "head_bias_shape")
_check_private_value("ex039_scores_aligned_shape", "ex039", "scores_aligned_shape")
_check_private_value("ex039_scores_expanded_axes", "ex039", "scores_expanded_axes")
_check_private_value("ex039_head_bias_aligned_shape", "ex039", "head_bias_aligned_shape")
_check_private_value("ex039_head_bias_expanded_axes", "ex039", "head_bias_expanded_axes")
_check_private_value("ex039_compatible", "ex039", "compatible")
_check_private_value("ex039_out_shape", "ex039", "out_shape")
_check_private_tensor("ex039_out", "ex039", "out")


### Exercise 040 — Pairwise query-key grid

**Purpose:** Broadcast query-side and key-side values across opposite sequence axes.

**Visible inputs:** `query_values`, `key_values`. Run the fixture below and read its construction code; it deliberately prints no shape answers.

**Operation to reason about:** `query_values + key_values`

**Task:** Build every query-key combination within each batch and head.

**Ingredients:** The final two singleton axes expand in opposite directions.

**Required predictions and outputs:**

- `ex040_query_values_shape`: a literal Python shape tuple
- `ex040_key_values_shape`: a literal Python shape tuple
- `ex040_query_values_aligned_shape`: a literal Python shape tuple
- `ex040_query_values_expanded_axes`: a tuple of zero-based aligned axes
- `ex040_key_values_aligned_shape`: a literal Python shape tuple
- `ex040_key_values_expanded_axes`: a tuple of zero-based aligned axes
- `ex040_compatible`: a Python `bool`
- `ex040_out_shape`: a literal Python shape tuple
- `ex040_out`: the requested PyTorch tensor

**Order:** Fill every shape/alignment prediction before writing tensor operations.

**Next concept:** Crossed rank-5 tensor


In [ ]:
# Supplied visible fixture: inspect these values, but predict shapes without printing them.
query_values = torch.arange(24, dtype=DTYPE).reshape(2, 3, 4, 1)
key_values = torch.arange(24, dtype=DTYPE).reshape(2, 3, 1, 4)
_register_case("ex040", {"query_values": query_values, "key_values": key_values})


In [ ]:
# Exercise 040: predict shapes first; do not use autograd.
# Define `ex040_query_values_shape`.
# Define `ex040_key_values_shape`.
# Define `ex040_query_values_aligned_shape`.
# Define `ex040_query_values_expanded_axes`.
# Define `ex040_key_values_aligned_shape`.
# Define `ex040_key_values_expanded_axes`.
# Define `ex040_compatible`.
# Define `ex040_out_shape`.
# Define `ex040_out` with PyTorch tensor operations.
# Write your work below, then run the supplied test cell.


In [ ]:
# Supplied test: expected shapes and values remain private.
_check_private_value("ex040_query_values_shape", "ex040", "query_values_shape")
_check_private_value("ex040_key_values_shape", "ex040", "key_values_shape")
_check_private_value("ex040_query_values_aligned_shape", "ex040", "query_values_aligned_shape")
_check_private_value("ex040_query_values_expanded_axes", "ex040", "query_values_expanded_axes")
_check_private_value("ex040_key_values_aligned_shape", "ex040", "key_values_aligned_shape")
_check_private_value("ex040_key_values_expanded_axes", "ex040", "key_values_expanded_axes")
_check_private_value("ex040_compatible", "ex040", "compatible")
_check_private_value("ex040_out_shape", "ex040", "out_shape")
_check_private_tensor("ex040_out", "ex040", "out")


### Exercise 041 — Crossed rank-5 tensor

**Purpose:** Apply the same rules without relying on familiar matrix intuition.

**Visible inputs:** `a`, `b`. Run the fixture below and read its construction code; it deliberately prints no shape answers.

**Operation to reason about:** `a + b`

**Task:** Predict the complete rank-5 result and expansion axes.

**Ingredients:** Work strictly from the rightmost dimension to the left.

**Required predictions and outputs:**

- `ex041_a_shape`: a literal Python shape tuple
- `ex041_b_shape`: a literal Python shape tuple
- `ex041_a_aligned_shape`: a literal Python shape tuple
- `ex041_a_expanded_axes`: a tuple of zero-based aligned axes
- `ex041_b_aligned_shape`: a literal Python shape tuple
- `ex041_b_expanded_axes`: a tuple of zero-based aligned axes
- `ex041_compatible`: a Python `bool`
- `ex041_out_shape`: a literal Python shape tuple
- `ex041_out`: the requested PyTorch tensor

**Order:** Fill every shape/alignment prediction before writing tensor operations.

**Next concept:** Zero-length axis


In [ ]:
# Supplied visible fixture: inspect these values, but predict shapes without printing them.
a = torch.arange(30, dtype=DTYPE).reshape(2, 1, 3, 1, 5)
b = torch.arange(24, dtype=DTYPE).reshape(1, 4, 1, 6, 1)
_register_case("ex041", {"a": a, "b": b})


In [ ]:
# Exercise 041: predict shapes first; do not use autograd.
# Define `ex041_a_shape`.
# Define `ex041_b_shape`.
# Define `ex041_a_aligned_shape`.
# Define `ex041_a_expanded_axes`.
# Define `ex041_b_aligned_shape`.
# Define `ex041_b_expanded_axes`.
# Define `ex041_compatible`.
# Define `ex041_out_shape`.
# Define `ex041_out` with PyTorch tensor operations.
# Write your work below, then run the supplied test cell.


In [ ]:
# Supplied test: expected shapes and values remain private.
_check_private_value("ex041_a_shape", "ex041", "a_shape")
_check_private_value("ex041_b_shape", "ex041", "b_shape")
_check_private_value("ex041_a_aligned_shape", "ex041", "a_aligned_shape")
_check_private_value("ex041_a_expanded_axes", "ex041", "a_expanded_axes")
_check_private_value("ex041_b_aligned_shape", "ex041", "b_aligned_shape")
_check_private_value("ex041_b_expanded_axes", "ex041", "b_expanded_axes")
_check_private_value("ex041_compatible", "ex041", "compatible")
_check_private_value("ex041_out_shape", "ex041", "out_shape")
_check_private_tensor("ex041_out", "ex041", "out")


### Exercise 042 — Zero-length axis

**Purpose:** Learn that size zero can broadcast with size one and remain empty.

**Visible inputs:** `a`, `b`. Run the fixture below and read its construction code; it deliberately prints no shape answers.

**Operation to reason about:** `a + b`

**Task:** Predict the valid empty result.

**Ingredients:** A `1` may expand to `0`; the resulting axis has no elements.

**Required predictions and outputs:**

- `ex042_a_shape`: a literal Python shape tuple
- `ex042_b_shape`: a literal Python shape tuple
- `ex042_a_aligned_shape`: a literal Python shape tuple
- `ex042_a_expanded_axes`: a tuple of zero-based aligned axes
- `ex042_b_aligned_shape`: a literal Python shape tuple
- `ex042_b_expanded_axes`: a tuple of zero-based aligned axes
- `ex042_compatible`: a Python `bool`
- `ex042_out_shape`: a literal Python shape tuple
- `ex042_out`: the requested PyTorch tensor

**Order:** Fill every shape/alignment prediction before writing tensor operations.

**Next concept:** Boolean row mask


In [ ]:
# Supplied visible fixture: inspect these values, but predict shapes without printing them.
a = torch.empty((0, 3), dtype=DTYPE)
b = torch.ones((1, 3), dtype=DTYPE)
_register_case("ex042", {"a": a, "b": b})


In [ ]:
# Exercise 042: predict shapes first; do not use autograd.
# Define `ex042_a_shape`.
# Define `ex042_b_shape`.
# Define `ex042_a_aligned_shape`.
# Define `ex042_a_expanded_axes`.
# Define `ex042_b_aligned_shape`.
# Define `ex042_b_expanded_axes`.
# Define `ex042_compatible`.
# Define `ex042_out_shape`.
# Define `ex042_out` with PyTorch tensor operations.
# Write your work below, then run the supplied test cell.


In [ ]:
# Supplied test: expected shapes and values remain private.
_check_private_value("ex042_a_shape", "ex042", "a_shape")
_check_private_value("ex042_b_shape", "ex042", "b_shape")
_check_private_value("ex042_a_aligned_shape", "ex042", "a_aligned_shape")
_check_private_value("ex042_a_expanded_axes", "ex042", "a_expanded_axes")
_check_private_value("ex042_b_aligned_shape", "ex042", "b_aligned_shape")
_check_private_value("ex042_b_expanded_axes", "ex042", "b_expanded_axes")
_check_private_value("ex042_compatible", "ex042", "compatible")
_check_private_value("ex042_out_shape", "ex042", "out_shape")
_check_private_tensor("ex042_out", "ex042", "out")


### Exercise 043 — Boolean row mask

**Purpose:** Broadcast a `(rows, 1)` Boolean mask over columns.

**Visible inputs:** `mask`, `values`. Run the fixture below and read its construction code; it deliberately prints no shape answers.

**Operation to reason about:** `torch.where(mask, values, torch.zeros_like(values))`

**Task:** Keep the first row and replace the second row with zeros.

**Ingredients:** The mask's singleton column axis expands.

**Required predictions and outputs:**

- `ex043_mask_shape`: a literal Python shape tuple
- `ex043_values_shape`: a literal Python shape tuple
- `ex043_mask_aligned_shape`: a literal Python shape tuple
- `ex043_mask_expanded_axes`: a tuple of zero-based aligned axes
- `ex043_values_aligned_shape`: a literal Python shape tuple
- `ex043_values_expanded_axes`: a tuple of zero-based aligned axes
- `ex043_compatible`: a Python `bool`
- `ex043_out_shape`: a literal Python shape tuple
- `ex043_out`: the requested PyTorch tensor

**Order:** Fill every shape/alignment prediction before writing tensor operations.

**Next concept:** Per-channel standardization subtraction


In [ ]:
# Supplied visible fixture: inspect these values, but predict shapes without printing them.
mask = torch.tensor([[True], [False]])
values = torch.arange(6, dtype=DTYPE).reshape(2, 3)
_register_case("ex043", {"mask": mask, "values": values})


In [ ]:
# Exercise 043: predict shapes first; do not use autograd.
# Define `ex043_mask_shape`.
# Define `ex043_values_shape`.
# Define `ex043_mask_aligned_shape`.
# Define `ex043_mask_expanded_axes`.
# Define `ex043_values_aligned_shape`.
# Define `ex043_values_expanded_axes`.
# Define `ex043_compatible`.
# Define `ex043_out_shape`.
# Define `ex043_out` with PyTorch tensor operations.
# Write your work below, then run the supplied test cell.


In [ ]:
# Supplied test: expected shapes and values remain private.
_check_private_value("ex043_mask_shape", "ex043", "mask_shape")
_check_private_value("ex043_values_shape", "ex043", "values_shape")
_check_private_value("ex043_mask_aligned_shape", "ex043", "mask_aligned_shape")
_check_private_value("ex043_mask_expanded_axes", "ex043", "mask_expanded_axes")
_check_private_value("ex043_values_aligned_shape", "ex043", "values_aligned_shape")
_check_private_value("ex043_values_expanded_axes", "ex043", "values_expanded_axes")
_check_private_value("ex043_compatible", "ex043", "compatible")
_check_private_value("ex043_out_shape", "ex043", "out_shape")
_check_private_tensor("ex043_out", "ex043", "out")


### Exercise 044 — Per-channel standardization subtraction

**Purpose:** Subtract a channel mean from every image location.

**Visible inputs:** `images`, `mean`. Run the fixture below and read its construction code; it deliberately prints no shape answers.

**Operation to reason about:** `images - mean`

**Task:** Center each channel with its own mean.

**Ingredients:** Preserve channel and expand batch, height, and width.

**Required predictions and outputs:**

- `ex044_images_shape`: a literal Python shape tuple
- `ex044_mean_shape`: a literal Python shape tuple
- `ex044_images_aligned_shape`: a literal Python shape tuple
- `ex044_images_expanded_axes`: a tuple of zero-based aligned axes
- `ex044_mean_aligned_shape`: a literal Python shape tuple
- `ex044_mean_expanded_axes`: a tuple of zero-based aligned axes
- `ex044_compatible`: a Python `bool`
- `ex044_out_shape`: a literal Python shape tuple
- `ex044_out`: the requested PyTorch tensor

**Order:** Fill every shape/alignment prediction before writing tensor operations.

**Next concept:** Three-operand affine broadcast


In [ ]:
# Supplied visible fixture: inspect these values, but predict shapes without printing them.
images = torch.arange(120, dtype=DTYPE).reshape(2, 3, 4, 5)
mean = torch.tensor([[[[10.0]], [[20.0]], [[30.0]]]], dtype=DTYPE)
_register_case("ex044", {"images": images, "mean": mean})


In [ ]:
# Exercise 044: predict shapes first; do not use autograd.
# Define `ex044_images_shape`.
# Define `ex044_mean_shape`.
# Define `ex044_images_aligned_shape`.
# Define `ex044_images_expanded_axes`.
# Define `ex044_mean_aligned_shape`.
# Define `ex044_mean_expanded_axes`.
# Define `ex044_compatible`.
# Define `ex044_out_shape`.
# Define `ex044_out` with PyTorch tensor operations.
# Write your work below, then run the supplied test cell.


In [ ]:
# Supplied test: expected shapes and values remain private.
_check_private_value("ex044_images_shape", "ex044", "images_shape")
_check_private_value("ex044_mean_shape", "ex044", "mean_shape")
_check_private_value("ex044_images_aligned_shape", "ex044", "images_aligned_shape")
_check_private_value("ex044_images_expanded_axes", "ex044", "images_expanded_axes")
_check_private_value("ex044_mean_aligned_shape", "ex044", "mean_aligned_shape")
_check_private_value("ex044_mean_expanded_axes", "ex044", "mean_expanded_axes")
_check_private_value("ex044_compatible", "ex044", "compatible")
_check_private_value("ex044_out_shape", "ex044", "out_shape")
_check_private_tensor("ex044_out", "ex044", "out")


### Exercise 045 — Three-operand affine broadcast

**Purpose:** Track scale and bias tensors through one affine expression.

**Visible inputs:** `x`, `scale`, `bias`. Run the fixture below and read its construction code; it deliberately prints no shape answers.

**Operation to reason about:** `x * scale + bias`

**Task:** Compute an affine transform where scale follows features and bias follows the middle axis.

**Ingredients:** Audit all three operands against the final result.

**Required predictions and outputs:**

- `ex045_x_shape`: a literal Python shape tuple
- `ex045_scale_shape`: a literal Python shape tuple
- `ex045_bias_shape`: a literal Python shape tuple
- `ex045_x_aligned_shape`: a literal Python shape tuple
- `ex045_x_expanded_axes`: a tuple of zero-based aligned axes
- `ex045_scale_aligned_shape`: a literal Python shape tuple
- `ex045_scale_expanded_axes`: a tuple of zero-based aligned axes
- `ex045_bias_aligned_shape`: a literal Python shape tuple
- `ex045_bias_expanded_axes`: a tuple of zero-based aligned axes
- `ex045_compatible`: a Python `bool`
- `ex045_out_shape`: a literal Python shape tuple
- `ex045_out`: the requested PyTorch tensor

**Order:** Fill every shape/alignment prediction before writing tensor operations.

**Next concept:** Length two versus three columns


In [ ]:
# Supplied visible fixture: inspect these values, but predict shapes without printing them.
x = torch.arange(24, dtype=DTYPE).reshape(2, 3, 4)
scale = torch.tensor([1.0, 2.0, 3.0, 4.0], dtype=DTYPE)
bias = torch.tensor([[[0.1], [0.2], [0.3]]], dtype=DTYPE)
_register_case("ex045", {"x": x, "scale": scale, "bias": bias})


In [ ]:
# Exercise 045: predict shapes first; do not use autograd.
# Define `ex045_x_shape`.
# Define `ex045_scale_shape`.
# Define `ex045_bias_shape`.
# Define `ex045_x_aligned_shape`.
# Define `ex045_x_expanded_axes`.
# Define `ex045_scale_aligned_shape`.
# Define `ex045_scale_expanded_axes`.
# Define `ex045_bias_aligned_shape`.
# Define `ex045_bias_expanded_axes`.
# Define `ex045_compatible`.
# Define `ex045_out_shape`.
# Define `ex045_out` with PyTorch tensor operations.
# Write your work below, then run the supplied test cell.


In [ ]:
# Supplied test: expected shapes and values remain private.
_check_private_value("ex045_x_shape", "ex045", "x_shape")
_check_private_value("ex045_scale_shape", "ex045", "scale_shape")
_check_private_value("ex045_bias_shape", "ex045", "bias_shape")
_check_private_value("ex045_x_aligned_shape", "ex045", "x_aligned_shape")
_check_private_value("ex045_x_expanded_axes", "ex045", "x_expanded_axes")
_check_private_value("ex045_scale_aligned_shape", "ex045", "scale_aligned_shape")
_check_private_value("ex045_scale_expanded_axes", "ex045", "scale_expanded_axes")
_check_private_value("ex045_bias_aligned_shape", "ex045", "bias_aligned_shape")
_check_private_value("ex045_bias_expanded_axes", "ex045", "bias_expanded_axes")
_check_private_value("ex045_compatible", "ex045", "compatible")
_check_private_value("ex045_out_shape", "ex045", "out_shape")
_check_private_tensor("ex045_out", "ex045", "out")


## 5. Diagnosing incompatible shapes

Do not rely on trial and error. Right-align first, then list every axis where two unequal non-one sizes collide.


### Exercise 046 — Length two versus three columns

**Purpose:** Diagnose the exact mistake from Exercise 040 before running the operation.

**Visible inputs:** `a`, `b`. Run the fixture below and read its construction code; it deliberately prints no shape answers.

**Operation to reason about:** `a * b`

**Task:** Right-align the operands and identify every incompatible axis.

**Ingredients:** PyTorch does not know that the length-two vector semantically means rows.

**Required predictions and outputs:**

- `ex046_a_shape`: a literal Python shape tuple
- `ex046_b_shape`: a literal Python shape tuple
- `ex046_a_aligned_shape`: a literal Python shape tuple
- `ex046_b_aligned_shape`: a literal Python shape tuple
- `ex046_compatible`: a Python `bool`
- `ex046_incompatible_axes`: a tuple of zero-based aligned axes

**Order:** Fill every shape/alignment prediction before writing tensor operations.

**Next concept:** Transposed matrices


In [ ]:
# Supplied visible fixture: inspect these values, but predict shapes without printing them.
a = torch.tensor([2.0, -1.0], dtype=DTYPE)
b = torch.ones((2, 3), dtype=DTYPE)
_register_case("ex046", {"a": a, "b": b})


In [ ]:
# Exercise 046: predict shapes first; do not use autograd.
# Define `ex046_a_shape`.
# Define `ex046_b_shape`.
# Define `ex046_a_aligned_shape`.
# Define `ex046_b_aligned_shape`.
# Define `ex046_compatible`.
# Define `ex046_incompatible_axes`.
# Write your work below, then run the supplied test cell.


In [ ]:
# Supplied test: expected shapes and values remain private.
_check_private_value("ex046_a_shape", "ex046", "a_shape")
_check_private_value("ex046_b_shape", "ex046", "b_shape")
_check_private_value("ex046_a_aligned_shape", "ex046", "a_aligned_shape")
_check_private_value("ex046_b_aligned_shape", "ex046", "b_aligned_shape")
_check_private_value("ex046_compatible", "ex046", "compatible")
_check_private_value("ex046_incompatible_axes", "ex046", "incompatible_axes")


### Exercise 047 — Transposed matrices

**Purpose:** See why equal element counts do not imply compatible shapes.

**Visible inputs:** `a`, `b`. Run the fixture below and read its construction code; it deliberately prints no shape answers.

**Operation to reason about:** `a + b`

**Task:** Predict incompatibility without executing the addition yourself.

**Ingredients:** Compare axis sizes; do not flatten mentally.

**Required predictions and outputs:**

- `ex047_a_shape`: a literal Python shape tuple
- `ex047_b_shape`: a literal Python shape tuple
- `ex047_a_aligned_shape`: a literal Python shape tuple
- `ex047_b_aligned_shape`: a literal Python shape tuple
- `ex047_compatible`: a Python `bool`
- `ex047_incompatible_axes`: a tuple of zero-based aligned axes

**Order:** Fill every shape/alignment prediction before writing tensor operations.

**Next concept:** Different final matrix widths


In [ ]:
# Supplied visible fixture: inspect these values, but predict shapes without printing them.
a = torch.ones((3, 2), dtype=DTYPE)
b = torch.ones((2, 3), dtype=DTYPE)
_register_case("ex047", {"a": a, "b": b})


In [ ]:
# Exercise 047: predict shapes first; do not use autograd.
# Define `ex047_a_shape`.
# Define `ex047_b_shape`.
# Define `ex047_a_aligned_shape`.
# Define `ex047_b_aligned_shape`.
# Define `ex047_compatible`.
# Define `ex047_incompatible_axes`.
# Write your work below, then run the supplied test cell.


In [ ]:
# Supplied test: expected shapes and values remain private.
_check_private_value("ex047_a_shape", "ex047", "a_shape")
_check_private_value("ex047_b_shape", "ex047", "b_shape")
_check_private_value("ex047_a_aligned_shape", "ex047", "a_aligned_shape")
_check_private_value("ex047_b_aligned_shape", "ex047", "b_aligned_shape")
_check_private_value("ex047_compatible", "ex047", "compatible")
_check_private_value("ex047_incompatible_axes", "ex047", "incompatible_axes")


### Exercise 048 — Different final matrix widths

**Purpose:** Locate a mismatch on the final axis.

**Visible inputs:** `a`, `b`. Run the fixture below and read its construction code; it deliberately prints no shape answers.

**Operation to reason about:** `a + b`

**Task:** Identify the incompatible aligned axis.

**Ingredients:** Neither non-singleton width can expand.

**Required predictions and outputs:**

- `ex048_a_shape`: a literal Python shape tuple
- `ex048_b_shape`: a literal Python shape tuple
- `ex048_a_aligned_shape`: a literal Python shape tuple
- `ex048_b_aligned_shape`: a literal Python shape tuple
- `ex048_compatible`: a Python `bool`
- `ex048_incompatible_axes`: a tuple of zero-based aligned axes

**Order:** Fill every shape/alignment prediction before writing tensor operations.

**Next concept:** Reversed rank-2 axes


In [ ]:
# Supplied visible fixture: inspect these values, but predict shapes without printing them.
a = torch.ones((2, 4), dtype=DTYPE)
b = torch.ones((2, 3), dtype=DTYPE)
_register_case("ex048", {"a": a, "b": b})


In [ ]:
# Exercise 048: predict shapes first; do not use autograd.
# Define `ex048_a_shape`.
# Define `ex048_b_shape`.
# Define `ex048_a_aligned_shape`.
# Define `ex048_b_aligned_shape`.
# Define `ex048_compatible`.
# Define `ex048_incompatible_axes`.
# Write your work below, then run the supplied test cell.


In [ ]:
# Supplied test: expected shapes and values remain private.
_check_private_value("ex048_a_shape", "ex048", "a_shape")
_check_private_value("ex048_b_shape", "ex048", "b_shape")
_check_private_value("ex048_a_aligned_shape", "ex048", "a_aligned_shape")
_check_private_value("ex048_b_aligned_shape", "ex048", "b_aligned_shape")
_check_private_value("ex048_compatible", "ex048", "compatible")
_check_private_value("ex048_incompatible_axes", "ex048", "incompatible_axes")


### Exercise 049 — Reversed rank-2 axes

**Purpose:** Practice finding more than one incompatible axis.

**Visible inputs:** `a`, `b`. Run the fixture below and read its construction code; it deliberately prints no shape answers.

**Operation to reason about:** `a + b`

**Task:** Return all incompatible aligned axes.

**Ingredients:** Check every axis instead of stopping at the first mismatch.

**Required predictions and outputs:**

- `ex049_a_shape`: a literal Python shape tuple
- `ex049_b_shape`: a literal Python shape tuple
- `ex049_a_aligned_shape`: a literal Python shape tuple
- `ex049_b_aligned_shape`: a literal Python shape tuple
- `ex049_compatible`: a Python `bool`
- `ex049_incompatible_axes`: a tuple of zero-based aligned axes

**Order:** Fill every shape/alignment prediction before writing tensor operations.

**Next concept:** Wrong feature-vector length


In [ ]:
# Supplied visible fixture: inspect these values, but predict shapes without printing them.
a = torch.ones((2, 3), dtype=DTYPE)
b = torch.ones((3, 2), dtype=DTYPE)
_register_case("ex049", {"a": a, "b": b})


In [ ]:
# Exercise 049: predict shapes first; do not use autograd.
# Define `ex049_a_shape`.
# Define `ex049_b_shape`.
# Define `ex049_a_aligned_shape`.
# Define `ex049_b_aligned_shape`.
# Define `ex049_compatible`.
# Define `ex049_incompatible_axes`.
# Write your work below, then run the supplied test cell.


In [ ]:
# Supplied test: expected shapes and values remain private.
_check_private_value("ex049_a_shape", "ex049", "a_shape")
_check_private_value("ex049_b_shape", "ex049", "b_shape")
_check_private_value("ex049_a_aligned_shape", "ex049", "a_aligned_shape")
_check_private_value("ex049_b_aligned_shape", "ex049", "b_aligned_shape")
_check_private_value("ex049_compatible", "ex049", "compatible")
_check_private_value("ex049_incompatible_axes", "ex049", "incompatible_axes")


### Exercise 050 — Wrong feature-vector length

**Purpose:** Diagnose a vector that misses the final rank-3 axis.

**Visible inputs:** `a`, `b`. Run the fixture below and read its construction code; it deliberately prints no shape answers.

**Operation to reason about:** `a + b`

**Task:** Right-align the vector and locate the mismatch.

**Ingredients:** A shorter tensor always starts under the rightmost axes.

**Required predictions and outputs:**

- `ex050_a_shape`: a literal Python shape tuple
- `ex050_b_shape`: a literal Python shape tuple
- `ex050_a_aligned_shape`: a literal Python shape tuple
- `ex050_b_aligned_shape`: a literal Python shape tuple
- `ex050_compatible`: a Python `bool`
- `ex050_incompatible_axes`: a tuple of zero-based aligned axes

**Order:** Fill every shape/alignment prediction before writing tensor operations.

**Next concept:** Missing singleton in the middle


In [ ]:
# Supplied visible fixture: inspect these values, but predict shapes without printing them.
a = torch.ones((2, 3, 4), dtype=DTYPE)
b = torch.ones((3,), dtype=DTYPE)
_register_case("ex050", {"a": a, "b": b})


In [ ]:
# Exercise 050: predict shapes first; do not use autograd.
# Define `ex050_a_shape`.
# Define `ex050_b_shape`.
# Define `ex050_a_aligned_shape`.
# Define `ex050_b_aligned_shape`.
# Define `ex050_compatible`.
# Define `ex050_incompatible_axes`.
# Write your work below, then run the supplied test cell.


In [ ]:
# Supplied test: expected shapes and values remain private.
_check_private_value("ex050_a_shape", "ex050", "a_shape")
_check_private_value("ex050_b_shape", "ex050", "b_shape")
_check_private_value("ex050_a_aligned_shape", "ex050", "a_aligned_shape")
_check_private_value("ex050_b_aligned_shape", "ex050", "b_aligned_shape")
_check_private_value("ex050_compatible", "ex050", "compatible")
_check_private_value("ex050_incompatible_axes", "ex050", "incompatible_axes")


### Exercise 051 — Missing singleton in the middle

**Purpose:** See why `(2, 4)` does not mean batch-by-feature under rank 3.

**Visible inputs:** `a`, `b`. Run the fixture below and read its construction code; it deliberately prints no shape answers.

**Operation to reason about:** `a + b`

**Task:** Diagnose the operation and consider where a singleton would be needed.

**Ingredients:** Right alignment maps the first `2` under the middle axis, not batch.

**Required predictions and outputs:**

- `ex051_a_shape`: a literal Python shape tuple
- `ex051_b_shape`: a literal Python shape tuple
- `ex051_a_aligned_shape`: a literal Python shape tuple
- `ex051_b_aligned_shape`: a literal Python shape tuple
- `ex051_compatible`: a Python `bool`
- `ex051_incompatible_axes`: a tuple of zero-based aligned axes

**Order:** Fill every shape/alignment prediction before writing tensor operations.

**Next concept:** Missing singleton at the end


In [ ]:
# Supplied visible fixture: inspect these values, but predict shapes without printing them.
a = torch.ones((2, 3, 4), dtype=DTYPE)
b = torch.ones((2, 4), dtype=DTYPE)
_register_case("ex051", {"a": a, "b": b})


In [ ]:
# Exercise 051: predict shapes first; do not use autograd.
# Define `ex051_a_shape`.
# Define `ex051_b_shape`.
# Define `ex051_a_aligned_shape`.
# Define `ex051_b_aligned_shape`.
# Define `ex051_compatible`.
# Define `ex051_incompatible_axes`.
# Write your work below, then run the supplied test cell.


In [ ]:
# Supplied test: expected shapes and values remain private.
_check_private_value("ex051_a_shape", "ex051", "a_shape")
_check_private_value("ex051_b_shape", "ex051", "b_shape")
_check_private_value("ex051_a_aligned_shape", "ex051", "a_aligned_shape")
_check_private_value("ex051_b_aligned_shape", "ex051", "b_aligned_shape")
_check_private_value("ex051_compatible", "ex051", "compatible")
_check_private_value("ex051_incompatible_axes", "ex051", "incompatible_axes")


### Exercise 052 — Missing singleton at the end

**Purpose:** See why `(2, 3)` cannot automatically represent batch-by-middle under rank 3.

**Visible inputs:** `a`, `b`. Run the fixture below and read its construction code; it deliberately prints no shape answers.

**Operation to reason about:** `a + b`

**Task:** Identify all mismatched aligned axes.

**Ingredients:** To target leading axes, trailing singleton axes must be explicit.

**Required predictions and outputs:**

- `ex052_a_shape`: a literal Python shape tuple
- `ex052_b_shape`: a literal Python shape tuple
- `ex052_a_aligned_shape`: a literal Python shape tuple
- `ex052_b_aligned_shape`: a literal Python shape tuple
- `ex052_compatible`: a Python `bool`
- `ex052_incompatible_axes`: a tuple of zero-based aligned axes

**Order:** Fill every shape/alignment prediction before writing tensor operations.

**Next concept:** Two mismatches after padding


In [ ]:
# Supplied visible fixture: inspect these values, but predict shapes without printing them.
a = torch.ones((2, 3, 4), dtype=DTYPE)
b = torch.ones((2, 3), dtype=DTYPE)
_register_case("ex052", {"a": a, "b": b})


In [ ]:
# Exercise 052: predict shapes first; do not use autograd.
# Define `ex052_a_shape`.
# Define `ex052_b_shape`.
# Define `ex052_a_aligned_shape`.
# Define `ex052_b_aligned_shape`.
# Define `ex052_compatible`.
# Define `ex052_incompatible_axes`.
# Write your work below, then run the supplied test cell.


In [ ]:
# Supplied test: expected shapes and values remain private.
_check_private_value("ex052_a_shape", "ex052", "a_shape")
_check_private_value("ex052_b_shape", "ex052", "b_shape")
_check_private_value("ex052_a_aligned_shape", "ex052", "a_aligned_shape")
_check_private_value("ex052_b_aligned_shape", "ex052", "b_aligned_shape")
_check_private_value("ex052_compatible", "ex052", "compatible")
_check_private_value("ex052_incompatible_axes", "ex052", "incompatible_axes")


### Exercise 053 — Two mismatches after padding

**Purpose:** Audit a rank-3 operation with multiple incompatible dimensions.

**Visible inputs:** `a`, `b`. Run the fixture below and read its construction code; it deliberately prints no shape answers.

**Operation to reason about:** `a + b`

**Task:** Pad the shorter shape and return every incompatible axis.

**Ingredients:** Padding happens only on the left.

**Required predictions and outputs:**

- `ex053_a_shape`: a literal Python shape tuple
- `ex053_b_shape`: a literal Python shape tuple
- `ex053_a_aligned_shape`: a literal Python shape tuple
- `ex053_b_aligned_shape`: a literal Python shape tuple
- `ex053_compatible`: a Python `bool`
- `ex053_incompatible_axes`: a tuple of zero-based aligned axes

**Order:** Fill every shape/alignment prediction before writing tensor operations.

**Next concept:** Rank-4 crossed mismatch


In [ ]:
# Supplied visible fixture: inspect these values, but predict shapes without printing them.
a = torch.ones((2, 1, 4), dtype=DTYPE)
b = torch.ones((3, 5), dtype=DTYPE)
_register_case("ex053", {"a": a, "b": b})


In [ ]:
# Exercise 053: predict shapes first; do not use autograd.
# Define `ex053_a_shape`.
# Define `ex053_b_shape`.
# Define `ex053_a_aligned_shape`.
# Define `ex053_b_aligned_shape`.
# Define `ex053_compatible`.
# Define `ex053_incompatible_axes`.
# Write your work below, then run the supplied test cell.


In [ ]:
# Supplied test: expected shapes and values remain private.
_check_private_value("ex053_a_shape", "ex053", "a_shape")
_check_private_value("ex053_b_shape", "ex053", "b_shape")
_check_private_value("ex053_a_aligned_shape", "ex053", "a_aligned_shape")
_check_private_value("ex053_b_aligned_shape", "ex053", "b_aligned_shape")
_check_private_value("ex053_compatible", "ex053", "compatible")
_check_private_value("ex053_incompatible_axes", "ex053", "incompatible_axes")


### Exercise 054 — Rank-4 crossed mismatch

**Purpose:** Handle an invalid case where several singleton axes still cannot rescue the operation.

**Visible inputs:** `a`, `b`. Run the fixture below and read its construction code; it deliberately prints no shape answers.

**Operation to reason about:** `a + b`

**Task:** Find all axes containing two unequal non-one sizes.

**Ingredients:** An axis is compatible only when sizes match or at least one is `1`.

**Required predictions and outputs:**

- `ex054_a_shape`: a literal Python shape tuple
- `ex054_b_shape`: a literal Python shape tuple
- `ex054_a_aligned_shape`: a literal Python shape tuple
- `ex054_b_aligned_shape`: a literal Python shape tuple
- `ex054_compatible`: a Python `bool`
- `ex054_incompatible_axes`: a tuple of zero-based aligned axes

**Order:** Fill every shape/alignment prediction before writing tensor operations.

**Next concept:** Three-way where mismatch


In [ ]:
# Supplied visible fixture: inspect these values, but predict shapes without printing them.
a = torch.ones((2, 3, 1, 5), dtype=DTYPE)
b = torch.ones((1, 4, 6, 1), dtype=DTYPE)
_register_case("ex054", {"a": a, "b": b})


In [ ]:
# Exercise 054: predict shapes first; do not use autograd.
# Define `ex054_a_shape`.
# Define `ex054_b_shape`.
# Define `ex054_a_aligned_shape`.
# Define `ex054_b_aligned_shape`.
# Define `ex054_compatible`.
# Define `ex054_incompatible_axes`.
# Write your work below, then run the supplied test cell.


In [ ]:
# Supplied test: expected shapes and values remain private.
_check_private_value("ex054_a_shape", "ex054", "a_shape")
_check_private_value("ex054_b_shape", "ex054", "b_shape")
_check_private_value("ex054_a_aligned_shape", "ex054", "a_aligned_shape")
_check_private_value("ex054_b_aligned_shape", "ex054", "b_aligned_shape")
_check_private_value("ex054_compatible", "ex054", "compatible")
_check_private_value("ex054_incompatible_axes", "ex054", "incompatible_axes")


### Exercise 055 — Three-way where mismatch

**Purpose:** Apply compatibility rules across a condition and both value tensors.

**Visible inputs:** `mask`, `x`, `y`. Run the fixture below and read its construction code; it deliberately prints no shape answers.

**Operation to reason about:** `torch.where(mask, x, y)`

**Task:** Align all three shapes and identify the incompatible axes.

**Ingredients:** All operands in `torch.where` must share one broadcastable result shape.

**Required predictions and outputs:**

- `ex055_mask_shape`: a literal Python shape tuple
- `ex055_x_shape`: a literal Python shape tuple
- `ex055_y_shape`: a literal Python shape tuple
- `ex055_mask_aligned_shape`: a literal Python shape tuple
- `ex055_x_aligned_shape`: a literal Python shape tuple
- `ex055_y_aligned_shape`: a literal Python shape tuple
- `ex055_compatible`: a Python `bool`
- `ex055_incompatible_axes`: a tuple of zero-based aligned axes

**Order:** Fill every shape/alignment prediction before writing tensor operations.

**Next concept:** Expand a row vector


In [ ]:
# Supplied visible fixture: inspect these values, but predict shapes without printing them.
mask = torch.ones((2, 3), dtype=torch.bool)
x = torch.ones((2, 1), dtype=DTYPE)
y = torch.ones((4, 3), dtype=DTYPE)
_register_case("ex055", {"mask": mask, "x": x, "y": y})


In [ ]:
# Exercise 055: predict shapes first; do not use autograd.
# Define `ex055_mask_shape`.
# Define `ex055_x_shape`.
# Define `ex055_y_shape`.
# Define `ex055_mask_aligned_shape`.
# Define `ex055_x_aligned_shape`.
# Define `ex055_y_aligned_shape`.
# Define `ex055_compatible`.
# Define `ex055_incompatible_axes`.
# Write your work below, then run the supplied test cell.


In [ ]:
# Supplied test: expected shapes and values remain private.
_check_private_value("ex055_mask_shape", "ex055", "mask_shape")
_check_private_value("ex055_x_shape", "ex055", "x_shape")
_check_private_value("ex055_y_shape", "ex055", "y_shape")
_check_private_value("ex055_mask_aligned_shape", "ex055", "mask_aligned_shape")
_check_private_value("ex055_x_aligned_shape", "ex055", "x_aligned_shape")
_check_private_value("ex055_y_aligned_shape", "ex055", "y_aligned_shape")
_check_private_value("ex055_compatible", "ex055", "compatible")
_check_private_value("ex055_incompatible_axes", "ex055", "incompatible_axes")


## 6. `expand`, `broadcast_to`, `repeat`, and storage

`expand` creates stride-zero views over singleton axes; `repeat` materializes repeated values. Similar visible results can have different storage semantics.


### Exercise 056 — Expand a row vector

**Purpose:** Use a stride-zero view instead of physically copying a row.

**Visible inputs:** `source`, `target`. Run the fixture below and read its construction code; it deliberately prints no shape answers.

**Operation to reason about:** `source.expand_as(target)`

**Task:** Expand `source` to the same shape as `target`.

**Ingredients:** `expand_as` may enlarge singleton dimensions only.

**Required predictions and outputs:**

- `ex056_source_shape`: a literal Python shape tuple
- `ex056_target_shape`: a literal Python shape tuple
- `ex056_out_shape`: a literal Python shape tuple
- `ex056_out`: the requested PyTorch tensor

**Order:** Fill every shape/alignment prediction before writing tensor operations.

**Next concept:** Expand a column vector


In [ ]:
# Supplied visible fixture: inspect these values, but predict shapes without printing them.
source = torch.tensor([[1.0, 2.0, 3.0]], dtype=DTYPE)
target = torch.empty((2, 3), dtype=DTYPE)
_register_case("ex056", {"source": source, "target": target})


In [ ]:
# Exercise 056: predict shapes first; do not use autograd.
# Define `ex056_source_shape`.
# Define `ex056_target_shape`.
# Define `ex056_out_shape`.
# Define `ex056_out` with PyTorch tensor operations.
# Write your work below, then run the supplied test cell.


In [ ]:
# Supplied test: expected shapes and values remain private.
_check_private_value("ex056_source_shape", "ex056", "source_shape")
_check_private_value("ex056_target_shape", "ex056", "target_shape")
_check_private_value("ex056_out_shape", "ex056", "out_shape")
_check_private_tensor("ex056_out", "ex056", "out")


### Exercise 057 — Expand a column vector

**Purpose:** Expand a singleton column across three columns.

**Visible inputs:** `source`, `target`. Run the fixture below and read its construction code; it deliberately prints no shape answers.

**Operation to reason about:** `source.expand_as(target)`

**Task:** Expand `source` to match `target`.

**Ingredients:** The existing row dimension must already match.

**Required predictions and outputs:**

- `ex057_source_shape`: a literal Python shape tuple
- `ex057_target_shape`: a literal Python shape tuple
- `ex057_out_shape`: a literal Python shape tuple
- `ex057_out`: the requested PyTorch tensor

**Order:** Fill every shape/alignment prediction before writing tensor operations.

**Next concept:** Expand a scalar


In [ ]:
# Supplied visible fixture: inspect these values, but predict shapes without printing them.
source = torch.tensor([[1.0], [2.0]], dtype=DTYPE)
target = torch.empty((2, 3), dtype=DTYPE)
_register_case("ex057", {"source": source, "target": target})


In [20]:
# Exercise 057: predict shapes first; do not use autograd.
# Define `ex057_source_shape`.
# Define `ex057_target_shape`.
# Define `ex057_out_shape`.
# Define `ex057_out` with PyTorch tensor operations.
# Write your work below, then run the supplied test cell.


In [21]:
# Supplied test: expected shapes and values remain private.
_check_private_value("ex057_source_shape", "ex057", "source_shape")
_check_private_value("ex057_target_shape", "ex057", "target_shape")
_check_private_value("ex057_out_shape", "ex057", "out_shape")
_check_private_tensor("ex057_out", "ex057", "out")


AssertionError: Define `ex057_source_shape` in the answer cell first.

### Exercise 058 — Expand a scalar

**Purpose:** View one scalar at every location without allocating copies.

**Visible inputs:** `source`, `target`. Run the fixture below and read its construction code; it deliberately prints no shape answers.

**Operation to reason about:** `source.expand_as(target)`

**Task:** Expand the scalar to the target tensor's shape.

**Ingredients:** A scalar can conceptually acquire any number of singleton axes.

**Required predictions and outputs:**

- `ex058_source_shape`: a literal Python shape tuple
- `ex058_target_shape`: a literal Python shape tuple
- `ex058_out_shape`: a literal Python shape tuple
- `ex058_out`: the requested PyTorch tensor

**Order:** Fill every shape/alignment prediction before writing tensor operations.

**Next concept:** Expand rank 3


In [ ]:
# Supplied visible fixture: inspect these values, but predict shapes without printing them.
source = torch.tensor(7.0, dtype=DTYPE)
target = torch.empty((2, 3), dtype=DTYPE)
_register_case("ex058", {"source": source, "target": target})


In [ ]:
# Exercise 058: predict shapes first; do not use autograd.
# Define `ex058_source_shape`.
# Define `ex058_target_shape`.
# Define `ex058_out_shape`.
# Define `ex058_out` with PyTorch tensor operations.
# Write your work below, then run the supplied test cell.


In [ ]:
# Supplied test: expected shapes and values remain private.
_check_private_value("ex058_source_shape", "ex058", "source_shape")
_check_private_value("ex058_target_shape", "ex058", "target_shape")
_check_private_value("ex058_out_shape", "ex058", "out_shape")
_check_private_tensor("ex058_out", "ex058", "out")


### Exercise 059 — Expand rank 3

**Purpose:** Expand batch and feature singleton axes simultaneously.

**Visible inputs:** `source`, `target`. Run the fixture below and read its construction code; it deliberately prints no shape answers.

**Operation to reason about:** `source.expand_as(target)`

**Task:** Expand `source` across batch and final axes.

**Ingredients:** Only size-one axes may change size.

**Required predictions and outputs:**

- `ex059_source_shape`: a literal Python shape tuple
- `ex059_target_shape`: a literal Python shape tuple
- `ex059_out_shape`: a literal Python shape tuple
- `ex059_out`: the requested PyTorch tensor

**Order:** Fill every shape/alignment prediction before writing tensor operations.

**Next concept:** Use broadcast_to


In [ ]:
# Supplied visible fixture: inspect these values, but predict shapes without printing them.
source = torch.arange(3, dtype=DTYPE).reshape(1, 3, 1)
target = torch.empty((2, 3, 4), dtype=DTYPE)
_register_case("ex059", {"source": source, "target": target})


In [ ]:
# Exercise 059: predict shapes first; do not use autograd.
# Define `ex059_source_shape`.
# Define `ex059_target_shape`.
# Define `ex059_out_shape`.
# Define `ex059_out` with PyTorch tensor operations.
# Write your work below, then run the supplied test cell.


In [ ]:
# Supplied test: expected shapes and values remain private.
_check_private_value("ex059_source_shape", "ex059", "source_shape")
_check_private_value("ex059_target_shape", "ex059", "target_shape")
_check_private_value("ex059_out_shape", "ex059", "out_shape")
_check_private_tensor("ex059_out", "ex059", "out")


### Exercise 060 — Use broadcast_to

**Purpose:** Express a target shape directly with `torch.broadcast_to`.

**Visible inputs:** `source`, `target`. Run the fixture below and read its construction code; it deliberately prints no shape answers.

**Operation to reason about:** `torch.broadcast_to(source, target.shape)`

**Task:** Broadcast `source` to the target shape.

**Ingredients:** The same right-alignment rules apply.

**Required predictions and outputs:**

- `ex060_source_shape`: a literal Python shape tuple
- `ex060_target_shape`: a literal Python shape tuple
- `ex060_out_shape`: a literal Python shape tuple
- `ex060_out`: the requested PyTorch tensor

**Order:** Fill every shape/alignment prediction before writing tensor operations.

**Next concept:** Expand to an image batch


In [ ]:
# Supplied visible fixture: inspect these values, but predict shapes without printing them.
source = torch.tensor([1.0, 2.0, 3.0], dtype=DTYPE)
target = torch.empty((2, 3), dtype=DTYPE)
_register_case("ex060", {"source": source, "target": target})


In [ ]:
# Exercise 060: predict shapes first; do not use autograd.
# Define `ex060_source_shape`.
# Define `ex060_target_shape`.
# Define `ex060_out_shape`.
# Define `ex060_out` with PyTorch tensor operations.
# Write your work below, then run the supplied test cell.


In [ ]:
# Supplied test: expected shapes and values remain private.
_check_private_value("ex060_source_shape", "ex060", "source_shape")
_check_private_value("ex060_target_shape", "ex060", "target_shape")
_check_private_value("ex060_out_shape", "ex060", "out_shape")
_check_private_tensor("ex060_out", "ex060", "out")


### Exercise 061 — Expand to an image batch

**Purpose:** Expand channel parameters over batch and spatial dimensions.

**Visible inputs:** `source`, `target`. Run the fixture below and read its construction code; it deliberately prints no shape answers.

**Operation to reason about:** `source.expand_as(target)`

**Task:** Expand the channel parameters to the complete image shape.

**Ingredients:** Batch, height, and width are singleton in `source`.

**Required predictions and outputs:**

- `ex061_source_shape`: a literal Python shape tuple
- `ex061_target_shape`: a literal Python shape tuple
- `ex061_out_shape`: a literal Python shape tuple
- `ex061_out`: the requested PyTorch tensor

**Order:** Fill every shape/alignment prediction before writing tensor operations.

**Next concept:** Repeat a row


In [ ]:
# Supplied visible fixture: inspect these values, but predict shapes without printing them.
source = torch.tensor([[[[1.0]], [[2.0]], [[3.0]]]], dtype=DTYPE)
target = torch.empty((2, 3, 4, 5), dtype=DTYPE)
_register_case("ex061", {"source": source, "target": target})


In [ ]:
# Exercise 061: predict shapes first; do not use autograd.
# Define `ex061_source_shape`.
# Define `ex061_target_shape`.
# Define `ex061_out_shape`.
# Define `ex061_out` with PyTorch tensor operations.
# Write your work below, then run the supplied test cell.


In [ ]:
# Supplied test: expected shapes and values remain private.
_check_private_value("ex061_source_shape", "ex061", "source_shape")
_check_private_value("ex061_target_shape", "ex061", "target_shape")
_check_private_value("ex061_out_shape", "ex061", "out_shape")
_check_private_tensor("ex061_out", "ex061", "out")


### Exercise 062 — Repeat a row

**Purpose:** Contrast physical repetition with a broadcasted view.

**Visible inputs:** `source`. Run the fixture below and read its construction code; it deliberately prints no shape answers.

**Operation to reason about:** `source.repeat(2, 1)`

**Task:** Repeat the row twice.

**Ingredients:** `repeat` takes one repetition count per axis and allocates repeated data.

**Required predictions and outputs:**

- `ex062_source_shape`: a literal Python shape tuple
- `ex062_out_shape`: a literal Python shape tuple
- `ex062_out`: the requested PyTorch tensor

**Order:** Fill every shape/alignment prediction before writing tensor operations.

**Next concept:** Repeat a column


In [ ]:
# Supplied visible fixture: inspect these values, but predict shapes without printing them.
source = torch.tensor([[1.0, 2.0, 3.0]], dtype=DTYPE)
_register_case("ex062", {"source": source})


In [ ]:
# Exercise 062: predict shapes first; do not use autograd.
# Define `ex062_source_shape`.
# Define `ex062_out_shape`.
# Define `ex062_out` with PyTorch tensor operations.
# Write your work below, then run the supplied test cell.


In [ ]:
# Supplied test: expected shapes and values remain private.
_check_private_value("ex062_source_shape", "ex062", "source_shape")
_check_private_value("ex062_out_shape", "ex062", "out_shape")
_check_private_tensor("ex062_out", "ex062", "out")


### Exercise 063 — Repeat a column

**Purpose:** Repeat values along the singleton column axis.

**Visible inputs:** `source`. Run the fixture below and read its construction code; it deliberately prints no shape answers.

**Operation to reason about:** `source.repeat(1, 3)`

**Task:** Repeat each row value across three columns.

**Ingredients:** The first repetition count preserves rows; the second repeats columns.

**Required predictions and outputs:**

- `ex063_source_shape`: a literal Python shape tuple
- `ex063_out_shape`: a literal Python shape tuple
- `ex063_out`: the requested PyTorch tensor

**Order:** Fill every shape/alignment prediction before writing tensor operations.

**Next concept:** Expanded view shares storage


In [ ]:
# Supplied visible fixture: inspect these values, but predict shapes without printing them.
source = torch.tensor([[1.0], [2.0]], dtype=DTYPE)
_register_case("ex063", {"source": source})


In [ ]:
# Exercise 063: predict shapes first; do not use autograd.
# Define `ex063_source_shape`.
# Define `ex063_out_shape`.
# Define `ex063_out` with PyTorch tensor operations.
# Write your work below, then run the supplied test cell.


In [ ]:
# Supplied test: expected shapes and values remain private.
_check_private_value("ex063_source_shape", "ex063", "source_shape")
_check_private_value("ex063_out_shape", "ex063", "out_shape")
_check_private_tensor("ex063_out", "ex063", "out")


### Exercise 064 — Expanded view shares storage

**Purpose:** Connect broadcasting views to storage aliasing.

**Visible inputs:** `source`. Run the fixture below and read its construction code; it deliberately prints no shape answers.

**Operation to reason about:** `source.expand(2, 3)`

**Task:** Create `ex064_out`, then predict whether it shares storage with `source`.

**Ingredients:** `expand` normally changes strides rather than copying values.

**Required predictions and outputs:**

- `ex064_source_shape`: a literal Python shape tuple
- `ex064_out_shape`: a literal Python shape tuple
- `ex064_shares_storage`: a Python `bool`
- `ex064_out`: the requested PyTorch tensor

**Order:** Fill every shape/alignment prediction before writing tensor operations.

**Next concept:** Materialize an expanded view


In [ ]:
# Supplied visible fixture: inspect these values, but predict shapes without printing them.
source = torch.tensor([[1.0, 2.0, 3.0]], dtype=DTYPE)
_register_case("ex064", {"source": source})


In [ ]:
# Exercise 064: predict shapes first; do not use autograd.
# Define `ex064_source_shape`.
# Define `ex064_out_shape`.
# Define `ex064_shares_storage`.
# Define `ex064_out` with PyTorch tensor operations.
# Write your work below, then run the supplied test cell.


In [ ]:
# Supplied test: expected shapes and values remain private.
_check_private_value("ex064_source_shape", "ex064", "source_shape")
_check_private_value("ex064_out_shape", "ex064", "out_shape")
_check_private_value("ex064_shares_storage", "ex064", "shares_storage")
_check_private_tensor("ex064_out", "ex064", "out")


### Exercise 065 — Materialize an expanded view

**Purpose:** Use `clone` to turn an expanded view into independent storage.

**Visible inputs:** `source`. Run the fixture below and read its construction code; it deliberately prints no shape answers.

**Operation to reason about:** `source.expand(2, 3).clone()`

**Task:** Expand and clone into `ex065_out`, then predict storage sharing.

**Ingredients:** Cloning materializes independent values.

**Required predictions and outputs:**

- `ex065_source_shape`: a literal Python shape tuple
- `ex065_out_shape`: a literal Python shape tuple
- `ex065_shares_storage`: a Python `bool`
- `ex065_out`: the requested PyTorch tensor

**Order:** Fill every shape/alignment prediction before writing tensor operations.

**Next concept:** Where with scalar fallback


In [ ]:
# Supplied visible fixture: inspect these values, but predict shapes without printing them.
source = torch.tensor([[1.0, 2.0, 3.0]], dtype=DTYPE)
_register_case("ex065", {"source": source})


In [ ]:
# Exercise 065: predict shapes first; do not use autograd.
# Define `ex065_source_shape`.
# Define `ex065_out_shape`.
# Define `ex065_shares_storage`.
# Define `ex065_out` with PyTorch tensor operations.
# Write your work below, then run the supplied test cell.


In [ ]:
# Supplied test: expected shapes and values remain private.
_check_private_value("ex065_source_shape", "ex065", "source_shape")
_check_private_value("ex065_out_shape", "ex065", "out_shape")
_check_private_value("ex065_shares_storage", "ex065", "shares_storage")
_check_private_tensor("ex065_out", "ex065", "out")


## 7. Multiple operands, masks, and elementwise functions

Operators such as `torch.where`, `maximum`, `minimum`, and `lerp` broadcast all participating tensors to one common shape.


### Exercise 066 — Where with scalar fallback

**Purpose:** Broadcast one scalar fallback through a matrix condition.

**Visible inputs:** `mask`, `x`, `fallback`. Run the fixture below and read its construction code; it deliberately prints no shape answers.

**Operation to reason about:** `torch.where(mask, x, fallback)`

**Task:** Select matrix values where true and the scalar elsewhere.

**Ingredients:** Condition, true branch, and false branch all broadcast together.

**Required predictions and outputs:**

- `ex066_mask_shape`: a literal Python shape tuple
- `ex066_x_shape`: a literal Python shape tuple
- `ex066_fallback_shape`: a literal Python shape tuple
- `ex066_mask_aligned_shape`: a literal Python shape tuple
- `ex066_mask_expanded_axes`: a tuple of zero-based aligned axes
- `ex066_x_aligned_shape`: a literal Python shape tuple
- `ex066_x_expanded_axes`: a tuple of zero-based aligned axes
- `ex066_fallback_aligned_shape`: a literal Python shape tuple
- `ex066_fallback_expanded_axes`: a tuple of zero-based aligned axes
- `ex066_compatible`: a Python `bool`
- `ex066_out_shape`: a literal Python shape tuple
- `ex066_out`: the requested PyTorch tensor

**Order:** Fill every shape/alignment prediction before writing tensor operations.

**Next concept:** Where with a row mask


In [ ]:
# Supplied visible fixture: inspect these values, but predict shapes without printing them.
mask = torch.tensor([[True, False, True], [False, True, False]])
x = torch.arange(6, dtype=DTYPE).reshape(2, 3)
fallback = torch.tensor(-1.0, dtype=DTYPE)
_register_case("ex066", {"mask": mask, "x": x, "fallback": fallback})


In [ ]:
# Exercise 066: predict shapes first; do not use autograd.
# Define `ex066_mask_shape`.
# Define `ex066_x_shape`.
# Define `ex066_fallback_shape`.
# Define `ex066_mask_aligned_shape`.
# Define `ex066_mask_expanded_axes`.
# Define `ex066_x_aligned_shape`.
# Define `ex066_x_expanded_axes`.
# Define `ex066_fallback_aligned_shape`.
# Define `ex066_fallback_expanded_axes`.
# Define `ex066_compatible`.
# Define `ex066_out_shape`.
# Define `ex066_out` with PyTorch tensor operations.
# Write your work below, then run the supplied test cell.


In [ ]:
# Supplied test: expected shapes and values remain private.
_check_private_value("ex066_mask_shape", "ex066", "mask_shape")
_check_private_value("ex066_x_shape", "ex066", "x_shape")
_check_private_value("ex066_fallback_shape", "ex066", "fallback_shape")
_check_private_value("ex066_mask_aligned_shape", "ex066", "mask_aligned_shape")
_check_private_value("ex066_mask_expanded_axes", "ex066", "mask_expanded_axes")
_check_private_value("ex066_x_aligned_shape", "ex066", "x_aligned_shape")
_check_private_value("ex066_x_expanded_axes", "ex066", "x_expanded_axes")
_check_private_value("ex066_fallback_aligned_shape", "ex066", "fallback_aligned_shape")
_check_private_value("ex066_fallback_expanded_axes", "ex066", "fallback_expanded_axes")
_check_private_value("ex066_compatible", "ex066", "compatible")
_check_private_value("ex066_out_shape", "ex066", "out_shape")
_check_private_tensor("ex066_out", "ex066", "out")


### Exercise 067 — Where with a row mask

**Purpose:** Expand one Boolean decision per row across columns.

**Visible inputs:** `mask`, `x`, `y`. Run the fixture below and read its construction code; it deliberately prints no shape answers.

**Operation to reason about:** `torch.where(mask, x, y)`

**Task:** Choose all of `x`'s first row and all of `y`'s second row.

**Ingredients:** The mask's final singleton axis expands.

**Required predictions and outputs:**

- `ex067_mask_shape`: a literal Python shape tuple
- `ex067_x_shape`: a literal Python shape tuple
- `ex067_y_shape`: a literal Python shape tuple
- `ex067_mask_aligned_shape`: a literal Python shape tuple
- `ex067_mask_expanded_axes`: a tuple of zero-based aligned axes
- `ex067_x_aligned_shape`: a literal Python shape tuple
- `ex067_x_expanded_axes`: a tuple of zero-based aligned axes
- `ex067_y_aligned_shape`: a literal Python shape tuple
- `ex067_y_expanded_axes`: a tuple of zero-based aligned axes
- `ex067_compatible`: a Python `bool`
- `ex067_out_shape`: a literal Python shape tuple
- `ex067_out`: the requested PyTorch tensor

**Order:** Fill every shape/alignment prediction before writing tensor operations.

**Next concept:** Where with a column mask


In [ ]:
# Supplied visible fixture: inspect these values, but predict shapes without printing them.
mask = torch.tensor([[True], [False]])
x = torch.arange(6, dtype=DTYPE).reshape(2, 3)
y = -torch.ones((2, 3), dtype=DTYPE)
_register_case("ex067", {"mask": mask, "x": x, "y": y})


In [ ]:
# Exercise 067: predict shapes first; do not use autograd.
# Define `ex067_mask_shape`.
# Define `ex067_x_shape`.
# Define `ex067_y_shape`.
# Define `ex067_mask_aligned_shape`.
# Define `ex067_mask_expanded_axes`.
# Define `ex067_x_aligned_shape`.
# Define `ex067_x_expanded_axes`.
# Define `ex067_y_aligned_shape`.
# Define `ex067_y_expanded_axes`.
# Define `ex067_compatible`.
# Define `ex067_out_shape`.
# Define `ex067_out` with PyTorch tensor operations.
# Write your work below, then run the supplied test cell.


In [ ]:
# Supplied test: expected shapes and values remain private.
_check_private_value("ex067_mask_shape", "ex067", "mask_shape")
_check_private_value("ex067_x_shape", "ex067", "x_shape")
_check_private_value("ex067_y_shape", "ex067", "y_shape")
_check_private_value("ex067_mask_aligned_shape", "ex067", "mask_aligned_shape")
_check_private_value("ex067_mask_expanded_axes", "ex067", "mask_expanded_axes")
_check_private_value("ex067_x_aligned_shape", "ex067", "x_aligned_shape")
_check_private_value("ex067_x_expanded_axes", "ex067", "x_expanded_axes")
_check_private_value("ex067_y_aligned_shape", "ex067", "y_aligned_shape")
_check_private_value("ex067_y_expanded_axes", "ex067", "y_expanded_axes")
_check_private_value("ex067_compatible", "ex067", "compatible")
_check_private_value("ex067_out_shape", "ex067", "out_shape")
_check_private_tensor("ex067_out", "ex067", "out")


### Exercise 068 — Where with a column mask

**Purpose:** Use one Boolean choice per column across every row.

**Visible inputs:** `mask`, `x`, `y`. Run the fixture below and read its construction code; it deliberately prints no shape answers.

**Operation to reason about:** `torch.where(mask, x, y)`

**Task:** Select columns from `x` or `y` according to `mask`.

**Ingredients:** A length-three mask right-aligns with columns.

**Required predictions and outputs:**

- `ex068_mask_shape`: a literal Python shape tuple
- `ex068_x_shape`: a literal Python shape tuple
- `ex068_y_shape`: a literal Python shape tuple
- `ex068_mask_aligned_shape`: a literal Python shape tuple
- `ex068_mask_expanded_axes`: a tuple of zero-based aligned axes
- `ex068_x_aligned_shape`: a literal Python shape tuple
- `ex068_x_expanded_axes`: a tuple of zero-based aligned axes
- `ex068_y_aligned_shape`: a literal Python shape tuple
- `ex068_y_expanded_axes`: a tuple of zero-based aligned axes
- `ex068_compatible`: a Python `bool`
- `ex068_out_shape`: a literal Python shape tuple
- `ex068_out`: the requested PyTorch tensor

**Order:** Fill every shape/alignment prediction before writing tensor operations.

**Next concept:** Three-way outer where


In [ ]:
# Supplied visible fixture: inspect these values, but predict shapes without printing them.
mask = torch.tensor([True, False, True])
x = torch.arange(6, dtype=DTYPE).reshape(2, 3)
y = torch.full((2, 3), 99.0, dtype=DTYPE)
_register_case("ex068", {"mask": mask, "x": x, "y": y})


In [ ]:
# Exercise 068: predict shapes first; do not use autograd.
# Define `ex068_mask_shape`.
# Define `ex068_x_shape`.
# Define `ex068_y_shape`.
# Define `ex068_mask_aligned_shape`.
# Define `ex068_mask_expanded_axes`.
# Define `ex068_x_aligned_shape`.
# Define `ex068_x_expanded_axes`.
# Define `ex068_y_aligned_shape`.
# Define `ex068_y_expanded_axes`.
# Define `ex068_compatible`.
# Define `ex068_out_shape`.
# Define `ex068_out` with PyTorch tensor operations.
# Write your work below, then run the supplied test cell.


In [ ]:
# Supplied test: expected shapes and values remain private.
_check_private_value("ex068_mask_shape", "ex068", "mask_shape")
_check_private_value("ex068_x_shape", "ex068", "x_shape")
_check_private_value("ex068_y_shape", "ex068", "y_shape")
_check_private_value("ex068_mask_aligned_shape", "ex068", "mask_aligned_shape")
_check_private_value("ex068_mask_expanded_axes", "ex068", "mask_expanded_axes")
_check_private_value("ex068_x_aligned_shape", "ex068", "x_aligned_shape")
_check_private_value("ex068_x_expanded_axes", "ex068", "x_expanded_axes")
_check_private_value("ex068_y_aligned_shape", "ex068", "y_aligned_shape")
_check_private_value("ex068_y_expanded_axes", "ex068", "y_expanded_axes")
_check_private_value("ex068_compatible", "ex068", "compatible")
_check_private_value("ex068_out_shape", "ex068", "out_shape")
_check_private_tensor("ex068_out", "ex068", "out")


### Exercise 069 — Three-way outer where

**Purpose:** Broadcast a column condition, row values, and one scalar fallback.

**Visible inputs:** `mask`, `x`, `y`. Run the fixture below and read its construction code; it deliberately prints no shape answers.

**Operation to reason about:** `torch.where(mask, x, y)`

**Task:** Predict all expansion axes and compute the result.

**Ingredients:** Find one common result shape for all three operands.

**Required predictions and outputs:**

- `ex069_mask_shape`: a literal Python shape tuple
- `ex069_x_shape`: a literal Python shape tuple
- `ex069_y_shape`: a literal Python shape tuple
- `ex069_mask_aligned_shape`: a literal Python shape tuple
- `ex069_mask_expanded_axes`: a tuple of zero-based aligned axes
- `ex069_x_aligned_shape`: a literal Python shape tuple
- `ex069_x_expanded_axes`: a tuple of zero-based aligned axes
- `ex069_y_aligned_shape`: a literal Python shape tuple
- `ex069_y_expanded_axes`: a tuple of zero-based aligned axes
- `ex069_compatible`: a Python `bool`
- `ex069_out_shape`: a literal Python shape tuple
- `ex069_out`: the requested PyTorch tensor

**Order:** Fill every shape/alignment prediction before writing tensor operations.

**Next concept:** Maximum with row thresholds


In [ ]:
# Supplied visible fixture: inspect these values, but predict shapes without printing them.
mask = torch.tensor([[True], [False]])
x = torch.tensor([[1.0, 2.0, 3.0]], dtype=DTYPE)
y = torch.tensor(-5.0, dtype=DTYPE)
_register_case("ex069", {"mask": mask, "x": x, "y": y})


In [ ]:
# Exercise 069: predict shapes first; do not use autograd.
# Define `ex069_mask_shape`.
# Define `ex069_x_shape`.
# Define `ex069_y_shape`.
# Define `ex069_mask_aligned_shape`.
# Define `ex069_mask_expanded_axes`.
# Define `ex069_x_aligned_shape`.
# Define `ex069_x_expanded_axes`.
# Define `ex069_y_aligned_shape`.
# Define `ex069_y_expanded_axes`.
# Define `ex069_compatible`.
# Define `ex069_out_shape`.
# Define `ex069_out` with PyTorch tensor operations.
# Write your work below, then run the supplied test cell.


In [ ]:
# Supplied test: expected shapes and values remain private.
_check_private_value("ex069_mask_shape", "ex069", "mask_shape")
_check_private_value("ex069_x_shape", "ex069", "x_shape")
_check_private_value("ex069_y_shape", "ex069", "y_shape")
_check_private_value("ex069_mask_aligned_shape", "ex069", "mask_aligned_shape")
_check_private_value("ex069_mask_expanded_axes", "ex069", "mask_expanded_axes")
_check_private_value("ex069_x_aligned_shape", "ex069", "x_aligned_shape")
_check_private_value("ex069_x_expanded_axes", "ex069", "x_expanded_axes")
_check_private_value("ex069_y_aligned_shape", "ex069", "y_aligned_shape")
_check_private_value("ex069_y_expanded_axes", "ex069", "y_expanded_axes")
_check_private_value("ex069_compatible", "ex069", "compatible")
_check_private_value("ex069_out_shape", "ex069", "out_shape")
_check_private_tensor("ex069_out", "ex069", "out")


### Exercise 070 — Maximum with row thresholds

**Purpose:** Broadcast one threshold per feature column.

**Visible inputs:** `x`, `threshold`. Run the fixture below and read its construction code; it deliberately prints no shape answers.

**Operation to reason about:** `torch.maximum(x, threshold)`

**Task:** Clamp each column from below with its own threshold.

**Ingredients:** Binary elementwise functions use the same broadcasting rules as operators.

**Required predictions and outputs:**

- `ex070_x_shape`: a literal Python shape tuple
- `ex070_threshold_shape`: a literal Python shape tuple
- `ex070_x_aligned_shape`: a literal Python shape tuple
- `ex070_x_expanded_axes`: a tuple of zero-based aligned axes
- `ex070_threshold_aligned_shape`: a literal Python shape tuple
- `ex070_threshold_expanded_axes`: a tuple of zero-based aligned axes
- `ex070_compatible`: a Python `bool`
- `ex070_out_shape`: a literal Python shape tuple
- `ex070_out`: the requested PyTorch tensor

**Order:** Fill every shape/alignment prediction before writing tensor operations.

**Next concept:** Minimum with column limits


In [ ]:
# Supplied visible fixture: inspect these values, but predict shapes without printing them.
x = torch.arange(6, dtype=DTYPE).reshape(2, 3)
threshold = torch.tensor([1.5, 2.5, 3.5], dtype=DTYPE)
_register_case("ex070", {"x": x, "threshold": threshold})


In [ ]:
# Exercise 070: predict shapes first; do not use autograd.
# Define `ex070_x_shape`.
# Define `ex070_threshold_shape`.
# Define `ex070_x_aligned_shape`.
# Define `ex070_x_expanded_axes`.
# Define `ex070_threshold_aligned_shape`.
# Define `ex070_threshold_expanded_axes`.
# Define `ex070_compatible`.
# Define `ex070_out_shape`.
# Define `ex070_out` with PyTorch tensor operations.
# Write your work below, then run the supplied test cell.


In [ ]:
# Supplied test: expected shapes and values remain private.
_check_private_value("ex070_x_shape", "ex070", "x_shape")
_check_private_value("ex070_threshold_shape", "ex070", "threshold_shape")
_check_private_value("ex070_x_aligned_shape", "ex070", "x_aligned_shape")
_check_private_value("ex070_x_expanded_axes", "ex070", "x_expanded_axes")
_check_private_value("ex070_threshold_aligned_shape", "ex070", "threshold_aligned_shape")
_check_private_value("ex070_threshold_expanded_axes", "ex070", "threshold_expanded_axes")
_check_private_value("ex070_compatible", "ex070", "compatible")
_check_private_value("ex070_out_shape", "ex070", "out_shape")
_check_private_tensor("ex070_out", "ex070", "out")


### Exercise 071 — Minimum with column limits

**Purpose:** Broadcast one limit per row.

**Visible inputs:** `x`, `limit`. Run the fixture below and read its construction code; it deliberately prints no shape answers.

**Operation to reason about:** `torch.minimum(x, limit)`

**Task:** Cap every row with its own limit.

**Ingredients:** The singleton column axis expands.

**Required predictions and outputs:**

- `ex071_x_shape`: a literal Python shape tuple
- `ex071_limit_shape`: a literal Python shape tuple
- `ex071_x_aligned_shape`: a literal Python shape tuple
- `ex071_x_expanded_axes`: a tuple of zero-based aligned axes
- `ex071_limit_aligned_shape`: a literal Python shape tuple
- `ex071_limit_expanded_axes`: a tuple of zero-based aligned axes
- `ex071_compatible`: a Python `bool`
- `ex071_out_shape`: a literal Python shape tuple
- `ex071_out`: the requested PyTorch tensor

**Order:** Fill every shape/alignment prediction before writing tensor operations.

**Next concept:** Boolean arithmetic mask


In [ ]:
# Supplied visible fixture: inspect these values, but predict shapes without printing them.
x = torch.arange(6, dtype=DTYPE).reshape(2, 3)
limit = torch.tensor([[1.0], [4.0]], dtype=DTYPE)
_register_case("ex071", {"x": x, "limit": limit})


In [ ]:
# Exercise 071: predict shapes first; do not use autograd.
# Define `ex071_x_shape`.
# Define `ex071_limit_shape`.
# Define `ex071_x_aligned_shape`.
# Define `ex071_x_expanded_axes`.
# Define `ex071_limit_aligned_shape`.
# Define `ex071_limit_expanded_axes`.
# Define `ex071_compatible`.
# Define `ex071_out_shape`.
# Define `ex071_out` with PyTorch tensor operations.
# Write your work below, then run the supplied test cell.


In [ ]:
# Supplied test: expected shapes and values remain private.
_check_private_value("ex071_x_shape", "ex071", "x_shape")
_check_private_value("ex071_limit_shape", "ex071", "limit_shape")
_check_private_value("ex071_x_aligned_shape", "ex071", "x_aligned_shape")
_check_private_value("ex071_x_expanded_axes", "ex071", "x_expanded_axes")
_check_private_value("ex071_limit_aligned_shape", "ex071", "limit_aligned_shape")
_check_private_value("ex071_limit_expanded_axes", "ex071", "limit_expanded_axes")
_check_private_value("ex071_compatible", "ex071", "compatible")
_check_private_value("ex071_out_shape", "ex071", "out_shape")
_check_private_tensor("ex071_out", "ex071", "out")


### Exercise 072 — Boolean arithmetic mask

**Purpose:** Observe that elementwise multiplication also broadcasts Boolean-derived values.

**Visible inputs:** `mask`, `x`. Run the fixture below and read its construction code; it deliberately prints no shape answers.

**Operation to reason about:** `mask * x`

**Task:** Keep one row and zero the other by broadcasting.

**Ingredients:** The operation is elementwise after shape expansion.

**Required predictions and outputs:**

- `ex072_mask_shape`: a literal Python shape tuple
- `ex072_x_shape`: a literal Python shape tuple
- `ex072_mask_aligned_shape`: a literal Python shape tuple
- `ex072_mask_expanded_axes`: a tuple of zero-based aligned axes
- `ex072_x_aligned_shape`: a literal Python shape tuple
- `ex072_x_expanded_axes`: a tuple of zero-based aligned axes
- `ex072_compatible`: a Python `bool`
- `ex072_out_shape`: a literal Python shape tuple
- `ex072_out`: the requested PyTorch tensor

**Order:** Fill every shape/alignment prediction before writing tensor operations.

**Next concept:** Add three operands


In [ ]:
# Supplied visible fixture: inspect these values, but predict shapes without printing them.
mask = torch.tensor([[1.0], [0.0]], dtype=DTYPE)
x = torch.arange(6, dtype=DTYPE).reshape(2, 3)
_register_case("ex072", {"mask": mask, "x": x})


In [ ]:
# Exercise 072: predict shapes first; do not use autograd.
# Define `ex072_mask_shape`.
# Define `ex072_x_shape`.
# Define `ex072_mask_aligned_shape`.
# Define `ex072_mask_expanded_axes`.
# Define `ex072_x_aligned_shape`.
# Define `ex072_x_expanded_axes`.
# Define `ex072_compatible`.
# Define `ex072_out_shape`.
# Define `ex072_out` with PyTorch tensor operations.
# Write your work below, then run the supplied test cell.


In [ ]:
# Supplied test: expected shapes and values remain private.
_check_private_value("ex072_mask_shape", "ex072", "mask_shape")
_check_private_value("ex072_x_shape", "ex072", "x_shape")
_check_private_value("ex072_mask_aligned_shape", "ex072", "mask_aligned_shape")
_check_private_value("ex072_mask_expanded_axes", "ex072", "mask_expanded_axes")
_check_private_value("ex072_x_aligned_shape", "ex072", "x_aligned_shape")
_check_private_value("ex072_x_expanded_axes", "ex072", "x_expanded_axes")
_check_private_value("ex072_compatible", "ex072", "compatible")
_check_private_value("ex072_out_shape", "ex072", "out_shape")
_check_private_tensor("ex072_out", "ex072", "out")


### Exercise 073 — Add three operands

**Purpose:** Find one result shape shared by a column, a row, and a scalar.

**Visible inputs:** `a`, `b`, `c`. Run the fixture below and read its construction code; it deliberately prints no shape answers.

**Operation to reason about:** `a + b + c`

**Task:** Compute the three-way broadcasted sum.

**Ingredients:** Audit each operand independently against the final output.

**Required predictions and outputs:**

- `ex073_a_shape`: a literal Python shape tuple
- `ex073_b_shape`: a literal Python shape tuple
- `ex073_c_shape`: a literal Python shape tuple
- `ex073_a_aligned_shape`: a literal Python shape tuple
- `ex073_a_expanded_axes`: a tuple of zero-based aligned axes
- `ex073_b_aligned_shape`: a literal Python shape tuple
- `ex073_b_expanded_axes`: a tuple of zero-based aligned axes
- `ex073_c_aligned_shape`: a literal Python shape tuple
- `ex073_c_expanded_axes`: a tuple of zero-based aligned axes
- `ex073_compatible`: a Python `bool`
- `ex073_out_shape`: a literal Python shape tuple
- `ex073_out`: the requested PyTorch tensor

**Order:** Fill every shape/alignment prediction before writing tensor operations.

**Next concept:** Broadcasted weighted blend


In [ ]:
# Supplied visible fixture: inspect these values, but predict shapes without printing them.
a = torch.tensor([[1.0], [2.0]], dtype=DTYPE)
b = torch.tensor([[10.0, 20.0, 30.0]], dtype=DTYPE)
c = torch.tensor(0.5, dtype=DTYPE)
_register_case("ex073", {"a": a, "b": b, "c": c})


In [ ]:
# Exercise 073: predict shapes first; do not use autograd.
# Define `ex073_a_shape`.
# Define `ex073_b_shape`.
# Define `ex073_c_shape`.
# Define `ex073_a_aligned_shape`.
# Define `ex073_a_expanded_axes`.
# Define `ex073_b_aligned_shape`.
# Define `ex073_b_expanded_axes`.
# Define `ex073_c_aligned_shape`.
# Define `ex073_c_expanded_axes`.
# Define `ex073_compatible`.
# Define `ex073_out_shape`.
# Define `ex073_out` with PyTorch tensor operations.
# Write your work below, then run the supplied test cell.


In [ ]:
# Supplied test: expected shapes and values remain private.
_check_private_value("ex073_a_shape", "ex073", "a_shape")
_check_private_value("ex073_b_shape", "ex073", "b_shape")
_check_private_value("ex073_c_shape", "ex073", "c_shape")
_check_private_value("ex073_a_aligned_shape", "ex073", "a_aligned_shape")
_check_private_value("ex073_a_expanded_axes", "ex073", "a_expanded_axes")
_check_private_value("ex073_b_aligned_shape", "ex073", "b_aligned_shape")
_check_private_value("ex073_b_expanded_axes", "ex073", "b_expanded_axes")
_check_private_value("ex073_c_aligned_shape", "ex073", "c_aligned_shape")
_check_private_value("ex073_c_expanded_axes", "ex073", "c_expanded_axes")
_check_private_value("ex073_compatible", "ex073", "compatible")
_check_private_value("ex073_out_shape", "ex073", "out_shape")
_check_private_tensor("ex073_out", "ex073", "out")


### Exercise 074 — Broadcasted weighted blend

**Purpose:** Use one interpolation weight per row.

**Visible inputs:** `alpha`, `x`, `y`. Run the fixture below and read its construction code; it deliberately prints no shape answers.

**Operation to reason about:** `alpha * x + (1 - alpha) * y`

**Task:** Blend each row with its own weight.

**Ingredients:** Both `alpha` and `1 - alpha` retain shape `(rows, 1)`.

**Required predictions and outputs:**

- `ex074_alpha_shape`: a literal Python shape tuple
- `ex074_x_shape`: a literal Python shape tuple
- `ex074_y_shape`: a literal Python shape tuple
- `ex074_alpha_aligned_shape`: a literal Python shape tuple
- `ex074_alpha_expanded_axes`: a tuple of zero-based aligned axes
- `ex074_x_aligned_shape`: a literal Python shape tuple
- `ex074_x_expanded_axes`: a tuple of zero-based aligned axes
- `ex074_y_aligned_shape`: a literal Python shape tuple
- `ex074_y_expanded_axes`: a tuple of zero-based aligned axes
- `ex074_compatible`: a Python `bool`
- `ex074_out_shape`: a literal Python shape tuple
- `ex074_out`: the requested PyTorch tensor

**Order:** Fill every shape/alignment prediction before writing tensor operations.

**Next concept:** torch.lerp row weights


In [ ]:
# Supplied visible fixture: inspect these values, but predict shapes without printing them.
alpha = torch.tensor([[0.25], [0.75]], dtype=DTYPE)
x = torch.zeros((2, 3), dtype=DTYPE)
y = torch.arange(6, dtype=DTYPE).reshape(2, 3)
_register_case("ex074", {"alpha": alpha, "x": x, "y": y})


In [ ]:
# Exercise 074: predict shapes first; do not use autograd.
# Define `ex074_alpha_shape`.
# Define `ex074_x_shape`.
# Define `ex074_y_shape`.
# Define `ex074_alpha_aligned_shape`.
# Define `ex074_alpha_expanded_axes`.
# Define `ex074_x_aligned_shape`.
# Define `ex074_x_expanded_axes`.
# Define `ex074_y_aligned_shape`.
# Define `ex074_y_expanded_axes`.
# Define `ex074_compatible`.
# Define `ex074_out_shape`.
# Define `ex074_out` with PyTorch tensor operations.
# Write your work below, then run the supplied test cell.


In [ ]:
# Supplied test: expected shapes and values remain private.
_check_private_value("ex074_alpha_shape", "ex074", "alpha_shape")
_check_private_value("ex074_x_shape", "ex074", "x_shape")
_check_private_value("ex074_y_shape", "ex074", "y_shape")
_check_private_value("ex074_alpha_aligned_shape", "ex074", "alpha_aligned_shape")
_check_private_value("ex074_alpha_expanded_axes", "ex074", "alpha_expanded_axes")
_check_private_value("ex074_x_aligned_shape", "ex074", "x_aligned_shape")
_check_private_value("ex074_x_expanded_axes", "ex074", "x_expanded_axes")
_check_private_value("ex074_y_aligned_shape", "ex074", "y_aligned_shape")
_check_private_value("ex074_y_expanded_axes", "ex074", "y_expanded_axes")
_check_private_value("ex074_compatible", "ex074", "compatible")
_check_private_value("ex074_out_shape", "ex074", "out_shape")
_check_private_tensor("ex074_out", "ex074", "out")


### Exercise 075 — torch.lerp row weights

**Purpose:** Recognize broadcasting inside a higher-level elementwise function.

**Visible inputs:** `start`, `end`, `weight`. Run the fixture below and read its construction code; it deliberately prints no shape answers.

**Operation to reason about:** `torch.lerp(start, end, weight)`

**Task:** Interpolate each row with its own weight.

**Ingredients:** `torch.lerp` broadcasts `start`, `end`, and `weight`.

**Required predictions and outputs:**

- `ex075_start_shape`: a literal Python shape tuple
- `ex075_end_shape`: a literal Python shape tuple
- `ex075_weight_shape`: a literal Python shape tuple
- `ex075_start_aligned_shape`: a literal Python shape tuple
- `ex075_start_expanded_axes`: a tuple of zero-based aligned axes
- `ex075_end_aligned_shape`: a literal Python shape tuple
- `ex075_end_expanded_axes`: a tuple of zero-based aligned axes
- `ex075_weight_aligned_shape`: a literal Python shape tuple
- `ex075_weight_expanded_axes`: a tuple of zero-based aligned axes
- `ex075_compatible`: a Python `bool`
- `ex075_out_shape`: a literal Python shape tuple
- `ex075_out`: the requested PyTorch tensor

**Order:** Fill every shape/alignment prediction before writing tensor operations.

**Next concept:** Batch feature bias


In [ ]:
# Supplied visible fixture: inspect these values, but predict shapes without printing them.
start = torch.zeros((2, 3), dtype=DTYPE)
end = torch.arange(6, dtype=DTYPE).reshape(2, 3)
weight = torch.tensor([[0.25], [0.75]], dtype=DTYPE)
_register_case("ex075", {"start": start, "end": end, "weight": weight})


In [ ]:
# Exercise 075: predict shapes first; do not use autograd.
# Define `ex075_start_shape`.
# Define `ex075_end_shape`.
# Define `ex075_weight_shape`.
# Define `ex075_start_aligned_shape`.
# Define `ex075_start_expanded_axes`.
# Define `ex075_end_aligned_shape`.
# Define `ex075_end_expanded_axes`.
# Define `ex075_weight_aligned_shape`.
# Define `ex075_weight_expanded_axes`.
# Define `ex075_compatible`.
# Define `ex075_out_shape`.
# Define `ex075_out` with PyTorch tensor operations.
# Write your work below, then run the supplied test cell.


In [ ]:
# Supplied test: expected shapes and values remain private.
_check_private_value("ex075_start_shape", "ex075", "start_shape")
_check_private_value("ex075_end_shape", "ex075", "end_shape")
_check_private_value("ex075_weight_shape", "ex075", "weight_shape")
_check_private_value("ex075_start_aligned_shape", "ex075", "start_aligned_shape")
_check_private_value("ex075_start_expanded_axes", "ex075", "start_expanded_axes")
_check_private_value("ex075_end_aligned_shape", "ex075", "end_aligned_shape")
_check_private_value("ex075_end_expanded_axes", "ex075", "end_expanded_axes")
_check_private_value("ex075_weight_aligned_shape", "ex075", "weight_aligned_shape")
_check_private_value("ex075_weight_expanded_axes", "ex075", "weight_expanded_axes")
_check_private_value("ex075_compatible", "ex075", "compatible")
_check_private_value("ex075_out_shape", "ex075", "out_shape")
_check_private_tensor("ex075_out", "ex075", "out")


## 8. Machine-learning broadcasting patterns

Practice the axis layouts used by biases, normalization parameters, masks, attention, image channels, and mixture weights.


### Exercise 076 — Batch feature bias

**Purpose:** Recognize the standard linear-layer output bias pattern.

**Visible inputs:** `activations`, `bias`. Run the fixture below and read its construction code; it deliberately prints no shape answers.

**Operation to reason about:** `activations + bias`

**Task:** Add one bias per feature to every training example.

**Ingredients:** Axes are `(examples, features)` and `(features,)`.

**Required predictions and outputs:**

- `ex076_activations_shape`: a literal Python shape tuple
- `ex076_bias_shape`: a literal Python shape tuple
- `ex076_activations_aligned_shape`: a literal Python shape tuple
- `ex076_activations_expanded_axes`: a tuple of zero-based aligned axes
- `ex076_bias_aligned_shape`: a literal Python shape tuple
- `ex076_bias_expanded_axes`: a tuple of zero-based aligned axes
- `ex076_compatible`: a Python `bool`
- `ex076_out_shape`: a literal Python shape tuple
- `ex076_out`: the requested PyTorch tensor

**Order:** Fill every shape/alignment prediction before writing tensor operations.

**Next concept:** Per-example scalar offset


In [ ]:
# Supplied visible fixture: inspect these values, but predict shapes without printing them.
activations = torch.arange(20, dtype=DTYPE).reshape(4, 5)
bias = torch.linspace(-1.0, 1.0, steps=5, dtype=DTYPE)
_register_case("ex076", {"activations": activations, "bias": bias})


In [ ]:
# Exercise 076: predict shapes first; do not use autograd.
# Define `ex076_activations_shape`.
# Define `ex076_bias_shape`.
# Define `ex076_activations_aligned_shape`.
# Define `ex076_activations_expanded_axes`.
# Define `ex076_bias_aligned_shape`.
# Define `ex076_bias_expanded_axes`.
# Define `ex076_compatible`.
# Define `ex076_out_shape`.
# Define `ex076_out` with PyTorch tensor operations.
# Write your work below, then run the supplied test cell.


In [ ]:
# Supplied test: expected shapes and values remain private.
_check_private_value("ex076_activations_shape", "ex076", "activations_shape")
_check_private_value("ex076_bias_shape", "ex076", "bias_shape")
_check_private_value("ex076_activations_aligned_shape", "ex076", "activations_aligned_shape")
_check_private_value("ex076_activations_expanded_axes", "ex076", "activations_expanded_axes")
_check_private_value("ex076_bias_aligned_shape", "ex076", "bias_aligned_shape")
_check_private_value("ex076_bias_expanded_axes", "ex076", "bias_expanded_axes")
_check_private_value("ex076_compatible", "ex076", "compatible")
_check_private_value("ex076_out_shape", "ex076", "out_shape")
_check_private_tensor("ex076_out", "ex076", "out")


### Exercise 077 — Per-example scalar offset

**Purpose:** Apply one scalar to every feature within each example.

**Visible inputs:** `activations`, `offset`. Run the fixture below and read its construction code; it deliberately prints no shape answers.

**Operation to reason about:** `activations + offset`

**Task:** Add each example's offset across its features.

**Ingredients:** Use `(examples, 1)` to preserve row ownership.

**Required predictions and outputs:**

- `ex077_activations_shape`: a literal Python shape tuple
- `ex077_offset_shape`: a literal Python shape tuple
- `ex077_activations_aligned_shape`: a literal Python shape tuple
- `ex077_activations_expanded_axes`: a tuple of zero-based aligned axes
- `ex077_offset_aligned_shape`: a literal Python shape tuple
- `ex077_offset_expanded_axes`: a tuple of zero-based aligned axes
- `ex077_compatible`: a Python `bool`
- `ex077_out_shape`: a literal Python shape tuple
- `ex077_out`: the requested PyTorch tensor

**Order:** Fill every shape/alignment prediction before writing tensor operations.

**Next concept:** Feature-wise gain


In [ ]:
# Supplied visible fixture: inspect these values, but predict shapes without printing them.
activations = torch.arange(20, dtype=DTYPE).reshape(4, 5)
offset = torch.tensor([[0.1], [0.2], [0.3], [0.4]], dtype=DTYPE)
_register_case("ex077", {"activations": activations, "offset": offset})


In [ ]:
# Exercise 077: predict shapes first; do not use autograd.
# Define `ex077_activations_shape`.
# Define `ex077_offset_shape`.
# Define `ex077_activations_aligned_shape`.
# Define `ex077_activations_expanded_axes`.
# Define `ex077_offset_aligned_shape`.
# Define `ex077_offset_expanded_axes`.
# Define `ex077_compatible`.
# Define `ex077_out_shape`.
# Define `ex077_out` with PyTorch tensor operations.
# Write your work below, then run the supplied test cell.


In [ ]:
# Supplied test: expected shapes and values remain private.
_check_private_value("ex077_activations_shape", "ex077", "activations_shape")
_check_private_value("ex077_offset_shape", "ex077", "offset_shape")
_check_private_value("ex077_activations_aligned_shape", "ex077", "activations_aligned_shape")
_check_private_value("ex077_activations_expanded_axes", "ex077", "activations_expanded_axes")
_check_private_value("ex077_offset_aligned_shape", "ex077", "offset_aligned_shape")
_check_private_value("ex077_offset_expanded_axes", "ex077", "offset_expanded_axes")
_check_private_value("ex077_compatible", "ex077", "compatible")
_check_private_value("ex077_out_shape", "ex077", "out_shape")
_check_private_tensor("ex077_out", "ex077", "out")


### Exercise 078 — Feature-wise gain

**Purpose:** Recognize the learned scale pattern used by normalization layers.

**Visible inputs:** `normalized`, `gain`. Run the fixture below and read its construction code; it deliberately prints no shape answers.

**Operation to reason about:** `normalized * gain`

**Task:** Apply one gain per feature.

**Ingredients:** The final feature axis matches directly.

**Required predictions and outputs:**

- `ex078_normalized_shape`: a literal Python shape tuple
- `ex078_gain_shape`: a literal Python shape tuple
- `ex078_normalized_aligned_shape`: a literal Python shape tuple
- `ex078_normalized_expanded_axes`: a tuple of zero-based aligned axes
- `ex078_gain_aligned_shape`: a literal Python shape tuple
- `ex078_gain_expanded_axes`: a tuple of zero-based aligned axes
- `ex078_compatible`: a Python `bool`
- `ex078_out_shape`: a literal Python shape tuple
- `ex078_out`: the requested PyTorch tensor

**Order:** Fill every shape/alignment prediction before writing tensor operations.

**Next concept:** Column standardization


In [ ]:
# Supplied visible fixture: inspect these values, but predict shapes without printing them.
normalized = torch.linspace(-2.0, 2.0, steps=20, dtype=DTYPE).reshape(4, 5)
gain = torch.linspace(0.5, 1.5, steps=5, dtype=DTYPE)
_register_case("ex078", {"normalized": normalized, "gain": gain})


In [ ]:
# Exercise 078: predict shapes first; do not use autograd.
# Define `ex078_normalized_shape`.
# Define `ex078_gain_shape`.
# Define `ex078_normalized_aligned_shape`.
# Define `ex078_normalized_expanded_axes`.
# Define `ex078_gain_aligned_shape`.
# Define `ex078_gain_expanded_axes`.
# Define `ex078_compatible`.
# Define `ex078_out_shape`.
# Define `ex078_out` with PyTorch tensor operations.
# Write your work below, then run the supplied test cell.


In [ ]:
# Supplied test: expected shapes and values remain private.
_check_private_value("ex078_normalized_shape", "ex078", "normalized_shape")
_check_private_value("ex078_gain_shape", "ex078", "gain_shape")
_check_private_value("ex078_normalized_aligned_shape", "ex078", "normalized_aligned_shape")
_check_private_value("ex078_normalized_expanded_axes", "ex078", "normalized_expanded_axes")
_check_private_value("ex078_gain_aligned_shape", "ex078", "gain_aligned_shape")
_check_private_value("ex078_gain_expanded_axes", "ex078", "gain_expanded_axes")
_check_private_value("ex078_compatible", "ex078", "compatible")
_check_private_value("ex078_out_shape", "ex078", "out_shape")
_check_private_tensor("ex078_out", "ex078", "out")


### Exercise 079 — Column standardization

**Purpose:** Combine two broadcasted feature statistics.

**Visible inputs:** `x`, `mean`, `std`. Run the fixture below and read its construction code; it deliberately prints no shape answers.

**Operation to reason about:** `(x - mean) / std`

**Task:** Center and scale every feature column.

**Ingredients:** Both statistics reuse one value down the example axis.

**Required predictions and outputs:**

- `ex079_x_shape`: a literal Python shape tuple
- `ex079_mean_shape`: a literal Python shape tuple
- `ex079_std_shape`: a literal Python shape tuple
- `ex079_x_aligned_shape`: a literal Python shape tuple
- `ex079_x_expanded_axes`: a tuple of zero-based aligned axes
- `ex079_mean_aligned_shape`: a literal Python shape tuple
- `ex079_mean_expanded_axes`: a tuple of zero-based aligned axes
- `ex079_std_aligned_shape`: a literal Python shape tuple
- `ex079_std_expanded_axes`: a tuple of zero-based aligned axes
- `ex079_compatible`: a Python `bool`
- `ex079_out_shape`: a literal Python shape tuple
- `ex079_out`: the requested PyTorch tensor

**Order:** Fill every shape/alignment prediction before writing tensor operations.

**Next concept:** LayerNorm affine parameters


In [ ]:
# Supplied visible fixture: inspect these values, but predict shapes without printing them.
x = torch.arange(20, dtype=DTYPE).reshape(4, 5)
mean = torch.linspace(1.0, 5.0, steps=5, dtype=DTYPE)
std = torch.linspace(1.0, 2.0, steps=5, dtype=DTYPE)
_register_case("ex079", {"x": x, "mean": mean, "std": std})


In [ ]:
# Exercise 079: predict shapes first; do not use autograd.
# Define `ex079_x_shape`.
# Define `ex079_mean_shape`.
# Define `ex079_std_shape`.
# Define `ex079_x_aligned_shape`.
# Define `ex079_x_expanded_axes`.
# Define `ex079_mean_aligned_shape`.
# Define `ex079_mean_expanded_axes`.
# Define `ex079_std_aligned_shape`.
# Define `ex079_std_expanded_axes`.
# Define `ex079_compatible`.
# Define `ex079_out_shape`.
# Define `ex079_out` with PyTorch tensor operations.
# Write your work below, then run the supplied test cell.


In [ ]:
# Supplied test: expected shapes and values remain private.
_check_private_value("ex079_x_shape", "ex079", "x_shape")
_check_private_value("ex079_mean_shape", "ex079", "mean_shape")
_check_private_value("ex079_std_shape", "ex079", "std_shape")
_check_private_value("ex079_x_aligned_shape", "ex079", "x_aligned_shape")
_check_private_value("ex079_x_expanded_axes", "ex079", "x_expanded_axes")
_check_private_value("ex079_mean_aligned_shape", "ex079", "mean_aligned_shape")
_check_private_value("ex079_mean_expanded_axes", "ex079", "mean_expanded_axes")
_check_private_value("ex079_std_aligned_shape", "ex079", "std_aligned_shape")
_check_private_value("ex079_std_expanded_axes", "ex079", "std_expanded_axes")
_check_private_value("ex079_compatible", "ex079", "compatible")
_check_private_value("ex079_out_shape", "ex079", "out_shape")
_check_private_tensor("ex079_out", "ex079", "out")


### Exercise 080 — LayerNorm affine parameters

**Purpose:** Apply feature gain and bias across batch and time axes.

**Visible inputs:** `normalized`, `gamma`, `beta`. Run the fixture below and read its construction code; it deliberately prints no shape answers.

**Operation to reason about:** `normalized * gamma + beta`

**Task:** Apply the same affine parameters at every token position.

**Ingredients:** Feature parameters naturally align with the final axis.

**Required predictions and outputs:**

- `ex080_normalized_shape`: a literal Python shape tuple
- `ex080_gamma_shape`: a literal Python shape tuple
- `ex080_beta_shape`: a literal Python shape tuple
- `ex080_normalized_aligned_shape`: a literal Python shape tuple
- `ex080_normalized_expanded_axes`: a tuple of zero-based aligned axes
- `ex080_gamma_aligned_shape`: a literal Python shape tuple
- `ex080_gamma_expanded_axes`: a tuple of zero-based aligned axes
- `ex080_beta_aligned_shape`: a literal Python shape tuple
- `ex080_beta_expanded_axes`: a tuple of zero-based aligned axes
- `ex080_compatible`: a Python `bool`
- `ex080_out_shape`: a literal Python shape tuple
- `ex080_out`: the requested PyTorch tensor

**Order:** Fill every shape/alignment prediction before writing tensor operations.

**Next concept:** Positional embedding addition


In [ ]:
# Supplied visible fixture: inspect these values, but predict shapes without printing them.
normalized = torch.arange(24, dtype=DTYPE).reshape(2, 3, 4)
gamma = torch.tensor([0.5, 1.0, 1.5, 2.0], dtype=DTYPE)
beta = torch.tensor([-1.0, 0.0, 1.0, 2.0], dtype=DTYPE)
_register_case("ex080", {"normalized": normalized, "gamma": gamma, "beta": beta})


In [ ]:
# Exercise 080: predict shapes first; do not use autograd.
# Define `ex080_normalized_shape`.
# Define `ex080_gamma_shape`.
# Define `ex080_beta_shape`.
# Define `ex080_normalized_aligned_shape`.
# Define `ex080_normalized_expanded_axes`.
# Define `ex080_gamma_aligned_shape`.
# Define `ex080_gamma_expanded_axes`.
# Define `ex080_beta_aligned_shape`.
# Define `ex080_beta_expanded_axes`.
# Define `ex080_compatible`.
# Define `ex080_out_shape`.
# Define `ex080_out` with PyTorch tensor operations.
# Write your work below, then run the supplied test cell.


In [ ]:
# Supplied test: expected shapes and values remain private.
_check_private_value("ex080_normalized_shape", "ex080", "normalized_shape")
_check_private_value("ex080_gamma_shape", "ex080", "gamma_shape")
_check_private_value("ex080_beta_shape", "ex080", "beta_shape")
_check_private_value("ex080_normalized_aligned_shape", "ex080", "normalized_aligned_shape")
_check_private_value("ex080_normalized_expanded_axes", "ex080", "normalized_expanded_axes")
_check_private_value("ex080_gamma_aligned_shape", "ex080", "gamma_aligned_shape")
_check_private_value("ex080_gamma_expanded_axes", "ex080", "gamma_expanded_axes")
_check_private_value("ex080_beta_aligned_shape", "ex080", "beta_aligned_shape")
_check_private_value("ex080_beta_expanded_axes", "ex080", "beta_expanded_axes")
_check_private_value("ex080_compatible", "ex080", "compatible")
_check_private_value("ex080_out_shape", "ex080", "out_shape")
_check_private_tensor("ex080_out", "ex080", "out")


### Exercise 081 — Positional embedding addition

**Purpose:** Reuse one position-feature table across a batch.

**Visible inputs:** `token_embeddings`, `position_embeddings`. Run the fixture below and read its construction code; it deliberately prints no shape answers.

**Operation to reason about:** `token_embeddings + position_embeddings`

**Task:** Add positions to every sequence.

**Ingredients:** The missing batch dimension is conceptually one.

**Required predictions and outputs:**

- `ex081_token_embeddings_shape`: a literal Python shape tuple
- `ex081_position_embeddings_shape`: a literal Python shape tuple
- `ex081_token_embeddings_aligned_shape`: a literal Python shape tuple
- `ex081_token_embeddings_expanded_axes`: a tuple of zero-based aligned axes
- `ex081_position_embeddings_aligned_shape`: a literal Python shape tuple
- `ex081_position_embeddings_expanded_axes`: a tuple of zero-based aligned axes
- `ex081_compatible`: a Python `bool`
- `ex081_out_shape`: a literal Python shape tuple
- `ex081_out`: the requested PyTorch tensor

**Order:** Fill every shape/alignment prediction before writing tensor operations.

**Next concept:** Token validity mask


In [ ]:
# Supplied visible fixture: inspect these values, but predict shapes without printing them.
token_embeddings = torch.arange(24, dtype=DTYPE).reshape(2, 3, 4)
position_embeddings = torch.linspace(-0.5, 0.5, steps=12, dtype=DTYPE).reshape(3, 4)
_register_case("ex081", {"token_embeddings": token_embeddings, "position_embeddings": position_embeddings})


In [ ]:
# Exercise 081: predict shapes first; do not use autograd.
# Define `ex081_token_embeddings_shape`.
# Define `ex081_position_embeddings_shape`.
# Define `ex081_token_embeddings_aligned_shape`.
# Define `ex081_token_embeddings_expanded_axes`.
# Define `ex081_position_embeddings_aligned_shape`.
# Define `ex081_position_embeddings_expanded_axes`.
# Define `ex081_compatible`.
# Define `ex081_out_shape`.
# Define `ex081_out` with PyTorch tensor operations.
# Write your work below, then run the supplied test cell.


In [ ]:
# Supplied test: expected shapes and values remain private.
_check_private_value("ex081_token_embeddings_shape", "ex081", "token_embeddings_shape")
_check_private_value("ex081_position_embeddings_shape", "ex081", "position_embeddings_shape")
_check_private_value("ex081_token_embeddings_aligned_shape", "ex081", "token_embeddings_aligned_shape")
_check_private_value("ex081_token_embeddings_expanded_axes", "ex081", "token_embeddings_expanded_axes")
_check_private_value("ex081_position_embeddings_aligned_shape", "ex081", "position_embeddings_aligned_shape")
_check_private_value("ex081_position_embeddings_expanded_axes", "ex081", "position_embeddings_expanded_axes")
_check_private_value("ex081_compatible", "ex081", "compatible")
_check_private_value("ex081_out_shape", "ex081", "out_shape")
_check_private_tensor("ex081_out", "ex081", "out")


### Exercise 082 — Token validity mask

**Purpose:** Expand one validity bit over embedding features.

**Visible inputs:** `mask`, `embeddings`. Run the fixture below and read its construction code; it deliberately prints no shape answers.

**Operation to reason about:** Create `mask_view`, then zero all features at invalid token positions.

**Task:** Create `mask_view`, then zero all features at invalid token positions.

**Ingredients:** Insert a trailing singleton feature axis.

**Required predictions and outputs:**

- `ex082_mask_shape`: a literal Python shape tuple
- `ex082_embeddings_shape`: a literal Python shape tuple
- `ex082_mask_view_shape`: a literal Python shape tuple
- `ex082_mask_view_aligned_shape`: a literal Python shape tuple
- `ex082_mask_view_expanded_axes`: a tuple of zero-based aligned axes
- `ex082_embeddings_aligned_shape`: a literal Python shape tuple
- `ex082_embeddings_expanded_axes`: a tuple of zero-based aligned axes
- `ex082_compatible`: a Python `bool`
- `ex082_out_shape`: a literal Python shape tuple
- `ex082_mask_view`: the requested PyTorch tensor
- `ex082_out`: the requested PyTorch tensor

**Order:** Fill every shape/alignment prediction before writing tensor operations.

**Next concept:** Per-example sequence weights


In [ ]:
# Supplied visible fixture: inspect these values, but predict shapes without printing them.
mask = torch.tensor([[1.0, 1.0, 0.0], [1.0, 0.0, 0.0]], dtype=DTYPE)
embeddings = torch.arange(24, dtype=DTYPE).reshape(2, 3, 4)
_register_case("ex082", {"mask": mask, "embeddings": embeddings})


In [ ]:
# Exercise 082: predict shapes first; do not use autograd.
# Define `ex082_mask_shape`.
# Define `ex082_embeddings_shape`.
# Define `ex082_mask_view_shape`.
# Define `ex082_mask_view_aligned_shape`.
# Define `ex082_mask_view_expanded_axes`.
# Define `ex082_embeddings_aligned_shape`.
# Define `ex082_embeddings_expanded_axes`.
# Define `ex082_compatible`.
# Define `ex082_out_shape`.
# Define `ex082_mask_view` with PyTorch tensor operations.
# Define `ex082_out` with PyTorch tensor operations.
# Write your work below, then run the supplied test cell.


In [ ]:
# Supplied test: expected shapes and values remain private.
_check_private_value("ex082_mask_shape", "ex082", "mask_shape")
_check_private_value("ex082_embeddings_shape", "ex082", "embeddings_shape")
_check_private_value("ex082_mask_view_shape", "ex082", "mask_view_shape")
_check_private_value("ex082_mask_view_aligned_shape", "ex082", "mask_view_aligned_shape")
_check_private_value("ex082_mask_view_expanded_axes", "ex082", "mask_view_expanded_axes")
_check_private_value("ex082_embeddings_aligned_shape", "ex082", "embeddings_aligned_shape")
_check_private_value("ex082_embeddings_expanded_axes", "ex082", "embeddings_expanded_axes")
_check_private_value("ex082_compatible", "ex082", "compatible")
_check_private_value("ex082_out_shape", "ex082", "out_shape")
_check_private_tensor("ex082_mask_view", "ex082", "mask_view")
_check_private_tensor("ex082_out", "ex082", "out")


### Exercise 083 — Per-example sequence weights

**Purpose:** Align one example weight with every token loss.

**Visible inputs:** `weights`, `token_losses`. Run the fixture below and read its construction code; it deliberately prints no shape answers.

**Operation to reason about:** Create `weights_view`, then weight each example's token losses.

**Task:** Create `weights_view`, then weight each example's token losses.

**Ingredients:** Example weights need a singleton token axis.

**Required predictions and outputs:**

- `ex083_weights_shape`: a literal Python shape tuple
- `ex083_token_losses_shape`: a literal Python shape tuple
- `ex083_weights_view_shape`: a literal Python shape tuple
- `ex083_weights_view_aligned_shape`: a literal Python shape tuple
- `ex083_weights_view_expanded_axes`: a tuple of zero-based aligned axes
- `ex083_token_losses_aligned_shape`: a literal Python shape tuple
- `ex083_token_losses_expanded_axes`: a tuple of zero-based aligned axes
- `ex083_compatible`: a Python `bool`
- `ex083_out_shape`: a literal Python shape tuple
- `ex083_weights_view`: the requested PyTorch tensor
- `ex083_out`: the requested PyTorch tensor

**Order:** Fill every shape/alignment prediction before writing tensor operations.

**Next concept:** Class weighting


In [ ]:
# Supplied visible fixture: inspect these values, but predict shapes without printing them.
weights = torch.tensor([1.0, 0.5, 2.0], dtype=DTYPE)
token_losses = torch.arange(12, dtype=DTYPE).reshape(3, 4)
_register_case("ex083", {"weights": weights, "token_losses": token_losses})


In [ ]:
# Exercise 083: predict shapes first; do not use autograd.
# Define `ex083_weights_shape`.
# Define `ex083_token_losses_shape`.
# Define `ex083_weights_view_shape`.
# Define `ex083_weights_view_aligned_shape`.
# Define `ex083_weights_view_expanded_axes`.
# Define `ex083_token_losses_aligned_shape`.
# Define `ex083_token_losses_expanded_axes`.
# Define `ex083_compatible`.
# Define `ex083_out_shape`.
# Define `ex083_weights_view` with PyTorch tensor operations.
# Define `ex083_out` with PyTorch tensor operations.
# Write your work below, then run the supplied test cell.


In [ ]:
# Supplied test: expected shapes and values remain private.
_check_private_value("ex083_weights_shape", "ex083", "weights_shape")
_check_private_value("ex083_token_losses_shape", "ex083", "token_losses_shape")
_check_private_value("ex083_weights_view_shape", "ex083", "weights_view_shape")
_check_private_value("ex083_weights_view_aligned_shape", "ex083", "weights_view_aligned_shape")
_check_private_value("ex083_weights_view_expanded_axes", "ex083", "weights_view_expanded_axes")
_check_private_value("ex083_token_losses_aligned_shape", "ex083", "token_losses_aligned_shape")
_check_private_value("ex083_token_losses_expanded_axes", "ex083", "token_losses_expanded_axes")
_check_private_value("ex083_compatible", "ex083", "compatible")
_check_private_value("ex083_out_shape", "ex083", "out_shape")
_check_private_tensor("ex083_weights_view", "ex083", "weights_view")
_check_private_tensor("ex083_out", "ex083", "out")


### Exercise 084 — Class weighting

**Purpose:** Apply one class weight to every example's candidate loss.

**Visible inputs:** `losses`, `class_weights`. Run the fixture below and read its construction code; it deliberately prints no shape answers.

**Operation to reason about:** `losses * class_weights`

**Task:** Weight each candidate column.

**Ingredients:** Classes occupy the final axis.

**Required predictions and outputs:**

- `ex084_losses_shape`: a literal Python shape tuple
- `ex084_class_weights_shape`: a literal Python shape tuple
- `ex084_losses_aligned_shape`: a literal Python shape tuple
- `ex084_losses_expanded_axes`: a tuple of zero-based aligned axes
- `ex084_class_weights_aligned_shape`: a literal Python shape tuple
- `ex084_class_weights_expanded_axes`: a tuple of zero-based aligned axes
- `ex084_compatible`: a Python `bool`
- `ex084_out_shape`: a literal Python shape tuple
- `ex084_out`: the requested PyTorch tensor

**Order:** Fill every shape/alignment prediction before writing tensor operations.

**Next concept:** Image channel bias with preparation


In [ ]:
# Supplied visible fixture: inspect these values, but predict shapes without printing them.
losses = torch.arange(20, dtype=DTYPE).reshape(4, 5)
class_weights = torch.tensor([1.0, 0.5, 2.0, 1.5, 3.0], dtype=DTYPE)
_register_case("ex084", {"losses": losses, "class_weights": class_weights})


In [ ]:
# Exercise 084: predict shapes first; do not use autograd.
# Define `ex084_losses_shape`.
# Define `ex084_class_weights_shape`.
# Define `ex084_losses_aligned_shape`.
# Define `ex084_losses_expanded_axes`.
# Define `ex084_class_weights_aligned_shape`.
# Define `ex084_class_weights_expanded_axes`.
# Define `ex084_compatible`.
# Define `ex084_out_shape`.
# Define `ex084_out` with PyTorch tensor operations.
# Write your work below, then run the supplied test cell.


In [ ]:
# Supplied test: expected shapes and values remain private.
_check_private_value("ex084_losses_shape", "ex084", "losses_shape")
_check_private_value("ex084_class_weights_shape", "ex084", "class_weights_shape")
_check_private_value("ex084_losses_aligned_shape", "ex084", "losses_aligned_shape")
_check_private_value("ex084_losses_expanded_axes", "ex084", "losses_expanded_axes")
_check_private_value("ex084_class_weights_aligned_shape", "ex084", "class_weights_aligned_shape")
_check_private_value("ex084_class_weights_expanded_axes", "ex084", "class_weights_expanded_axes")
_check_private_value("ex084_compatible", "ex084", "compatible")
_check_private_value("ex084_out_shape", "ex084", "out_shape")
_check_private_tensor("ex084_out", "ex084", "out")


### Exercise 085 — Image channel bias with preparation

**Purpose:** Convert a compact channel vector into image-axis form.

**Visible inputs:** `images`, `bias`. Run the fixture below and read its construction code; it deliberately prints no shape answers.

**Operation to reason about:** Create `bias_view`, then add one bias per channel.

**Task:** Create `bias_view`, then add one bias per channel.

**Ingredients:** Prepare `(channel,)` as `(1, channel, 1, 1)`.

**Required predictions and outputs:**

- `ex085_images_shape`: a literal Python shape tuple
- `ex085_bias_shape`: a literal Python shape tuple
- `ex085_bias_view_shape`: a literal Python shape tuple
- `ex085_bias_view_aligned_shape`: a literal Python shape tuple
- `ex085_bias_view_expanded_axes`: a tuple of zero-based aligned axes
- `ex085_images_aligned_shape`: a literal Python shape tuple
- `ex085_images_expanded_axes`: a tuple of zero-based aligned axes
- `ex085_compatible`: a Python `bool`
- `ex085_out_shape`: a literal Python shape tuple
- `ex085_bias_view`: the requested PyTorch tensor
- `ex085_out`: the requested PyTorch tensor

**Order:** Fill every shape/alignment prediction before writing tensor operations.

**Next concept:** Image channel gain with preparation


In [ ]:
# Supplied visible fixture: inspect these values, but predict shapes without printing them.
images = torch.arange(120, dtype=DTYPE).reshape(2, 3, 4, 5)
bias = torch.tensor([0.1, 0.2, 0.3], dtype=DTYPE)
_register_case("ex085", {"images": images, "bias": bias})


In [ ]:
# Exercise 085: predict shapes first; do not use autograd.
# Define `ex085_images_shape`.
# Define `ex085_bias_shape`.
# Define `ex085_bias_view_shape`.
# Define `ex085_bias_view_aligned_shape`.
# Define `ex085_bias_view_expanded_axes`.
# Define `ex085_images_aligned_shape`.
# Define `ex085_images_expanded_axes`.
# Define `ex085_compatible`.
# Define `ex085_out_shape`.
# Define `ex085_bias_view` with PyTorch tensor operations.
# Define `ex085_out` with PyTorch tensor operations.
# Write your work below, then run the supplied test cell.


In [ ]:
# Supplied test: expected shapes and values remain private.
_check_private_value("ex085_images_shape", "ex085", "images_shape")
_check_private_value("ex085_bias_shape", "ex085", "bias_shape")
_check_private_value("ex085_bias_view_shape", "ex085", "bias_view_shape")
_check_private_value("ex085_bias_view_aligned_shape", "ex085", "bias_view_aligned_shape")
_check_private_value("ex085_bias_view_expanded_axes", "ex085", "bias_view_expanded_axes")
_check_private_value("ex085_images_aligned_shape", "ex085", "images_aligned_shape")
_check_private_value("ex085_images_expanded_axes", "ex085", "images_expanded_axes")
_check_private_value("ex085_compatible", "ex085", "compatible")
_check_private_value("ex085_out_shape", "ex085", "out_shape")
_check_private_tensor("ex085_bias_view", "ex085", "bias_view")
_check_private_tensor("ex085_out", "ex085", "out")


### Exercise 086 — Image channel gain with preparation

**Purpose:** Apply a compact gain vector over batch and spatial axes.

**Visible inputs:** `images`, `gain`. Run the fixture below and read its construction code; it deliberately prints no shape answers.

**Operation to reason about:** Create `gain_view`, then scale every channel.

**Task:** Create `gain_view`, then scale every channel.

**Ingredients:** Use the same axis layout as channel bias.

**Required predictions and outputs:**

- `ex086_images_shape`: a literal Python shape tuple
- `ex086_gain_shape`: a literal Python shape tuple
- `ex086_gain_view_shape`: a literal Python shape tuple
- `ex086_gain_view_aligned_shape`: a literal Python shape tuple
- `ex086_gain_view_expanded_axes`: a tuple of zero-based aligned axes
- `ex086_images_aligned_shape`: a literal Python shape tuple
- `ex086_images_expanded_axes`: a tuple of zero-based aligned axes
- `ex086_compatible`: a Python `bool`
- `ex086_out_shape`: a literal Python shape tuple
- `ex086_gain_view`: the requested PyTorch tensor
- `ex086_out`: the requested PyTorch tensor

**Order:** Fill every shape/alignment prediction before writing tensor operations.

**Next concept:** Attention mask


In [ ]:
# Supplied visible fixture: inspect these values, but predict shapes without printing them.
images = torch.ones((2, 3, 4, 5), dtype=DTYPE)
gain = torch.tensor([0.5, 1.0, 1.5], dtype=DTYPE)
_register_case("ex086", {"images": images, "gain": gain})


In [22]:
# Exercise 086: predict shapes first; do not use autograd.
# Define `ex086_images_shape`.
# Define `ex086_gain_shape`.
# Define `ex086_gain_view_shape`.
# Define `ex086_gain_view_aligned_shape`.
# Define `ex086_gain_view_expanded_axes`.
# Define `ex086_images_aligned_shape`.
# Define `ex086_images_expanded_axes`.
# Define `ex086_compatible`.
# Define `ex086_out_shape`.
# Define `ex086_gain_view` with PyTorch tensor operations.
# Define `ex086_out` with PyTorch tensor operations.
# Write your work below, then run the supplied test cell.


In [23]:
# Supplied test: expected shapes and values remain private.
_check_private_value("ex086_images_shape", "ex086", "images_shape")
_check_private_value("ex086_gain_shape", "ex086", "gain_shape")
_check_private_value("ex086_gain_view_shape", "ex086", "gain_view_shape")
_check_private_value("ex086_gain_view_aligned_shape", "ex086", "gain_view_aligned_shape")
_check_private_value("ex086_gain_view_expanded_axes", "ex086", "gain_view_expanded_axes")
_check_private_value("ex086_images_aligned_shape", "ex086", "images_aligned_shape")
_check_private_value("ex086_images_expanded_axes", "ex086", "images_expanded_axes")
_check_private_value("ex086_compatible", "ex086", "compatible")
_check_private_value("ex086_out_shape", "ex086", "out_shape")
_check_private_tensor("ex086_gain_view", "ex086", "gain_view")
_check_private_tensor("ex086_out", "ex086", "out")


AssertionError: Define `ex086_images_shape` in the answer cell first.

### Exercise 087 — Attention mask

**Purpose:** Broadcast one query-key mask across batches and heads.

**Visible inputs:** `scores`, `mask`. Run the fixture below and read its construction code; it deliberately prints no shape answers.

**Operation to reason about:** `scores.masked_fill(~mask, -1e9)`

**Task:** Mask future key positions in every batch and head.

**Ingredients:** The `(query, key)` mask right-aligns with the final two score axes.

**Required predictions and outputs:**

- `ex087_scores_shape`: a literal Python shape tuple
- `ex087_mask_shape`: a literal Python shape tuple
- `ex087_scores_aligned_shape`: a literal Python shape tuple
- `ex087_scores_expanded_axes`: a tuple of zero-based aligned axes
- `ex087_mask_aligned_shape`: a literal Python shape tuple
- `ex087_mask_expanded_axes`: a tuple of zero-based aligned axes
- `ex087_compatible`: a Python `bool`
- `ex087_out_shape`: a literal Python shape tuple
- `ex087_out`: the requested PyTorch tensor

**Order:** Fill every shape/alignment prediction before writing tensor operations.

**Next concept:** Per-head attention scale


In [ ]:
# Supplied visible fixture: inspect these values, but predict shapes without printing them.
scores = torch.arange(96, dtype=DTYPE).reshape(2, 3, 4, 4)
mask = torch.tril(torch.ones((4, 4), dtype=torch.bool))
_register_case("ex087", {"scores": scores, "mask": mask})


In [ ]:
# Exercise 087: predict shapes first; do not use autograd.
# Define `ex087_scores_shape`.
# Define `ex087_mask_shape`.
# Define `ex087_scores_aligned_shape`.
# Define `ex087_scores_expanded_axes`.
# Define `ex087_mask_aligned_shape`.
# Define `ex087_mask_expanded_axes`.
# Define `ex087_compatible`.
# Define `ex087_out_shape`.
# Define `ex087_out` with PyTorch tensor operations.
# Write your work below, then run the supplied test cell.


In [ ]:
# Supplied test: expected shapes and values remain private.
_check_private_value("ex087_scores_shape", "ex087", "scores_shape")
_check_private_value("ex087_mask_shape", "ex087", "mask_shape")
_check_private_value("ex087_scores_aligned_shape", "ex087", "scores_aligned_shape")
_check_private_value("ex087_scores_expanded_axes", "ex087", "scores_expanded_axes")
_check_private_value("ex087_mask_aligned_shape", "ex087", "mask_aligned_shape")
_check_private_value("ex087_mask_expanded_axes", "ex087", "mask_expanded_axes")
_check_private_value("ex087_compatible", "ex087", "compatible")
_check_private_value("ex087_out_shape", "ex087", "out_shape")
_check_private_tensor("ex087_out", "ex087", "out")


### Exercise 088 — Per-head attention scale

**Purpose:** Prepare one scale per head for a four-axis attention tensor.

**Visible inputs:** `scores`, `head_scale`. Run the fixture below and read its construction code; it deliberately prints no shape answers.

**Operation to reason about:** Create `scale_view`, then scale each attention head.

**Task:** Create `scale_view`, then scale each attention head.

**Ingredients:** Place head at axis 1 and make batch, query, and key singleton.

**Required predictions and outputs:**

- `ex088_scores_shape`: a literal Python shape tuple
- `ex088_head_scale_shape`: a literal Python shape tuple
- `ex088_scale_view_shape`: a literal Python shape tuple
- `ex088_scale_view_aligned_shape`: a literal Python shape tuple
- `ex088_scale_view_expanded_axes`: a tuple of zero-based aligned axes
- `ex088_scores_aligned_shape`: a literal Python shape tuple
- `ex088_scores_expanded_axes`: a tuple of zero-based aligned axes
- `ex088_compatible`: a Python `bool`
- `ex088_out_shape`: a literal Python shape tuple
- `ex088_scale_view`: the requested PyTorch tensor
- `ex088_out`: the requested PyTorch tensor

**Order:** Fill every shape/alignment prediction before writing tensor operations.

**Next concept:** Mixture-of-experts weights


In [ ]:
# Supplied visible fixture: inspect these values, but predict shapes without printing them.
scores = torch.ones((2, 3, 4, 4), dtype=DTYPE)
head_scale = torch.tensor([0.5, 1.0, 2.0], dtype=DTYPE)
_register_case("ex088", {"scores": scores, "head_scale": head_scale})


In [ ]:
# Exercise 088: predict shapes first; do not use autograd.
# Define `ex088_scores_shape`.
# Define `ex088_head_scale_shape`.
# Define `ex088_scale_view_shape`.
# Define `ex088_scale_view_aligned_shape`.
# Define `ex088_scale_view_expanded_axes`.
# Define `ex088_scores_aligned_shape`.
# Define `ex088_scores_expanded_axes`.
# Define `ex088_compatible`.
# Define `ex088_out_shape`.
# Define `ex088_scale_view` with PyTorch tensor operations.
# Define `ex088_out` with PyTorch tensor operations.
# Write your work below, then run the supplied test cell.


In [ ]:
# Supplied test: expected shapes and values remain private.
_check_private_value("ex088_scores_shape", "ex088", "scores_shape")
_check_private_value("ex088_head_scale_shape", "ex088", "head_scale_shape")
_check_private_value("ex088_scale_view_shape", "ex088", "scale_view_shape")
_check_private_value("ex088_scale_view_aligned_shape", "ex088", "scale_view_aligned_shape")
_check_private_value("ex088_scale_view_expanded_axes", "ex088", "scale_view_expanded_axes")
_check_private_value("ex088_scores_aligned_shape", "ex088", "scores_aligned_shape")
_check_private_value("ex088_scores_expanded_axes", "ex088", "scores_expanded_axes")
_check_private_value("ex088_compatible", "ex088", "compatible")
_check_private_value("ex088_out_shape", "ex088", "out_shape")
_check_private_tensor("ex088_scale_view", "ex088", "scale_view")
_check_private_tensor("ex088_out", "ex088", "out")


### Exercise 089 — Mixture-of-experts weights

**Purpose:** Expand one expert weight across each expert's output features.

**Visible inputs:** `weights`, `experts`. Run the fixture below and read its construction code; it deliberately prints no shape answers.

**Operation to reason about:** Create `weight_view` and compute weighted expert outputs without reducing experts yet.

**Task:** Create `weight_view` and compute weighted expert outputs without reducing experts yet.

**Ingredients:** Insert one final feature axis into `(batch, experts)` weights.

**Required predictions and outputs:**

- `ex089_weights_shape`: a literal Python shape tuple
- `ex089_experts_shape`: a literal Python shape tuple
- `ex089_weight_view_shape`: a literal Python shape tuple
- `ex089_weight_view_aligned_shape`: a literal Python shape tuple
- `ex089_weight_view_expanded_axes`: a tuple of zero-based aligned axes
- `ex089_experts_aligned_shape`: a literal Python shape tuple
- `ex089_experts_expanded_axes`: a tuple of zero-based aligned axes
- `ex089_compatible`: a Python `bool`
- `ex089_out_shape`: a literal Python shape tuple
- `ex089_weight_view`: the requested PyTorch tensor
- `ex089_out`: the requested PyTorch tensor

**Order:** Fill every shape/alignment prediction before writing tensor operations.

**Next concept:** Pairwise feature differences


In [ ]:
# Supplied visible fixture: inspect these values, but predict shapes without printing them.
weights = torch.tensor([[0.2, 0.3, 0.5], [0.5, 0.25, 0.25]], dtype=DTYPE)
experts = torch.arange(24, dtype=DTYPE).reshape(2, 3, 4)
_register_case("ex089", {"weights": weights, "experts": experts})


In [ ]:
# Exercise 089: predict shapes first; do not use autograd.
# Define `ex089_weights_shape`.
# Define `ex089_experts_shape`.
# Define `ex089_weight_view_shape`.
# Define `ex089_weight_view_aligned_shape`.
# Define `ex089_weight_view_expanded_axes`.
# Define `ex089_experts_aligned_shape`.
# Define `ex089_experts_expanded_axes`.
# Define `ex089_compatible`.
# Define `ex089_out_shape`.
# Define `ex089_weight_view` with PyTorch tensor operations.
# Define `ex089_out` with PyTorch tensor operations.
# Write your work below, then run the supplied test cell.


In [ ]:
# Supplied test: expected shapes and values remain private.
_check_private_value("ex089_weights_shape", "ex089", "weights_shape")
_check_private_value("ex089_experts_shape", "ex089", "experts_shape")
_check_private_value("ex089_weight_view_shape", "ex089", "weight_view_shape")
_check_private_value("ex089_weight_view_aligned_shape", "ex089", "weight_view_aligned_shape")
_check_private_value("ex089_weight_view_expanded_axes", "ex089", "weight_view_expanded_axes")
_check_private_value("ex089_experts_aligned_shape", "ex089", "experts_aligned_shape")
_check_private_value("ex089_experts_expanded_axes", "ex089", "experts_expanded_axes")
_check_private_value("ex089_compatible", "ex089", "compatible")
_check_private_value("ex089_out_shape", "ex089", "out_shape")
_check_private_tensor("ex089_weight_view", "ex089", "weight_view")
_check_private_tensor("ex089_out", "ex089", "out")


### Exercise 090 — Pairwise feature differences

**Purpose:** Use opposite singleton item axes to build every pair.

**Visible inputs:** `x`, `y`. Run the fixture below and read its construction code; it deliberately prints no shape answers.

**Operation to reason about:** Create `x_rows` and `y_rows`, then compute every feature-wise pair difference.

**Task:** Create `x_rows` and `y_rows`, then compute every feature-wise pair difference.

**Ingredients:** Represent `x` as `(x_items, 1, features)` and `y` as `(1, y_items, features)`.

**Required predictions and outputs:**

- `ex090_x_shape`: a literal Python shape tuple
- `ex090_y_shape`: a literal Python shape tuple
- `ex090_x_rows_shape`: a literal Python shape tuple
- `ex090_y_rows_shape`: a literal Python shape tuple
- `ex090_x_rows_aligned_shape`: a literal Python shape tuple
- `ex090_x_rows_expanded_axes`: a tuple of zero-based aligned axes
- `ex090_y_rows_aligned_shape`: a literal Python shape tuple
- `ex090_y_rows_expanded_axes`: a tuple of zero-based aligned axes
- `ex090_compatible`: a Python `bool`
- `ex090_out_shape`: a literal Python shape tuple
- `ex090_x_rows`: the requested PyTorch tensor
- `ex090_y_rows`: the requested PyTorch tensor
- `ex090_out`: the requested PyTorch tensor

**Order:** Fill every shape/alignment prediction before writing tensor operations.

**Next concept:** Backward through row bias


In [ ]:
# Supplied visible fixture: inspect these values, but predict shapes without printing them.
x = torch.arange(6, dtype=DTYPE).reshape(2, 3)
y = torch.arange(12, dtype=DTYPE).reshape(4, 3)
_register_case("ex090", {"x": x, "y": y})


In [ ]:
# Exercise 090: predict shapes first; do not use autograd.
# Define `ex090_x_shape`.
# Define `ex090_y_shape`.
# Define `ex090_x_rows_shape`.
# Define `ex090_y_rows_shape`.
# Define `ex090_x_rows_aligned_shape`.
# Define `ex090_x_rows_expanded_axes`.
# Define `ex090_y_rows_aligned_shape`.
# Define `ex090_y_rows_expanded_axes`.
# Define `ex090_compatible`.
# Define `ex090_out_shape`.
# Define `ex090_x_rows` with PyTorch tensor operations.
# Define `ex090_y_rows` with PyTorch tensor operations.
# Define `ex090_out` with PyTorch tensor operations.
# Write your work below, then run the supplied test cell.


In [ ]:
# Supplied test: expected shapes and values remain private.
_check_private_value("ex090_x_shape", "ex090", "x_shape")
_check_private_value("ex090_y_shape", "ex090", "y_shape")
_check_private_value("ex090_x_rows_shape", "ex090", "x_rows_shape")
_check_private_value("ex090_y_rows_shape", "ex090", "y_rows_shape")
_check_private_value("ex090_x_rows_aligned_shape", "ex090", "x_rows_aligned_shape")
_check_private_value("ex090_x_rows_expanded_axes", "ex090", "x_rows_expanded_axes")
_check_private_value("ex090_y_rows_aligned_shape", "ex090", "y_rows_aligned_shape")
_check_private_value("ex090_y_rows_expanded_axes", "ex090", "y_rows_expanded_axes")
_check_private_value("ex090_compatible", "ex090", "compatible")
_check_private_value("ex090_out_shape", "ex090", "out_shape")
_check_private_tensor("ex090_x_rows", "ex090", "x_rows")
_check_private_tensor("ex090_y_rows", "ex090", "y_rows")
_check_private_tensor("ex090_out", "ex090", "out")


## 9. Backward must unbroadcast

Forward reuses compact values. Backward adds all reuse paths until each gradient returns to its forward variable's original shape.

For a tensor `p` expanded during forward, begin with the output-sized path gradient and sum every axis that was added or expanded. Preserve original singleton axes when `p` preserved them.

$$
\operatorname{shape}(dp)
=
\operatorname{shape}(p)
$$



### Exercise 091 — Backward through row bias

**Purpose:** Unbroadcast a feature bias reused across examples.

**Visible inputs:** `x`, `bias`, `dout`. Run the fixture below and read its construction code; it deliberately prints no shape answers.

**Operation to reason about:** `x + bias`

**Task:** Derive `dx` and `dbias` manually.

**Ingredients:** `dx` follows the full output; `dbias` sums all reuse paths over examples.

**Required predictions and outputs:**

- `ex091_x_shape`: a literal Python shape tuple
- `ex091_bias_shape`: a literal Python shape tuple
- `ex091_dout_shape`: a literal Python shape tuple
- `ex091_y_shape`: a literal Python shape tuple
- `ex091_dx_shape`: a literal Python shape tuple
- `ex091_dbias_shape`: a literal Python shape tuple
- `ex091_dx`: the requested PyTorch tensor
- `ex091_dbias`: the requested PyTorch tensor

**Order:** Fill every shape/alignment prediction before writing tensor operations.

**Next concept:** Backward through column bias


In [ ]:
# Supplied visible fixture: inspect these values, but predict shapes without printing them.
x = torch.arange(6, dtype=DTYPE).reshape(2, 3).requires_grad_()
bias = torch.tensor([0.1, 0.2, 0.3], dtype=DTYPE, requires_grad=True)
dout = torch.tensor([[1.0, 2.0, 3.0], [-1.0, 0.5, 2.0]], dtype=DTYPE)
_register_case("ex091", {"x": x, "bias": bias, "dout": dout})


In [ ]:
# Exercise 091: predict shapes first; do not use autograd.
# Define `ex091_x_shape`.
# Define `ex091_bias_shape`.
# Define `ex091_dout_shape`.
# Define `ex091_y_shape`.
# Define `ex091_dx_shape`.
# Define `ex091_dbias_shape`.
# Define `ex091_dx` with PyTorch tensor operations.
# Define `ex091_dbias` with PyTorch tensor operations.
# Write your work below, then run the supplied test cell.


In [ ]:
# Supplied test: expected shapes and values remain private.
_check_private_value("ex091_x_shape", "ex091", "x_shape")
_check_private_value("ex091_bias_shape", "ex091", "bias_shape")
_check_private_value("ex091_dout_shape", "ex091", "dout_shape")
_check_private_value("ex091_y_shape", "ex091", "y_shape")
_check_private_value("ex091_dx_shape", "ex091", "dx_shape")
_check_private_value("ex091_dbias_shape", "ex091", "dbias_shape")
_check_private_tensor("ex091_dx", "ex091", "dx")
_check_private_tensor("ex091_dbias", "ex091", "dbias")


### Exercise 092 — Backward through column bias

**Purpose:** Unbroadcast one row-owned parameter reused across columns.

**Visible inputs:** `x`, `bias`, `dout`. Run the fixture below and read its construction code; it deliberately prints no shape answers.

**Operation to reason about:** `x + bias`

**Task:** Derive `dx` and `dbias`.

**Ingredients:** Sum `dout` over the axis where the singleton bias expanded, keeping that axis.

**Required predictions and outputs:**

- `ex092_x_shape`: a literal Python shape tuple
- `ex092_bias_shape`: a literal Python shape tuple
- `ex092_dout_shape`: a literal Python shape tuple
- `ex092_y_shape`: a literal Python shape tuple
- `ex092_dx_shape`: a literal Python shape tuple
- `ex092_dbias_shape`: a literal Python shape tuple
- `ex092_dx`: the requested PyTorch tensor
- `ex092_dbias`: the requested PyTorch tensor

**Order:** Fill every shape/alignment prediction before writing tensor operations.

**Next concept:** Backward through scalar bias


In [ ]:
# Supplied visible fixture: inspect these values, but predict shapes without printing them.
x = torch.arange(6, dtype=DTYPE).reshape(2, 3).requires_grad_()
bias = torch.tensor([[0.5], [-0.5]], dtype=DTYPE, requires_grad=True)
dout = torch.tensor([[1.0, 2.0, 3.0], [-1.0, 0.5, 2.0]], dtype=DTYPE)
_register_case("ex092", {"x": x, "bias": bias, "dout": dout})


In [ ]:
# Exercise 092: predict shapes first; do not use autograd.
# Define `ex092_x_shape`.
# Define `ex092_bias_shape`.
# Define `ex092_dout_shape`.
# Define `ex092_y_shape`.
# Define `ex092_dx_shape`.
# Define `ex092_dbias_shape`.
# Define `ex092_dx` with PyTorch tensor operations.
# Define `ex092_dbias` with PyTorch tensor operations.
# Write your work below, then run the supplied test cell.


In [ ]:
# Supplied test: expected shapes and values remain private.
_check_private_value("ex092_x_shape", "ex092", "x_shape")
_check_private_value("ex092_bias_shape", "ex092", "bias_shape")
_check_private_value("ex092_dout_shape", "ex092", "dout_shape")
_check_private_value("ex092_y_shape", "ex092", "y_shape")
_check_private_value("ex092_dx_shape", "ex092", "dx_shape")
_check_private_value("ex092_dbias_shape", "ex092", "dbias_shape")
_check_private_tensor("ex092_dx", "ex092", "dx")
_check_private_tensor("ex092_dbias", "ex092", "dbias")


### Exercise 093 — Backward through scalar bias

**Purpose:** Accumulate every output contribution into one scalar.

**Visible inputs:** `x`, `bias`, `dout`. Run the fixture below and read its construction code; it deliberately prints no shape answers.

**Operation to reason about:** `x + bias`

**Task:** Derive `dx` and scalar `dbias`.

**Ingredients:** A scalar was reused at every output position.

**Required predictions and outputs:**

- `ex093_x_shape`: a literal Python shape tuple
- `ex093_bias_shape`: a literal Python shape tuple
- `ex093_dout_shape`: a literal Python shape tuple
- `ex093_y_shape`: a literal Python shape tuple
- `ex093_dx_shape`: a literal Python shape tuple
- `ex093_dbias_shape`: a literal Python shape tuple
- `ex093_dx`: the requested PyTorch tensor
- `ex093_dbias`: the requested PyTorch tensor

**Order:** Fill every shape/alignment prediction before writing tensor operations.

**Next concept:** Backward through row scale


In [ ]:
# Supplied visible fixture: inspect these values, but predict shapes without printing them.
x = torch.arange(6, dtype=DTYPE).reshape(2, 3).requires_grad_()
bias = torch.tensor(0.5, dtype=DTYPE, requires_grad=True)
dout = torch.linspace(-1.0, 1.0, steps=6, dtype=DTYPE).reshape(2, 3)
_register_case("ex093", {"x": x, "bias": bias, "dout": dout})


In [ ]:
# Exercise 093: predict shapes first; do not use autograd.
# Define `ex093_x_shape`.
# Define `ex093_bias_shape`.
# Define `ex093_dout_shape`.
# Define `ex093_y_shape`.
# Define `ex093_dx_shape`.
# Define `ex093_dbias_shape`.
# Define `ex093_dx` with PyTorch tensor operations.
# Define `ex093_dbias` with PyTorch tensor operations.
# Write your work below, then run the supplied test cell.


In [ ]:
# Supplied test: expected shapes and values remain private.
_check_private_value("ex093_x_shape", "ex093", "x_shape")
_check_private_value("ex093_bias_shape", "ex093", "bias_shape")
_check_private_value("ex093_dout_shape", "ex093", "dout_shape")
_check_private_value("ex093_y_shape", "ex093", "y_shape")
_check_private_value("ex093_dx_shape", "ex093", "dx_shape")
_check_private_value("ex093_dbias_shape", "ex093", "dbias_shape")
_check_private_tensor("ex093_dx", "ex093", "dx")
_check_private_tensor("ex093_dbias", "ex093", "dbias")


### Exercise 094 — Backward through row scale

**Purpose:** Combine multiplication's local derivative with unbroadcasting.

**Visible inputs:** `x`, `scale`, `dout`. Run the fixture below and read its construction code; it deliberately prints no shape answers.

**Operation to reason about:** `x * scale`

**Task:** Derive `dx` and `dscale`.

**Ingredients:** Multiply by the opposite operand first; then sum only the gradient of the expanded operand.

**Required predictions and outputs:**

- `ex094_x_shape`: a literal Python shape tuple
- `ex094_scale_shape`: a literal Python shape tuple
- `ex094_dout_shape`: a literal Python shape tuple
- `ex094_y_shape`: a literal Python shape tuple
- `ex094_dx_shape`: a literal Python shape tuple
- `ex094_dscale_shape`: a literal Python shape tuple
- `ex094_dx`: the requested PyTorch tensor
- `ex094_dscale`: the requested PyTorch tensor

**Order:** Fill every shape/alignment prediction before writing tensor operations.

**Next concept:** Backward through column scale


In [ ]:
# Supplied visible fixture: inspect these values, but predict shapes without printing them.
x = (torch.arange(6, dtype=DTYPE).reshape(2, 3) + 1).requires_grad_()
scale = torch.tensor([1.0, 2.0, 3.0], dtype=DTYPE, requires_grad=True)
dout = torch.tensor([[1.0, -1.0, 0.5], [2.0, 1.0, -0.5]], dtype=DTYPE)
_register_case("ex094", {"x": x, "scale": scale, "dout": dout})


In [ ]:
# Exercise 094: predict shapes first; do not use autograd.
# Define `ex094_x_shape`.
# Define `ex094_scale_shape`.
# Define `ex094_dout_shape`.
# Define `ex094_y_shape`.
# Define `ex094_dx_shape`.
# Define `ex094_dscale_shape`.
# Define `ex094_dx` with PyTorch tensor operations.
# Define `ex094_dscale` with PyTorch tensor operations.
# Write your work below, then run the supplied test cell.


In [ ]:
# Supplied test: expected shapes and values remain private.
_check_private_value("ex094_x_shape", "ex094", "x_shape")
_check_private_value("ex094_scale_shape", "ex094", "scale_shape")
_check_private_value("ex094_dout_shape", "ex094", "dout_shape")
_check_private_value("ex094_y_shape", "ex094", "y_shape")
_check_private_value("ex094_dx_shape", "ex094", "dx_shape")
_check_private_value("ex094_dscale_shape", "ex094", "dscale_shape")
_check_private_tensor("ex094_dx", "ex094", "dx")
_check_private_tensor("ex094_dscale", "ex094", "dscale")


### Exercise 095 — Backward through column scale

**Purpose:** Unbroadcast a `(rows, 1)` scale after multiplication.

**Visible inputs:** `x`, `scale`, `dout`. Run the fixture below and read its construction code; it deliberately prints no shape answers.

**Operation to reason about:** `x * scale`

**Task:** Derive `dx` and `dscale`.

**Ingredients:** The scale gradient must sum across columns and retain its singleton axis.

**Required predictions and outputs:**

- `ex095_x_shape`: a literal Python shape tuple
- `ex095_scale_shape`: a literal Python shape tuple
- `ex095_dout_shape`: a literal Python shape tuple
- `ex095_y_shape`: a literal Python shape tuple
- `ex095_dx_shape`: a literal Python shape tuple
- `ex095_dscale_shape`: a literal Python shape tuple
- `ex095_dx`: the requested PyTorch tensor
- `ex095_dscale`: the requested PyTorch tensor

**Order:** Fill every shape/alignment prediction before writing tensor operations.

**Next concept:** Backward through rank-3 feature bias


In [ ]:
# Supplied visible fixture: inspect these values, but predict shapes without printing them.
x = (torch.arange(6, dtype=DTYPE).reshape(2, 3) + 1).requires_grad_()
scale = torch.tensor([[2.0], [-1.0]], dtype=DTYPE, requires_grad=True)
dout = torch.tensor([[1.0, -1.0, 0.5], [2.0, 1.0, -0.5]], dtype=DTYPE)
_register_case("ex095", {"x": x, "scale": scale, "dout": dout})


In [ ]:
# Exercise 095: predict shapes first; do not use autograd.
# Define `ex095_x_shape`.
# Define `ex095_scale_shape`.
# Define `ex095_dout_shape`.
# Define `ex095_y_shape`.
# Define `ex095_dx_shape`.
# Define `ex095_dscale_shape`.
# Define `ex095_dx` with PyTorch tensor operations.
# Define `ex095_dscale` with PyTorch tensor operations.
# Write your work below, then run the supplied test cell.


In [ ]:
# Supplied test: expected shapes and values remain private.
_check_private_value("ex095_x_shape", "ex095", "x_shape")
_check_private_value("ex095_scale_shape", "ex095", "scale_shape")
_check_private_value("ex095_dout_shape", "ex095", "dout_shape")
_check_private_value("ex095_y_shape", "ex095", "y_shape")
_check_private_value("ex095_dx_shape", "ex095", "dx_shape")
_check_private_value("ex095_dscale_shape", "ex095", "dscale_shape")
_check_private_tensor("ex095_dx", "ex095", "dx")
_check_private_tensor("ex095_dscale", "ex095", "dscale")


### Exercise 096 — Backward through rank-3 feature bias

**Purpose:** Sum a feature bias gradient over two reused leading axes.

**Visible inputs:** `x`, `bias`, `dout`. Run the fixture below and read its construction code; it deliberately prints no shape answers.

**Operation to reason about:** `x + bias`

**Task:** Derive `dx` and `dbias`.

**Ingredients:** The compact bias was reused over batch and time.

**Required predictions and outputs:**

- `ex096_x_shape`: a literal Python shape tuple
- `ex096_bias_shape`: a literal Python shape tuple
- `ex096_dout_shape`: a literal Python shape tuple
- `ex096_y_shape`: a literal Python shape tuple
- `ex096_dx_shape`: a literal Python shape tuple
- `ex096_dbias_shape`: a literal Python shape tuple
- `ex096_dx`: the requested PyTorch tensor
- `ex096_dbias`: the requested PyTorch tensor

**Order:** Fill every shape/alignment prediction before writing tensor operations.

**Next concept:** Backward through image channel bias


In [ ]:
# Supplied visible fixture: inspect these values, but predict shapes without printing them.
x = torch.arange(24, dtype=DTYPE).reshape(2, 3, 4).requires_grad_()
bias = torch.tensor([0.1, 0.2, 0.3, 0.4], dtype=DTYPE, requires_grad=True)
dout = torch.linspace(-1.0, 1.0, steps=24, dtype=DTYPE).reshape(2, 3, 4)
_register_case("ex096", {"x": x, "bias": bias, "dout": dout})


In [ ]:
# Exercise 096: predict shapes first; do not use autograd.
# Define `ex096_x_shape`.
# Define `ex096_bias_shape`.
# Define `ex096_dout_shape`.
# Define `ex096_y_shape`.
# Define `ex096_dx_shape`.
# Define `ex096_dbias_shape`.
# Define `ex096_dx` with PyTorch tensor operations.
# Define `ex096_dbias` with PyTorch tensor operations.
# Write your work below, then run the supplied test cell.


In [ ]:
# Supplied test: expected shapes and values remain private.
_check_private_value("ex096_x_shape", "ex096", "x_shape")
_check_private_value("ex096_bias_shape", "ex096", "bias_shape")
_check_private_value("ex096_dout_shape", "ex096", "dout_shape")
_check_private_value("ex096_y_shape", "ex096", "y_shape")
_check_private_value("ex096_dx_shape", "ex096", "dx_shape")
_check_private_value("ex096_dbias_shape", "ex096", "dbias_shape")
_check_private_tensor("ex096_dx", "ex096", "dx")
_check_private_tensor("ex096_dbias", "ex096", "dbias")


### Exercise 097 — Backward through image channel bias

**Purpose:** Return a rank-4 broadcast gradient to `(1, channel, 1, 1)`.

**Visible inputs:** `images`, `bias`, `dout`. Run the fixture below and read its construction code; it deliberately prints no shape answers.

**Operation to reason about:** `images + bias`

**Task:** Derive `dimages` and `dbias`.

**Ingredients:** Sum over batch, height, and width while preserving all original bias axes.

**Required predictions and outputs:**

- `ex097_images_shape`: a literal Python shape tuple
- `ex097_bias_shape`: a literal Python shape tuple
- `ex097_dout_shape`: a literal Python shape tuple
- `ex097_y_shape`: a literal Python shape tuple
- `ex097_dimages_shape`: a literal Python shape tuple
- `ex097_dbias_shape`: a literal Python shape tuple
- `ex097_dimages`: the requested PyTorch tensor
- `ex097_dbias`: the requested PyTorch tensor

**Order:** Fill every shape/alignment prediction before writing tensor operations.

**Next concept:** Backward through outer addition


In [ ]:
# Supplied visible fixture: inspect these values, but predict shapes without printing them.
images = torch.arange(120, dtype=DTYPE).reshape(2, 3, 4, 5).requires_grad_()
bias = torch.tensor([[[[0.1]], [[0.2]], [[0.3]]]], dtype=DTYPE, requires_grad=True)
dout = torch.linspace(-1.0, 1.0, steps=120, dtype=DTYPE).reshape(2, 3, 4, 5)
_register_case("ex097", {"images": images, "bias": bias, "dout": dout})


In [ ]:
# Exercise 097: predict shapes first; do not use autograd.
# Define `ex097_images_shape`.
# Define `ex097_bias_shape`.
# Define `ex097_dout_shape`.
# Define `ex097_y_shape`.
# Define `ex097_dimages_shape`.
# Define `ex097_dbias_shape`.
# Define `ex097_dimages` with PyTorch tensor operations.
# Define `ex097_dbias` with PyTorch tensor operations.
# Write your work below, then run the supplied test cell.


In [ ]:
# Supplied test: expected shapes and values remain private.
_check_private_value("ex097_images_shape", "ex097", "images_shape")
_check_private_value("ex097_bias_shape", "ex097", "bias_shape")
_check_private_value("ex097_dout_shape", "ex097", "dout_shape")
_check_private_value("ex097_y_shape", "ex097", "y_shape")
_check_private_value("ex097_dimages_shape", "ex097", "dimages_shape")
_check_private_value("ex097_dbias_shape", "ex097", "dbias_shape")
_check_private_tensor("ex097_dimages", "ex097", "dimages")
_check_private_tensor("ex097_dbias", "ex097", "dbias")


### Exercise 098 — Backward through outer addition

**Purpose:** Unbroadcast two operands along different axes.

**Visible inputs:** `a`, `b`, `dout`. Run the fixture below and read its construction code; it deliberately prints no shape answers.

**Operation to reason about:** `a + b`

**Task:** Derive `da` and `db`.

**Ingredients:** Each operand sums over the axis where it had size one.

**Required predictions and outputs:**

- `ex098_a_shape`: a literal Python shape tuple
- `ex098_b_shape`: a literal Python shape tuple
- `ex098_dout_shape`: a literal Python shape tuple
- `ex098_y_shape`: a literal Python shape tuple
- `ex098_da_shape`: a literal Python shape tuple
- `ex098_db_shape`: a literal Python shape tuple
- `ex098_da`: the requested PyTorch tensor
- `ex098_db`: the requested PyTorch tensor

**Order:** Fill every shape/alignment prediction before writing tensor operations.

**Next concept:** Backward after broadcast and square


In [ ]:
# Supplied visible fixture: inspect these values, but predict shapes without printing them.
a = torch.tensor([[1.0], [2.0]], dtype=DTYPE, requires_grad=True)
b = torch.tensor([[10.0, 20.0, 30.0]], dtype=DTYPE, requires_grad=True)
dout = torch.tensor([[1.0, 2.0, 3.0], [-1.0, 0.5, 2.0]], dtype=DTYPE)
_register_case("ex098", {"a": a, "b": b, "dout": dout})


In [ ]:
# Exercise 098: predict shapes first; do not use autograd.
# Define `ex098_a_shape`.
# Define `ex098_b_shape`.
# Define `ex098_dout_shape`.
# Define `ex098_y_shape`.
# Define `ex098_da_shape`.
# Define `ex098_db_shape`.
# Define `ex098_da` with PyTorch tensor operations.
# Define `ex098_db` with PyTorch tensor operations.
# Write your work below, then run the supplied test cell.


In [ ]:
# Supplied test: expected shapes and values remain private.
_check_private_value("ex098_a_shape", "ex098", "a_shape")
_check_private_value("ex098_b_shape", "ex098", "b_shape")
_check_private_value("ex098_dout_shape", "ex098", "dout_shape")
_check_private_value("ex098_y_shape", "ex098", "y_shape")
_check_private_value("ex098_da_shape", "ex098", "da_shape")
_check_private_value("ex098_db_shape", "ex098", "db_shape")
_check_private_tensor("ex098_da", "ex098", "da")
_check_private_tensor("ex098_db", "ex098", "db")


### Exercise 099 — Backward after broadcast and square

**Purpose:** Trace a broadcasted bias through a nonlinear local derivative.

**Visible inputs:** `x`, `bias`, `dout`. Run the fixture below and read its construction code; it deliberately prints no shape answers.

**Operation to reason about:** `(x + bias) ** 2`

**Task:** Derive `dx` and `dbias`.

**Ingredients:** First differentiate the square at every output position; then unbroadcast the bias path.

**Required predictions and outputs:**

- `ex099_x_shape`: a literal Python shape tuple
- `ex099_bias_shape`: a literal Python shape tuple
- `ex099_dout_shape`: a literal Python shape tuple
- `ex099_y_shape`: a literal Python shape tuple
- `ex099_dx_shape`: a literal Python shape tuple
- `ex099_dbias_shape`: a literal Python shape tuple
- `ex099_dx`: the requested PyTorch tensor
- `ex099_dbias`: the requested PyTorch tensor

**Order:** Fill every shape/alignment prediction before writing tensor operations.

**Next concept:** Backward through row division


In [ ]:
# Supplied visible fixture: inspect these values, but predict shapes without printing them.
x = torch.arange(6, dtype=DTYPE).reshape(2, 3).requires_grad_()
bias = torch.tensor([0.5, -1.0, 2.0], dtype=DTYPE, requires_grad=True)
dout = torch.tensor([[1.0, -1.0, 0.5], [2.0, 1.0, -0.5]], dtype=DTYPE)
_register_case("ex099", {"x": x, "bias": bias, "dout": dout})


In [ ]:
# Exercise 099: predict shapes first; do not use autograd.
# Define `ex099_x_shape`.
# Define `ex099_bias_shape`.
# Define `ex099_dout_shape`.
# Define `ex099_y_shape`.
# Define `ex099_dx_shape`.
# Define `ex099_dbias_shape`.
# Define `ex099_dx` with PyTorch tensor operations.
# Define `ex099_dbias` with PyTorch tensor operations.
# Write your work below, then run the supplied test cell.


In [ ]:
# Supplied test: expected shapes and values remain private.
_check_private_value("ex099_x_shape", "ex099", "x_shape")
_check_private_value("ex099_bias_shape", "ex099", "bias_shape")
_check_private_value("ex099_dout_shape", "ex099", "dout_shape")
_check_private_value("ex099_y_shape", "ex099", "y_shape")
_check_private_value("ex099_dx_shape", "ex099", "dx_shape")
_check_private_value("ex099_dbias_shape", "ex099", "dbias_shape")
_check_private_tensor("ex099_dx", "ex099", "dx")
_check_private_tensor("ex099_dbias", "ex099", "dbias")


### Exercise 100 — Backward through row division

**Purpose:** Handle reciprocal scaling and unbroadcasting together.

**Visible inputs:** `x`, `scale`, `dout`. Run the fixture below and read its construction code; it deliberately prints no shape answers.

**Operation to reason about:** `x / scale`

**Task:** Derive `dx` and `dscale`.

**Ingredients:** The scale's local derivative includes a negative reciprocal-square factor before summing columns.

**Required predictions and outputs:**

- `ex100_x_shape`: a literal Python shape tuple
- `ex100_scale_shape`: a literal Python shape tuple
- `ex100_dout_shape`: a literal Python shape tuple
- `ex100_y_shape`: a literal Python shape tuple
- `ex100_dx_shape`: a literal Python shape tuple
- `ex100_dscale_shape`: a literal Python shape tuple
- `ex100_dx`: the requested PyTorch tensor
- `ex100_dscale`: the requested PyTorch tensor

**Order:** Fill every shape/alignment prediction before writing tensor operations.

**Next concept:** Backward through column centering


In [ ]:
# Supplied visible fixture: inspect these values, but predict shapes without printing them.
x = (torch.arange(6, dtype=DTYPE).reshape(2, 3) + 1).requires_grad_()
scale = torch.tensor([[2.0], [4.0]], dtype=DTYPE, requires_grad=True)
dout = torch.tensor([[1.0, -1.0, 0.5], [2.0, 1.0, -0.5]], dtype=DTYPE)
_register_case("ex100", {"x": x, "scale": scale, "dout": dout})


In [ ]:
# Exercise 100: predict shapes first; do not use autograd.
# Define `ex100_x_shape`.
# Define `ex100_scale_shape`.
# Define `ex100_dout_shape`.
# Define `ex100_y_shape`.
# Define `ex100_dx_shape`.
# Define `ex100_dscale_shape`.
# Define `ex100_dx` with PyTorch tensor operations.
# Define `ex100_dscale` with PyTorch tensor operations.
# Write your work below, then run the supplied test cell.


In [ ]:
# Supplied test: expected shapes and values remain private.
_check_private_value("ex100_x_shape", "ex100", "x_shape")
_check_private_value("ex100_scale_shape", "ex100", "scale_shape")
_check_private_value("ex100_dout_shape", "ex100", "dout_shape")
_check_private_value("ex100_y_shape", "ex100", "y_shape")
_check_private_value("ex100_dx_shape", "ex100", "dx_shape")
_check_private_value("ex100_dscale_shape", "ex100", "dscale_shape")
_check_private_tensor("ex100_dx", "ex100", "dx")
_check_private_tensor("ex100_dscale", "ex100", "dscale")


### Exercise 101 — Backward through column centering

**Purpose:** See forward reduction and forward broadcasting in one graph.

**Visible inputs:** `x`, `dout`. Run the fixture below and read its construction code; it deliberately prints no shape answers.

**Operation to reason about:** `x - x.mean(dim=0, keepdim=True)`

**Task:** Derive the complete `dx`, including direct and mean paths.

**Ingredients:** The computed mean is broadcast down rows, and every input also has a direct subtraction path.

**Required predictions and outputs:**

- `ex101_x_shape`: a literal Python shape tuple
- `ex101_dout_shape`: a literal Python shape tuple
- `ex101_y_shape`: a literal Python shape tuple
- `ex101_dx_shape`: a literal Python shape tuple
- `ex101_dx`: the requested PyTorch tensor

**Order:** Fill every shape/alignment prediction before writing tensor operations.

**Next concept:** Backward through BatchNorm affine


In [ ]:
# Supplied visible fixture: inspect these values, but predict shapes without printing them.
x = torch.arange(12, dtype=DTYPE).reshape(3, 4).requires_grad_()
dout = torch.linspace(-1.0, 1.0, steps=12, dtype=DTYPE).reshape(3, 4)
_register_case("ex101", {"x": x, "dout": dout})


In [ ]:
# Exercise 101: predict shapes first; do not use autograd.
# Define `ex101_x_shape`.
# Define `ex101_dout_shape`.
# Define `ex101_y_shape`.
# Define `ex101_dx_shape`.
# Define `ex101_dx` with PyTorch tensor operations.
# Write your work below, then run the supplied test cell.


In [ ]:
# Supplied test: expected shapes and values remain private.
_check_private_value("ex101_x_shape", "ex101", "x_shape")
_check_private_value("ex101_dout_shape", "ex101", "dout_shape")
_check_private_value("ex101_y_shape", "ex101", "y_shape")
_check_private_value("ex101_dx_shape", "ex101", "dx_shape")
_check_private_tensor("ex101_dx", "ex101", "dx")


### Exercise 102 — Backward through BatchNorm affine

**Purpose:** Unbroadcast learned feature gain and bias.

**Visible inputs:** `normalized`, `gamma`, `beta`, `dout`. Run the fixture below and read its construction code; it deliberately prints no shape answers.

**Operation to reason about:** `normalized * gamma + beta`

**Task:** Derive `dnormalized`, `dgamma`, and `dbeta`.

**Ingredients:** Both learned parameters are reused down the batch axis and must return to `(1, features)`.

**Required predictions and outputs:**

- `ex102_normalized_shape`: a literal Python shape tuple
- `ex102_gamma_shape`: a literal Python shape tuple
- `ex102_beta_shape`: a literal Python shape tuple
- `ex102_dout_shape`: a literal Python shape tuple
- `ex102_y_shape`: a literal Python shape tuple
- `ex102_dnormalized_shape`: a literal Python shape tuple
- `ex102_dgamma_shape`: a literal Python shape tuple
- `ex102_dbeta_shape`: a literal Python shape tuple
- `ex102_dnormalized`: the requested PyTorch tensor
- `ex102_dgamma`: the requested PyTorch tensor
- `ex102_dbeta`: the requested PyTorch tensor

**Order:** Fill every shape/alignment prediction before writing tensor operations.

**Next concept:** Capstone: column normalization


In [ ]:
# Supplied visible fixture: inspect these values, but predict shapes without printing them.
normalized = torch.linspace(-2.0, 2.0, steps=20, dtype=DTYPE).reshape(4, 5).requires_grad_()
gamma = torch.linspace(0.5, 1.5, steps=5, dtype=DTYPE).reshape(1, 5).requires_grad_()
beta = torch.linspace(-0.2, 0.2, steps=5, dtype=DTYPE).reshape(1, 5).requires_grad_()
dout = torch.linspace(-1.0, 1.0, steps=20, dtype=DTYPE).reshape(4, 5)
_register_case("ex102", {"normalized": normalized, "gamma": gamma, "beta": beta, "dout": dout})


In [ ]:
# Exercise 102: predict shapes first; do not use autograd.
# Define `ex102_normalized_shape`.
# Define `ex102_gamma_shape`.
# Define `ex102_beta_shape`.
# Define `ex102_dout_shape`.
# Define `ex102_y_shape`.
# Define `ex102_dnormalized_shape`.
# Define `ex102_dgamma_shape`.
# Define `ex102_dbeta_shape`.
# Define `ex102_dnormalized` with PyTorch tensor operations.
# Define `ex102_dgamma` with PyTorch tensor operations.
# Define `ex102_dbeta` with PyTorch tensor operations.
# Write your work below, then run the supplied test cell.


In [ ]:
# Supplied test: expected shapes and values remain private.
_check_private_value("ex102_normalized_shape", "ex102", "normalized_shape")
_check_private_value("ex102_gamma_shape", "ex102", "gamma_shape")
_check_private_value("ex102_beta_shape", "ex102", "beta_shape")
_check_private_value("ex102_dout_shape", "ex102", "dout_shape")
_check_private_value("ex102_y_shape", "ex102", "y_shape")
_check_private_value("ex102_dnormalized_shape", "ex102", "dnormalized_shape")
_check_private_value("ex102_dgamma_shape", "ex102", "dgamma_shape")
_check_private_value("ex102_dbeta_shape", "ex102", "dbeta_shape")
_check_private_tensor("ex102_dnormalized", "ex102", "dnormalized")
_check_private_tensor("ex102_dgamma", "ex102", "dgamma")
_check_private_tensor("ex102_dbeta", "ex102", "dbeta")


## 10. Broadcasting mastery capstones

Combine shape preparation, broadcasting, reductions, and named intermediates without loops.


### Exercise 103 — Capstone: column normalization

**Purpose:** Combine reduction shapes, broadcasting, and several named intermediates.

**Visible inputs:** `x`. Run the fixture below and read its construction code; it deliberately prints no shape answers.

**Operation to reason about:** Compute every named intermediate and the normalized output.

**Task:** Compute every named intermediate and the normalized output.

**Ingredients:** Keep feature-statistic axes so subtraction and multiplication broadcast explicitly.

**Required predictions and outputs:**

- `ex103_x_shape`: a literal Python shape tuple
- `ex103_mean_shape`: a literal Python shape tuple
- `ex103_centered_shape`: a literal Python shape tuple
- `ex103_variance_shape`: a literal Python shape tuple
- `ex103_inv_std_shape`: a literal Python shape tuple
- `ex103_normalized_shape`: a literal Python shape tuple
- `ex103_mean`: the requested PyTorch tensor
- `ex103_centered`: the requested PyTorch tensor
- `ex103_variance`: the requested PyTorch tensor
- `ex103_inv_std`: the requested PyTorch tensor
- `ex103_normalized`: the requested PyTorch tensor

**Order:** Fill every shape/alignment prediction before writing tensor operations.

**Next concept:** Capstone: image normalization views


In [ ]:
# Supplied visible fixture: inspect these values, but predict shapes without printing them.
x = torch.arange(20, dtype=DTYPE).reshape(4, 5)
_register_case("ex103", {"x": x})


In [ ]:
# Exercise 103: predict shapes first; do not use autograd.
# Define `ex103_x_shape`.
# Define `ex103_mean_shape`.
# Define `ex103_centered_shape`.
# Define `ex103_variance_shape`.
# Define `ex103_inv_std_shape`.
# Define `ex103_normalized_shape`.
# Define `ex103_mean` with PyTorch tensor operations.
# Define `ex103_centered` with PyTorch tensor operations.
# Define `ex103_variance` with PyTorch tensor operations.
# Define `ex103_inv_std` with PyTorch tensor operations.
# Define `ex103_normalized` with PyTorch tensor operations.
# Write your work below, then run the supplied test cell.


In [ ]:
# Supplied test: expected shapes and values remain private.
_check_private_value("ex103_x_shape", "ex103", "x_shape")
_check_private_value("ex103_mean_shape", "ex103", "mean_shape")
_check_private_value("ex103_centered_shape", "ex103", "centered_shape")
_check_private_value("ex103_variance_shape", "ex103", "variance_shape")
_check_private_value("ex103_inv_std_shape", "ex103", "inv_std_shape")
_check_private_value("ex103_normalized_shape", "ex103", "normalized_shape")
_check_private_tensor("ex103_mean", "ex103", "mean")
_check_private_tensor("ex103_centered", "ex103", "centered")
_check_private_tensor("ex103_variance", "ex103", "variance")
_check_private_tensor("ex103_inv_std", "ex103", "inv_std")
_check_private_tensor("ex103_normalized", "ex103", "normalized")


### Exercise 104 — Capstone: image normalization views

**Purpose:** Prepare compact channel statistics for image-shaped broadcasting.

**Visible inputs:** `images`, `mean`, `std`. Run the fixture below and read its construction code; it deliberately prints no shape answers.

**Operation to reason about:** Create both rank-4 statistic views, then normalize the images.

**Task:** Create both rank-4 statistic views, then normalize the images.

**Ingredients:** Place channel at axis 1 and insert singleton batch and spatial axes.

**Required predictions and outputs:**

- `ex104_images_shape`: a literal Python shape tuple
- `ex104_mean_shape`: a literal Python shape tuple
- `ex104_std_shape`: a literal Python shape tuple
- `ex104_mean_view_shape`: a literal Python shape tuple
- `ex104_std_view_shape`: a literal Python shape tuple
- `ex104_normalized_shape`: a literal Python shape tuple
- `ex104_mean_view`: the requested PyTorch tensor
- `ex104_std_view`: the requested PyTorch tensor
- `ex104_normalized`: the requested PyTorch tensor

**Order:** Fill every shape/alignment prediction before writing tensor operations.

**Next concept:** Capstone: attention masking


In [ ]:
# Supplied visible fixture: inspect these values, but predict shapes without printing them.
images = torch.arange(120, dtype=DTYPE).reshape(2, 3, 4, 5)
mean = torch.tensor([10.0, 20.0, 30.0], dtype=DTYPE)
std = torch.tensor([2.0, 4.0, 5.0], dtype=DTYPE)
_register_case("ex104", {"images": images, "mean": mean, "std": std})


In [ ]:
# Exercise 104: predict shapes first; do not use autograd.
# Define `ex104_images_shape`.
# Define `ex104_mean_shape`.
# Define `ex104_std_shape`.
# Define `ex104_mean_view_shape`.
# Define `ex104_std_view_shape`.
# Define `ex104_normalized_shape`.
# Define `ex104_mean_view` with PyTorch tensor operations.
# Define `ex104_std_view` with PyTorch tensor operations.
# Define `ex104_normalized` with PyTorch tensor operations.
# Write your work below, then run the supplied test cell.


In [ ]:
# Supplied test: expected shapes and values remain private.
_check_private_value("ex104_images_shape", "ex104", "images_shape")
_check_private_value("ex104_mean_shape", "ex104", "mean_shape")
_check_private_value("ex104_std_shape", "ex104", "std_shape")
_check_private_value("ex104_mean_view_shape", "ex104", "mean_view_shape")
_check_private_value("ex104_std_view_shape", "ex104", "std_view_shape")
_check_private_value("ex104_normalized_shape", "ex104", "normalized_shape")
_check_private_tensor("ex104_mean_view", "ex104", "mean_view")
_check_private_tensor("ex104_std_view", "ex104", "std_view")
_check_private_tensor("ex104_normalized", "ex104", "normalized")


### Exercise 105 — Capstone: attention masking

**Purpose:** Prepare a causal mask for batch and head broadcasting.

**Visible inputs:** `scores`, `causal`. Run the fixture below and read its construction code; it deliberately prints no shape answers.

**Operation to reason about:** Create the rank-4 mask view and apply it to every attention matrix.

**Task:** Create the rank-4 mask view and apply it to every attention matrix.

**Ingredients:** Insert singleton batch and head axes before query and key.

**Required predictions and outputs:**

- `ex105_scores_shape`: a literal Python shape tuple
- `ex105_causal_shape`: a literal Python shape tuple
- `ex105_mask_view_shape`: a literal Python shape tuple
- `ex105_masked_scores_shape`: a literal Python shape tuple
- `ex105_mask_view`: the requested PyTorch tensor
- `ex105_masked_scores`: the requested PyTorch tensor

**Order:** Fill every shape/alignment prediction before writing tensor operations.

**Next concept:** Capstone: scaled masked attention scores


In [ ]:
# Supplied visible fixture: inspect these values, but predict shapes without printing them.
scores = torch.arange(96, dtype=DTYPE).reshape(2, 3, 4, 4)
causal = torch.tril(torch.ones((4, 4), dtype=torch.bool))
_register_case("ex105", {"scores": scores, "causal": causal})


In [ ]:
# Exercise 105: predict shapes first; do not use autograd.
# Define `ex105_scores_shape`.
# Define `ex105_causal_shape`.
# Define `ex105_mask_view_shape`.
# Define `ex105_masked_scores_shape`.
# Define `ex105_mask_view` with PyTorch tensor operations.
# Define `ex105_masked_scores` with PyTorch tensor operations.
# Write your work below, then run the supplied test cell.


In [ ]:
# Supplied test: expected shapes and values remain private.
_check_private_value("ex105_scores_shape", "ex105", "scores_shape")
_check_private_value("ex105_causal_shape", "ex105", "causal_shape")
_check_private_value("ex105_mask_view_shape", "ex105", "mask_view_shape")
_check_private_value("ex105_masked_scores_shape", "ex105", "masked_scores_shape")
_check_private_tensor("ex105_mask_view", "ex105", "mask_view")
_check_private_tensor("ex105_masked_scores", "ex105", "masked_scores")


### Exercise 106 — Capstone: scaled masked attention scores

**Purpose:** Combine per-head scaling with a shared causal mask.

**Visible inputs:** `scores`, `head_scale`, `causal`. Run the fixture below and read its construction code; it deliberately prints no shape answers.

**Operation to reason about:** Prepare head scales and the mask, then scale and mask the scores.

**Task:** Prepare head scales and the mask, then scale and mask the scores.

**Ingredients:** Track batch, head, query, and key axes separately.

**Required predictions and outputs:**

- `ex106_scores_shape`: a literal Python shape tuple
- `ex106_head_scale_shape`: a literal Python shape tuple
- `ex106_causal_shape`: a literal Python shape tuple
- `ex106_scale_view_shape`: a literal Python shape tuple
- `ex106_mask_view_shape`: a literal Python shape tuple
- `ex106_scaled_shape`: a literal Python shape tuple
- `ex106_masked_shape`: a literal Python shape tuple
- `ex106_scale_view`: the requested PyTorch tensor
- `ex106_mask_view`: the requested PyTorch tensor
- `ex106_scaled`: the requested PyTorch tensor
- `ex106_masked`: the requested PyTorch tensor

**Order:** Fill every shape/alignment prediction before writing tensor operations.

**Next concept:** Capstone: pairwise squared distances


In [ ]:
# Supplied visible fixture: inspect these values, but predict shapes without printing them.
scores = torch.arange(96, dtype=DTYPE).reshape(2, 3, 4, 4)
head_scale = torch.tensor([0.5, 1.0, 2.0], dtype=DTYPE)
causal = torch.tril(torch.ones((4, 4), dtype=torch.bool))
_register_case("ex106", {"scores": scores, "head_scale": head_scale, "causal": causal})


In [ ]:
# Exercise 106: predict shapes first; do not use autograd.
# Define `ex106_scores_shape`.
# Define `ex106_head_scale_shape`.
# Define `ex106_causal_shape`.
# Define `ex106_scale_view_shape`.
# Define `ex106_mask_view_shape`.
# Define `ex106_scaled_shape`.
# Define `ex106_masked_shape`.
# Define `ex106_scale_view` with PyTorch tensor operations.
# Define `ex106_mask_view` with PyTorch tensor operations.
# Define `ex106_scaled` with PyTorch tensor operations.
# Define `ex106_masked` with PyTorch tensor operations.
# Write your work below, then run the supplied test cell.


In [ ]:
# Supplied test: expected shapes and values remain private.
_check_private_value("ex106_scores_shape", "ex106", "scores_shape")
_check_private_value("ex106_head_scale_shape", "ex106", "head_scale_shape")
_check_private_value("ex106_causal_shape", "ex106", "causal_shape")
_check_private_value("ex106_scale_view_shape", "ex106", "scale_view_shape")
_check_private_value("ex106_mask_view_shape", "ex106", "mask_view_shape")
_check_private_value("ex106_scaled_shape", "ex106", "scaled_shape")
_check_private_value("ex106_masked_shape", "ex106", "masked_shape")
_check_private_tensor("ex106_scale_view", "ex106", "scale_view")
_check_private_tensor("ex106_mask_view", "ex106", "mask_view")
_check_private_tensor("ex106_scaled", "ex106", "scaled")
_check_private_tensor("ex106_masked", "ex106", "masked")


### Exercise 107 — Capstone: pairwise squared distances

**Purpose:** Build every pair of vectors without Python loops.

**Visible inputs:** `x`, `y`. Run the fixture below and read its construction code; it deliberately prints no shape answers.

**Operation to reason about:** Create both item-axis views, compute pairwise differences, then reduce features.

**Task:** Create both item-axis views, compute pairwise differences, then reduce features.

**Ingredients:** Broadcast only the item axes; preserve the shared feature axis until the final sum.

**Required predictions and outputs:**

- `ex107_x_shape`: a literal Python shape tuple
- `ex107_y_shape`: a literal Python shape tuple
- `ex107_x_rows_shape`: a literal Python shape tuple
- `ex107_y_rows_shape`: a literal Python shape tuple
- `ex107_difference_shape`: a literal Python shape tuple
- `ex107_sq_distance_shape`: a literal Python shape tuple
- `ex107_x_rows`: the requested PyTorch tensor
- `ex107_y_rows`: the requested PyTorch tensor
- `ex107_difference`: the requested PyTorch tensor
- `ex107_sq_distance`: the requested PyTorch tensor

**Order:** Fill every shape/alignment prediction before writing tensor operations.

**Next concept:** Capstone: mixture of experts


In [ ]:
# Supplied visible fixture: inspect these values, but predict shapes without printing them.
x = torch.tensor([[0.0, 0.0, 0.0], [1.0, 2.0, 3.0]], dtype=DTYPE)
y = torch.tensor([[1.0, 0.0, 0.0], [0.0, 2.0, 0.0], [1.0, 1.0, 1.0], [2.0, 2.0, 2.0]], dtype=DTYPE)
_register_case("ex107", {"x": x, "y": y})


In [ ]:
# Exercise 107: predict shapes first; do not use autograd.
# Define `ex107_x_shape`.
# Define `ex107_y_shape`.
# Define `ex107_x_rows_shape`.
# Define `ex107_y_rows_shape`.
# Define `ex107_difference_shape`.
# Define `ex107_sq_distance_shape`.
# Define `ex107_x_rows` with PyTorch tensor operations.
# Define `ex107_y_rows` with PyTorch tensor operations.
# Define `ex107_difference` with PyTorch tensor operations.
# Define `ex107_sq_distance` with PyTorch tensor operations.
# Write your work below, then run the supplied test cell.


In [ ]:
# Supplied test: expected shapes and values remain private.
_check_private_value("ex107_x_shape", "ex107", "x_shape")
_check_private_value("ex107_y_shape", "ex107", "y_shape")
_check_private_value("ex107_x_rows_shape", "ex107", "x_rows_shape")
_check_private_value("ex107_y_rows_shape", "ex107", "y_rows_shape")
_check_private_value("ex107_difference_shape", "ex107", "difference_shape")
_check_private_value("ex107_sq_distance_shape", "ex107", "sq_distance_shape")
_check_private_tensor("ex107_x_rows", "ex107", "x_rows")
_check_private_tensor("ex107_y_rows", "ex107", "y_rows")
_check_private_tensor("ex107_difference", "ex107", "difference")
_check_private_tensor("ex107_sq_distance", "ex107", "sq_distance")


### Exercise 108 — Capstone: mixture of experts

**Purpose:** Broadcast expert weights over features and then reduce experts.

**Visible inputs:** `weights`, `experts`. Run the fixture below and read its construction code; it deliberately prints no shape answers.

**Operation to reason about:** Prepare weights, compute weighted expert outputs, and sum the expert axis.

**Task:** Prepare weights, compute weighted expert outputs, and sum the expert axis.

**Ingredients:** Do not remove the expert axis until after elementwise weighting.

**Required predictions and outputs:**

- `ex108_weights_shape`: a literal Python shape tuple
- `ex108_experts_shape`: a literal Python shape tuple
- `ex108_weight_view_shape`: a literal Python shape tuple
- `ex108_weighted_shape`: a literal Python shape tuple
- `ex108_mixture_shape`: a literal Python shape tuple
- `ex108_weight_view`: the requested PyTorch tensor
- `ex108_weighted`: the requested PyTorch tensor
- `ex108_mixture`: the requested PyTorch tensor

**Order:** Fill every shape/alignment prediction before writing tensor operations.

**Next concept:** Capstone: masked class losses


In [ ]:
# Supplied visible fixture: inspect these values, but predict shapes without printing them.
weights = torch.tensor([[0.2, 0.3, 0.5], [0.5, 0.25, 0.25]], dtype=DTYPE)
experts = torch.arange(24, dtype=DTYPE).reshape(2, 3, 4)
_register_case("ex108", {"weights": weights, "experts": experts})


In [ ]:
# Exercise 108: predict shapes first; do not use autograd.
# Define `ex108_weights_shape`.
# Define `ex108_experts_shape`.
# Define `ex108_weight_view_shape`.
# Define `ex108_weighted_shape`.
# Define `ex108_mixture_shape`.
# Define `ex108_weight_view` with PyTorch tensor operations.
# Define `ex108_weighted` with PyTorch tensor operations.
# Define `ex108_mixture` with PyTorch tensor operations.
# Write your work below, then run the supplied test cell.


In [ ]:
# Supplied test: expected shapes and values remain private.
_check_private_value("ex108_weights_shape", "ex108", "weights_shape")
_check_private_value("ex108_experts_shape", "ex108", "experts_shape")
_check_private_value("ex108_weight_view_shape", "ex108", "weight_view_shape")
_check_private_value("ex108_weighted_shape", "ex108", "weighted_shape")
_check_private_value("ex108_mixture_shape", "ex108", "mixture_shape")
_check_private_tensor("ex108_weight_view", "ex108", "weight_view")
_check_private_tensor("ex108_weighted", "ex108", "weighted")
_check_private_tensor("ex108_mixture", "ex108", "mixture")


### Exercise 109 — Capstone: masked class losses

**Purpose:** Combine class weighting, token masking, and a final reduction.

**Visible inputs:** `losses`, `class_weights`, `token_mask`. Run the fixture below and read its construction code; it deliberately prints no shape answers.

**Operation to reason about:** Weight classes, prepare the token mask, mask every class loss, and total the result.

**Task:** Weight classes, prepare the token mask, mask every class loss, and total the result.

**Ingredients:** Class weights align last; token validity needs a trailing singleton class axis.

**Required predictions and outputs:**

- `ex109_losses_shape`: a literal Python shape tuple
- `ex109_class_weights_shape`: a literal Python shape tuple
- `ex109_token_mask_shape`: a literal Python shape tuple
- `ex109_class_weighted_shape`: a literal Python shape tuple
- `ex109_mask_view_shape`: a literal Python shape tuple
- `ex109_masked_shape`: a literal Python shape tuple
- `ex109_total_shape`: a literal Python shape tuple
- `ex109_class_weighted`: the requested PyTorch tensor
- `ex109_mask_view`: the requested PyTorch tensor
- `ex109_masked`: the requested PyTorch tensor
- `ex109_total`: the requested PyTorch tensor

**Order:** Fill every shape/alignment prediction before writing tensor operations.

**Next concept:** Capstone: masked LayerNorm affine


In [ ]:
# Supplied visible fixture: inspect these values, but predict shapes without printing them.
losses = torch.arange(30, dtype=DTYPE).reshape(2, 3, 5) / 10
class_weights = torch.tensor([1.0, 0.5, 2.0, 1.5, 3.0], dtype=DTYPE)
token_mask = torch.tensor([[1.0, 1.0, 0.0], [1.0, 0.0, 0.0]], dtype=DTYPE)
_register_case("ex109", {"losses": losses, "class_weights": class_weights, "token_mask": token_mask})


In [ ]:
# Exercise 109: predict shapes first; do not use autograd.
# Define `ex109_losses_shape`.
# Define `ex109_class_weights_shape`.
# Define `ex109_token_mask_shape`.
# Define `ex109_class_weighted_shape`.
# Define `ex109_mask_view_shape`.
# Define `ex109_masked_shape`.
# Define `ex109_total_shape`.
# Define `ex109_class_weighted` with PyTorch tensor operations.
# Define `ex109_mask_view` with PyTorch tensor operations.
# Define `ex109_masked` with PyTorch tensor operations.
# Define `ex109_total` with PyTorch tensor operations.
# Write your work below, then run the supplied test cell.


In [ ]:
# Supplied test: expected shapes and values remain private.
_check_private_value("ex109_losses_shape", "ex109", "losses_shape")
_check_private_value("ex109_class_weights_shape", "ex109", "class_weights_shape")
_check_private_value("ex109_token_mask_shape", "ex109", "token_mask_shape")
_check_private_value("ex109_class_weighted_shape", "ex109", "class_weighted_shape")
_check_private_value("ex109_mask_view_shape", "ex109", "mask_view_shape")
_check_private_value("ex109_masked_shape", "ex109", "masked_shape")
_check_private_value("ex109_total_shape", "ex109", "total_shape")
_check_private_tensor("ex109_class_weighted", "ex109", "class_weighted")
_check_private_tensor("ex109_mask_view", "ex109", "mask_view")
_check_private_tensor("ex109_masked", "ex109", "masked")
_check_private_tensor("ex109_total", "ex109", "total")


### Exercise 110 — Capstone: masked LayerNorm affine

**Purpose:** Integrate reductions, feature broadcasting, learned affine parameters, and token masking.

**Visible inputs:** `x`, `gamma`, `beta`, `token_mask`. Run the fixture below and read its construction code; it deliberately prints no shape answers.

**Operation to reason about:** Compute feature-wise normalization, affine transformation, a mask view, and masked output.

**Task:** Compute feature-wise normalization, affine transformation, a mask view, and masked output.

**Ingredients:** Statistics keep the final feature axis reduced; gamma and beta align last; the token mask gains one trailing axis.

**Required predictions and outputs:**

- `ex110_x_shape`: a literal Python shape tuple
- `ex110_gamma_shape`: a literal Python shape tuple
- `ex110_beta_shape`: a literal Python shape tuple
- `ex110_token_mask_shape`: a literal Python shape tuple
- `ex110_mean_shape`: a literal Python shape tuple
- `ex110_centered_shape`: a literal Python shape tuple
- `ex110_variance_shape`: a literal Python shape tuple
- `ex110_normalized_shape`: a literal Python shape tuple
- `ex110_affine_shape`: a literal Python shape tuple
- `ex110_mask_view_shape`: a literal Python shape tuple
- `ex110_out_shape`: a literal Python shape tuple
- `ex110_mean`: the requested PyTorch tensor
- `ex110_centered`: the requested PyTorch tensor
- `ex110_variance`: the requested PyTorch tensor
- `ex110_normalized`: the requested PyTorch tensor
- `ex110_affine`: the requested PyTorch tensor
- `ex110_mask_view`: the requested PyTorch tensor
- `ex110_out`: the requested PyTorch tensor

**Order:** Fill every shape/alignment prediction before writing tensor operations.

**Next concept:** Fresh-kernel mastery run.


In [ ]:
# Supplied visible fixture: inspect these values, but predict shapes without printing them.
x = torch.arange(24, dtype=DTYPE).reshape(2, 3, 4)
gamma = torch.tensor([0.5, 1.0, 1.5, 2.0], dtype=DTYPE)
beta = torch.tensor([-1.0, 0.0, 1.0, 2.0], dtype=DTYPE)
token_mask = torch.tensor([[1.0, 1.0, 0.0], [1.0, 0.0, 0.0]], dtype=DTYPE)
_register_case("ex110", {"x": x, "gamma": gamma, "beta": beta, "token_mask": token_mask})


In [ ]:
# Exercise 110: predict shapes first; do not use autograd.
# Define `ex110_x_shape`.
# Define `ex110_gamma_shape`.
# Define `ex110_beta_shape`.
# Define `ex110_token_mask_shape`.
# Define `ex110_mean_shape`.
# Define `ex110_centered_shape`.
# Define `ex110_variance_shape`.
# Define `ex110_normalized_shape`.
# Define `ex110_affine_shape`.
# Define `ex110_mask_view_shape`.
# Define `ex110_out_shape`.
# Define `ex110_mean` with PyTorch tensor operations.
# Define `ex110_centered` with PyTorch tensor operations.
# Define `ex110_variance` with PyTorch tensor operations.
# Define `ex110_normalized` with PyTorch tensor operations.
# Define `ex110_affine` with PyTorch tensor operations.
# Define `ex110_mask_view` with PyTorch tensor operations.
# Define `ex110_out` with PyTorch tensor operations.
# Write your work below, then run the supplied test cell.


In [ ]:
# Supplied test: expected shapes and values remain private.
_check_private_value("ex110_x_shape", "ex110", "x_shape")
_check_private_value("ex110_gamma_shape", "ex110", "gamma_shape")
_check_private_value("ex110_beta_shape", "ex110", "beta_shape")
_check_private_value("ex110_token_mask_shape", "ex110", "token_mask_shape")
_check_private_value("ex110_mean_shape", "ex110", "mean_shape")
_check_private_value("ex110_centered_shape", "ex110", "centered_shape")
_check_private_value("ex110_variance_shape", "ex110", "variance_shape")
_check_private_value("ex110_normalized_shape", "ex110", "normalized_shape")
_check_private_value("ex110_affine_shape", "ex110", "affine_shape")
_check_private_value("ex110_mask_view_shape", "ex110", "mask_view_shape")
_check_private_value("ex110_out_shape", "ex110", "out_shape")
_check_private_tensor("ex110_mean", "ex110", "mean")
_check_private_tensor("ex110_centered", "ex110", "centered")
_check_private_tensor("ex110_variance", "ex110", "variance")
_check_private_tensor("ex110_normalized", "ex110", "normalized")
_check_private_tensor("ex110_affine", "ex110", "affine")
_check_private_tensor("ex110_mask_view", "ex110", "mask_view")
_check_private_tensor("ex110_out", "ex110", "out")


## Completion standard

You have mastered this workbook when you can restart the kernel, run every completed exercise, and explain aloud:

- why broadcasting aligns dimensions from the right;
- why missing dimensions are leading conceptual ones;
- why size `1` means “available for expansion” rather than “automatically repeated now”;
- how `None`, `unsqueeze`, and `reshape` place semantic axes deliberately;
- why `(rows,)` does not automatically mean the row axis of `(rows, columns)`;
- how `expand` differs from `repeat` and cloning;
- how masks, normalization parameters, image channels, attention tensors, and mixture weights align;
- why backward sums every axis that forward broadcasting expanded.

## Official references

- [PyTorch broadcasting semantics](https://docs.pytorch.org/docs/stable/notes/broadcasting.html)
- [`Tensor.unsqueeze`](https://docs.pytorch.org/docs/stable/generated/torch.unsqueeze.html)
- [`Tensor.expand`](https://docs.pytorch.org/docs/stable/generated/torch.Tensor.expand.html)
- [`torch.broadcast_to`](https://docs.pytorch.org/docs/stable/generated/torch.broadcast_to.html)
- [`torch.broadcast_tensors`](https://docs.pytorch.org/docs/stable/generated/torch.broadcast_tensors.html)
- [`Tensor.repeat`](https://docs.pytorch.org/docs/stable/generated/torch.Tensor.repeat.html)
- [Tensor views](https://docs.pytorch.org/docs/stable/tensor_view.html)
- [Autograd mechanics](https://docs.pytorch.org/docs/stable/notes/autograd.html)
